# Main Data 

## Imports, Helpers, Parameters

### Imports

In [39]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textwrap import wrap
import numpy as np
import re
import os
from pathlib import Path
from collections import Counter


### Helper Functions

In [40]:
#Helper Functions

def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()                   # lowercase + trim
    text = re.sub(r"[^\w\s]", "", text)            # remove punctuation
    text = re.sub(r"\s+", " ", text)               # collapse multiple spaces
    return text


### Adjust Parameters

In [41]:
# Frequency mapping. Assuming a 52 week year with 5 working days per week, these are corresponding survey questions::
# 1 Once per year or less (Assuming 1 time per year)
# 2 More than once per year (Assuming 3 times per year)
# 3 More than once per month (Assuming 48 times per year, 3 times per month)
# 4 More than once per week (Assuming 130 times per year, 2.5 times per week)
# 5 Daily
# 6 Several times per day (Assuming 3 times per day)
# 7 Hourly or more often (Assuming 12 times per day, 1.5 times per hour)
frequency_weights = {
    1: 1 / 260,
    2: 3 / 260,
    3: 48 / 260,
    4: 130 / 260,
    5: 1,
    6: 3,
    7: 12
}

# Adjust inflation for scraped wage data (Jan 2020 to now)
jan_2020_inflation_factor = 1.24

# Adjust inflation for BLS wage data (May 2015 to now)
may_2015_inflation_factor = 1.36


## CSV Inputs and Config

In [42]:
# Change to true to get each data frame saved to a csv after every step to a folder named merged_data_files
save_files_each_step = False
run_name = "first_pass_mcp_v4.csv"

In [43]:
data_dir = Path("../data")

runs = {
    "first_pass_microsoft.csv": "iwa_metrics.csv",
    "first_pass_aei_v1.csv": "task_pct_v1.csv",
    "first_pass_aei_v2.csv": "task_pct_v2.csv",
    "first_pass_mcp_v1.csv": "task_results_2025-04-24.csv",
    "first_pass_mcp_v2.csv": "task_results_2025-05-24.csv",
    "first_pass_mcp_v3.csv": "task_results_2025-07-23.csv",
    "first_pass_aei_v3.csv": "task_pct_v3.csv",
    "first_pass_aei_api_v3.csv": "task_pct_api_v3.csv",
    "first_pass_aei_v4.csv": "task_pct_v4.csv",
    "first_pass_aei_api_v4.csv": "task_pct_api_v4.csv",
    "first_pass_aei_v5.csv": "task_pct_v5.csv",
    "first_pass_aei_api_v5.csv": "task_pct_api_v5.csv",
    "first_pass_mcp_v4.csv": "task_results_2026-02-18.csv",
    "first_pass_eco_tasks_2015.csv": "task_pct_eco_2015.csv",
    "first_pass_eco_tasks_2025.csv": "task_pct_eco_2025.csv"
}

runs_v3_and_up = {
    "first_pass_aei_v3.csv": "aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv",
    "first_pass_aei_api_v3.csv": "aei_raw_1p_api_2025-08-04_to_2025-08-11.csv",
    "first_pass_aei_v4.csv": "aei_raw_claude_ai_2025-11-13_to_2025-11-20.csv",
    "first_pass_aei_api_v4.csv": "aei_raw_1p_api_2025-11-13_to_2025-11-20.csv",
    "first_pass_aei_v5.csv": "aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv",
    "first_pass_aei_api_v5.csv": "aei_raw_1p_api_2026-02-05_to_2026-02-12.csv",
}

# ---- AUTO DETECT TYPE ----

aei_run = "aei" in run_name.lower()
mcp_run = "mcp" in run_name.lower()
microsoft_run = "microsoft" in run_name.lower()
eco_run_2015 = "first_pass_eco_tasks_2015.csv" == run_name
eco_run_2025 = "first_pass_eco_tasks_2025.csv" == run_name

aei_v3_and_up = aei_run and any(v in run_name for v in ["v3", "v4", "v5"])

# ---- SET INPUT FILE ----

input_file = data_dir / runs[run_name]

# ---- SET SPECIAL AEI RAW FILE (if needed) ----

if aei_v3_and_up:
    aei_v3_and_up_pct_df = data_dir / runs_v3_and_up[run_name]
else:
    aei_v3_and_up_pct_df = None

# ---- SET NOTEBOOK STEP INPUTS ----

pct_df = input_file if aei_run else None
mcp_data = input_file if mcp_run else None
microsoft_data = input_file if microsoft_run else None
eco_data_2015 = input_file if eco_run_2015 else None
eco_data_2025 = input_file if eco_run_2025 else None
# ---- OUTPUT FILE ----

output_file_main_1 = data_dir / run_name

print("Run type:")
print("AEI:", aei_run)
print("MCP:", mcp_run)
print("Microsoft:", microsoft_run)
print("AEI v3+:", aei_v3_and_up)
print("Input file:", input_file)
print("Output file:", output_file_main_1)

Run type:
AEI: False
MCP: True
Microsoft: False
AEI v3+: False
Input file: ..\data\task_results_2026-02-18.csv
Output file: ..\data\first_pass_mcp_v4.csv


## Step 1: Map Anthropic Task %s to O*NET v20.1 Task Statements

In [44]:
if mcp_run or microsoft_run:
    save_files_each_step = False

if save_files_each_step:
    import os
    os.makedirs("../merged_data_files", exist_ok=True)

#### AEI v3+ Pre Processing

In [45]:
if aei_v3_and_up:
    df = pd.read_csv(aei_v3_and_up_pct_df)

    filtered = df[
        (df["geo_id"] == "GLOBAL") &
        (df["facet"] == "onet_task") &
        (df["variable"] == "onet_task_pct")
    ][["cluster_name", "value"]].rename(columns={"cluster_name": "task_name", "value": "pct"})

    filtered.to_csv(pct_df, index=False)

#### ONET Version

In [46]:
def pct_to_onet_tasks(pct_df, task_statements_df) -> pd.DataFrame:
    """
    Description:
        This loads in the tasks and percentage of occurrences from the Anthropic data, and merges it with the tasks statement data. 
        It normalizes the percents based on a weighted and non weighted approach.
        See documentation for more details.

    Args:
        pct_df (pd.DataFrame): DataFrame containing the Anthropic data of percent occurances of every task in their conversation data
        task_statements_df (pd.DataFrame): DataFrame containing O*NET tasks and SOC titles.
    
    Returns:
        pd.DataFrame: Updated DataFrame with percentage of occurrences added.
    """

    task_statements_df.rename(columns={
    "O*NET-SOC Code": "soc_code_2010",
    "Title": "title",
    "Task ID": "task_id",
    "Task": "task",
    "Task Type": "task_type",
    "Incumbents Responding": "n_responding",
    "Date": "date",
    "Domain Source": "domain_source",
    }, inplace=True)

    # Normalize task columns
    pct_df["task_normalized_temp"] = pct_df["task_name"].apply(normalize_text)
    task_statements_df["task_normalized"] = task_statements_df["task"].apply(normalize_text)
    # task_statements_df["task_normalized"] = task_statements_df["task"].str.lower().str.strip()

    pct_df = pct_df.groupby("task_normalized_temp", as_index=False).agg({
    "task_name": "first",  # Keep the first task name
    "pct": "sum"  # Sum the percentages for duplicates
    })
    
    # Merge dfs
    merged = pct_df.merge(
        task_statements_df,
        left_on="task_normalized_temp",
        right_on="task_normalized",
        how="left"
    )
    
    # Calculate weighted and normalized percentages
    merged["n_occurrences"] = merged.groupby("task_normalized")["title"].transform("nunique")
    merged["pct_weighted"] = 100 * merged["pct"] / merged["pct"].sum()
    merged["pct_normalized"] = 100 * (merged["pct"] / merged["n_occurrences"]) / (merged["pct"] / merged["n_occurrences"]).sum()

    # Drop unnecessary columns
    merged.drop(columns=["task_name", "task_normalized_temp", "pct"], inplace=True)

    # Reorder so `task` is first and `task_normalized` is second
    cols = ["task", "task_normalized"] + [c for c in merged.columns if c not in ["task", "task_normalized"]]
    merged = merged[cols]
    
    # Sort by O*NET-SOC Code
    merged.sort_values(by="soc_code_2010", ascending=True, inplace=True)

    return merged.reset_index(drop=True)


#### MCP Version

In [47]:
if mcp_run:

    # -----------------------------
    # Load base data
    # -----------------------------
    df = pd.read_csv(mcp_data)
    task_statements = pd.read_csv("../data/task_statements_v30.1.csv")
    crosswalk = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")

    # -----------------------------
    # Winsorize n_ratings + pct
    # -----------------------------
    df["n_ratings"] = pd.to_numeric(df["n_ratings"], errors="coerce")

    q1 = df["n_ratings"].quantile(0.25)
    q3 = df["n_ratings"].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    df["n_ratings_capped"] = df["n_ratings"].clip(upper=upper_bound)
    df["pct"] = 100 * df["n_ratings_capped"] / df["n_ratings_capped"].sum()

    df = df[["occupation", "task", "pct"]].copy()

    # -----------------------------
    # Normalize merge keys
    # -----------------------------
    df["occupation_normalized"] = df["occupation"].apply(normalize_text)
    df["task_normalized"] = df["task"].apply(normalize_text)

    task_statements["occupation_normalized"] = task_statements["Title"].apply(normalize_text)
    task_statements["task_normalized"] = task_statements["Task"].apply(normalize_text)

    # -----------------------------
    # Exact merge on occupation + task
    # -----------------------------
    df = df.merge(
        task_statements,
        on=["occupation_normalized", "task_normalized"],
        how="left"
    )

    # -----------------------------
    # Task substring fallback
    # -----------------------------
    task_lookup = task_statements.groupby("occupation_normalized")

    def fallback_match(row):
        if pd.notna(row["O*NET-SOC Code"]):
            return row["O*NET-SOC Code"]
        occ = row["occupation_normalized"]
        if occ not in task_lookup.groups:
            return None
        for _, cand in task_lookup.get_group(occ).iterrows():
            if row["task_normalized"] in cand["task_normalized"] or \
            cand["task_normalized"] in row["task_normalized"]:
                return cand["O*NET-SOC Code"]
        return None

    df["O*NET-SOC Code"] = df.apply(fallback_match, axis=1)

    # -----------------------------
    # Crosswalk 2019 → 2010
    # -----------------------------
    crosswalk = crosswalk.drop_duplicates(
        subset=["O*NET-SOC 2019 Code"],
        keep="first"
    )

    df = df.merge(
        crosswalk[[
            "O*NET-SOC 2019 Code",
            "O*NET-SOC 2010 Code",
            "O*NET-SOC 2010 Title"
        ]],
        left_on="O*NET-SOC Code",
        right_on="O*NET-SOC 2019 Code",
        how="left"
    )

    # Decimal fallback
    df["soc_2019_base"] = df["O*NET-SOC Code"].astype(str).str.split(".").str[0]
    crosswalk["soc_2019_base"] = crosswalk["O*NET-SOC 2019 Code"].astype(str).str.split(".").str[0]

    crosswalk_base = crosswalk[[
        "soc_2019_base",
        "O*NET-SOC 2010 Code",
        "O*NET-SOC 2010 Title"
    ]].drop_duplicates(subset=["soc_2019_base"], keep="first")

    missing_mask = df["O*NET-SOC 2010 Code"].isna()

    fallback = df.loc[missing_mask, ["soc_2019_base"]].merge(
        crosswalk_base,
        on="soc_2019_base",
        how="left"
    )

    df.loc[missing_mask, "O*NET-SOC 2010 Code"] = fallback["O*NET-SOC 2010 Code"].values
    df.loc[missing_mask, "O*NET-SOC 2010 Title"] = fallback["O*NET-SOC 2010 Title"].values

    # -----------------------------
    # Final column formatting
    # -----------------------------
    df = df.rename(columns={
        "pct": "pct_normalized",
        "O*NET-SOC Code": "soc_code_2019_full",
        "O*NET-SOC 2010 Code": "soc_code_2010",
        "O*NET-SOC 2010 Title": "title",
        "Task ID": "task_id",
        "Task Type": "task_type",
        "Incumbents Responding": "n_responding",
        "Date": "date",
        "Domain Source": "domain_source",
        "occupation": "title_current"
    })

    df["n_occurrences"] = (
        df.groupby("task_normalized")["title"]
        .transform("nunique")
    )

    df["pct_weighted"] = df["pct_normalized"] * df["n_occurrences"]
    df["pct_weighted"] = df["pct_weighted"] / df["pct_weighted"].sum()

    df_finalized = df[[
        "task",
        "task_normalized",
        "soc_code_2019_full",
        "soc_code_2010",
        "title_current",
        "title",
        "task_id",
        "task_type",
        "n_responding",
        "date",
        "domain_source",
        "n_occurrences",
        "pct_weighted",
        "pct_normalized"
    ]].copy()

    df_finalized



#### Microsoft Data Version

In [48]:
if microsoft_run:

    # 1. Load and Filter iwa_metrics
    iwa_metrics = pd.read_csv(microsoft_data)
    iwa_metrics = iwa_metrics[iwa_metrics[['share_user', 'share_ai']].max(axis=1) >= 0.0005].copy()

    # 2. Load Tasks (Task-Occupation pairs)
    tasks_df = pd.read_csv("../data/tasks_dwa_iwa_gwa_v30.1_physical.csv")

    # Create normalized keys
    iwa_metrics['norm_title'] = iwa_metrics['title'].apply(normalize_text)
    tasks_df['norm_iwa_title'] = tasks_df['iwa_title'].apply(normalize_text)

    # 3. Per-IWA: total_share (normalized to sum to 1 across IWAs) and impact_scope adjustments
    iwa_metrics['total_share'] = iwa_metrics[['share_user', 'share_ai']].sum(axis=1)
    iwa_metrics['total_share'] = iwa_metrics['total_share'] / iwa_metrics['total_share'].sum()
    iwa_metrics["iwa_title"] = iwa_metrics["title"]
    iwa_metrics["impact_scope_ai_adj"] = iwa_metrics["impact_scope_ai"] * (iwa_metrics["share_ai"] / (iwa_metrics["share_user"] + iwa_metrics["share_ai"]))
    iwa_metrics["impact_scope_user_adj"] = iwa_metrics["impact_scope_user"] * (iwa_metrics["share_user"] / (iwa_metrics["share_user"] + iwa_metrics["share_ai"]))

    # 4. Divide each IWA's total_share by the count of unique (Title, task) pairs under it
    unique_pair_iwa = tasks_df[['Title', 'task', 'norm_iwa_title']].drop_duplicates()
    pairs_per_iwa = unique_pair_iwa.groupby('norm_iwa_title').size().reset_index(name='n_pairs')
    iwa_calc = iwa_metrics.merge(pairs_per_iwa, left_on='norm_title', right_on='norm_iwa_title', how='left')
    iwa_calc['pair_share'] = iwa_calc['total_share'] / iwa_calc['n_pairs']

    # 5. Each pair gets its IWA's slice; average across the IWAs the pair appears in
    # (each IWA appearance represents the same underlying conversation, so we split evenly, not sum);
    # then rescale to 100 across unique pairs.
    pair_iwa_slices = unique_pair_iwa.merge(iwa_calc[['norm_iwa_title', 'pair_share']], on='norm_iwa_title', how='left')
    pair_totals = pair_iwa_slices.groupby(['Title', 'task'], as_index=False)['pair_share'].mean()
    pair_totals = pair_totals[pair_totals['pair_share'] > 0].copy()
    total = pair_totals['pair_share'].sum()
    pair_totals['pct'] = 100 * pair_totals['pair_share'] / total if total > 0 else 0.0

    # 6. Attach O*NET taxonomy — pct duplicates across (DWA, IWA, GWA) rows for each pair.
    #    Inner-merge on norm_iwa_title drops taxonomy rows whose IWA was filtered out of
    #    iwa_metrics (share < 0.0005). Pairs only appear under their surviving IWA(s),
    #    which avoids NaN impact_scope rows downstream.
    merged_pairs = (tasks_df[['Title', 'task', 'norm_iwa_title', 'dwa_title', 'iwa_title', 'gwa_title']]
                    .merge(pair_totals[['Title', 'task', 'pct']], on=['Title', 'task'], how='inner')
                    .merge(iwa_calc[['norm_iwa_title', 'impact_scope_ai', 'impact_scope_user',
                                     'impact_scope_ai_adj', 'impact_scope_user_adj']],
                           on='norm_iwa_title', how='inner'))
    # print(merged_pairs['impact_scope_ai'].isna().value_counts())
    # 6. Deduplicate Task-Occupation pairs just in case
    # We group by occupation (title) and task name
    # final_output = merged_pairs.drop_duplicates(subset=['Title', 'task']).copy()
    final_output = merged_pairs

    # Rename for final schema
    final_output = final_output.rename(columns={'task': 'task_name', 'Title': 'title_current'})

    # Final cleanup
    final_output = final_output[['title_current', 'task_name', 'pct', 'impact_scope_ai', 'impact_scope_user', 'impact_scope_ai_adj', 'impact_scope_user_adj', "dwa_title", "iwa_title", "gwa_title"]]

    print(f"Total Task-Occupation Pairs: {len(final_output)}")
    print(f"Pairs with valid pct: {final_output['pct'].notna().sum()}")

    final_output.to_csv("../data/task_pct_microsoft.csv", index=False)






    # -----------------------------
    # 1. Load Data
    # -----------------------------
    # This is the file we just created with [task_name, pct]
    # df = pd.read_csv("task_weights_by_iwa.csv") 
    df = final_output
    task_statements = pd.read_csv("../data/task_statements_v30.1.csv")
    crosswalk = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")

    # -----------------------------
    # 2. Normalize and Merge
    # -----------------------------
    # We treat the 'pct' from the IWA file as our base 'pct_normalized'
    df = df.rename(columns={"task_name": "task", "pct": "pct_normalized"})

    # Create normalized keys for both Task and Occupation (Title)
    df["task_normalized"] = df["task"].apply(normalize_text)
    df["title_normalized"] = df["title_current"].apply(normalize_text)

    task_statements["task_normalized"] = task_statements["Task"].apply(normalize_text)
    task_statements["title_normalized"] = task_statements["Title"].apply(normalize_text)

    # Merge on BOTH task and occupation to get the correct O*NET metadata
    df = df.merge(
        task_statements,
        on=["task_normalized", "title_normalized"],
        how="left"
    )

    # -----------------------------
    # 3. Calculate Occurrences and Normalized Pct
    # -----------------------------
    # n_occurrences is how many occupations this task belongs to
    df["n_occurrences"] = (
        df.groupby("task_normalized")["title_current"]
        .transform("nunique")
    )

    # Per your instruction: divide pct by n_occurrences to get pct_normalized
    # Then re-normalize so the column sums to 1.0
    df["pct_weighted"] = df["pct_normalized"] * df["n_occurrences"]
    df["pct_weighted"] = df["pct_weighted"] / df["pct_weighted"].sum()

    # -----------------------------
    # 4. Crosswalk 2019 → 2010
    # -----------------------------
    crosswalk_clean = crosswalk.drop_duplicates(subset=["O*NET-SOC 2019 Code"], keep="first")

    df = df.merge(
        crosswalk_clean[[
            "O*NET-SOC 2019 Code",
            "O*NET-SOC 2010 Code",
            "O*NET-SOC 2010 Title"
        ]],
        left_on="O*NET-SOC Code",
        right_on="O*NET-SOC 2019 Code",
        how="left"
    )

    # Decimal fallback for missing SOC codes
    df["soc_2019_base"] = df["O*NET-SOC Code"].astype(str).str.split(".").str[0]
    crosswalk["soc_2019_base"] = crosswalk["O*NET-SOC 2019 Code"].astype(str).str.split(".").str[0]
    crosswalk_base = crosswalk[["soc_2019_base", "O*NET-SOC 2010 Code", "O*NET-SOC 2010 Title"]].drop_duplicates("soc_2019_base")

    missing_mask = df["O*NET-SOC 2010 Code"].isna()
    fallback = df.loc[missing_mask, ["soc_2019_base"]].merge(crosswalk_base, on="soc_2019_base", how="left")

    df.loc[missing_mask, "O*NET-SOC 2010 Code"] = fallback["O*NET-SOC 2010 Code"].values
    df.loc[missing_mask, "O*NET-SOC 2010 Title"] = fallback["O*NET-SOC 2010 Title"].values

    # -----------------------------
    # 5. Final Column Formatting
    # -----------------------------
    df = df.rename(columns={
        "O*NET-SOC Code": "soc_code_2019_full",
        "O*NET-SOC 2010 Code": "soc_code_2010",
        "O*NET-SOC 2010 Title": "title",
        "Task ID": "task_id",
        "Task Type": "task_type",
        "Incumbents Responding": "n_responding",
        "Date": "date",
        "Domain Source": "domain_source",
    })

    df_finalized = df[[
        "task",
        "task_normalized",
        "soc_code_2019_full",
        "soc_code_2010",
        "title_current",
        "title",
        "task_id",
        "task_type",
        "n_responding",
        "date",
        "domain_source",
        "n_occurrences",
        "pct_weighted",
        "pct_normalized",
        'impact_scope_ai',
        'impact_scope_user',
        'impact_scope_ai_adj',
        'impact_scope_user_adj',
        "dwa_title",
        "iwa_title",
        "gwa_title"
    ]].copy()




#### Eco 2025

In [49]:
if eco_run_2025:

    DATA_DIR = Path("../data")


    # -----------------------------
    # 1. Load your Eco DF and O*NET base data
    # -----------------------------
    df_eco = pd.read_csv(DATA_DIR / "task_pct_eco_2025.csv") 
    task_statements = pd.read_csv(DATA_DIR / "task_statements_v30.1.csv")
    crosswalk = pd.read_csv(DATA_DIR / "2010_to_2019_soc_crosswalk.csv")

    # -----------------------------
    # 2. Normalize and Join
    # -----------------------------
    df_eco["task_normalized"] = df_eco["task_name"].apply(normalize_text)
    task_statements["task_normalized"] = task_statements["Task"].apply(normalize_text)

    # We include the extra O*NET columns here
    df_merged = df_eco.merge(
        task_statements[[
            'task_normalized', "Task", 'Title', 'O*NET-SOC Code', 'Task ID', 
            'Task Type', 'Incumbents Responding', 'Date', 'Domain Source'
        ]],
        on="task_normalized",
        how="left"
    )

    # -----------------------------
    # 3. Crosswalk 2019 -> 2010
    # -----------------------------
    crosswalk_clean = crosswalk.drop_duplicates(subset=["O*NET-SOC 2019 Code"], keep="first")

    df_merged = df_merged.merge(
        crosswalk_clean[["O*NET-SOC 2019 Code", "O*NET-SOC 2010 Code", "O*NET-SOC 2010 Title"]],
        left_on="O*NET-SOC Code",
        right_on="O*NET-SOC 2019 Code",
        how="left"
    )

    # Decimal Fallback Logic
    df_merged["soc_2019_base"] = df_merged["O*NET-SOC Code"].astype(str).str.split(".").str[0]
    crosswalk["soc_2019_base"] = crosswalk["O*NET-SOC 2019 Code"].astype(str).str.split(".").str[0]

    crosswalk_base = crosswalk[[
        "soc_2019_base", "O*NET-SOC 2010 Code", "O*NET-SOC 2010 Title"
    ]].drop_duplicates(subset=["soc_2019_base"], keep="first")

    missing_mask = df_merged["O*NET-SOC 2010 Code"].isna()
    fallback = df_merged.loc[missing_mask, ["soc_2019_base"]].merge(
        crosswalk_base, on="soc_2019_base", how="left"
    )

    df_merged.loc[missing_mask, "O*NET-SOC 2010 Code"] = fallback["O*NET-SOC 2010 Code"].values
    df_merged.loc[missing_mask, "O*NET-SOC 2010 Title"] = fallback["O*NET-SOC 2010 Title"].values

    # -----------------------------
    # 4. Rename & Weights (Mirroring the MCP logic)
    # -----------------------------
    df = df_merged.rename(columns={
        "Task": "task",
        "pct": "pct_normalized",
        "Title": "title_current",
        "O*NET-SOC Code": "soc_code_2019_full",
        "O*NET-SOC 2010 Code": "soc_code_2010",
        "O*NET-SOC 2010 Title": "title",
        "Task ID": "task_id",
        "Task Type": "task_type",
        "Incumbents Responding": "n_responding",
        "Date": "date",
        "Domain Source": "domain_source"
    })

    # Calculate how many Job Titles share each task
    df["n_occurrences"] = (
        df.groupby("task_normalized")["title"]
        .transform("nunique")
    )

    # Even though pct_normalized is 0, we perform the calculation to keep the column structure
    df["pct_weighted"] = df["pct_normalized"] * df["n_occurrences"]
    # If everything is zero, we handle the division by zero to keep it 0.0
    total_sum = df["pct_weighted"].sum()
    if total_sum > 0:
        df["pct_weighted"] = df["pct_weighted"] / total_sum
    else:
        df["pct_weighted"] = 0.0

    # -----------------------------
    # 5. Final Selection (Exact Columns)
    # -----------------------------
    df_finalized = df[[
        "task",
        "task_normalized",
        "soc_code_2019_full",
        "soc_code_2010",
        "title_current",
        "title",
        "task_id",
        "task_type",
        "n_responding",
        "date",
        "domain_source",
        "n_occurrences",
        "pct_weighted",
        "pct_normalized"
    ]].copy()

    output_path = DATA_DIR / "enriched_task_pct_eco_2025.csv"

    print(f"Success! Finalized data with {len(df_finalized)} rows saved to {output_path}")


#### Run Function

In [50]:
if mcp_run or microsoft_run or eco_run_2025:
    pct_onet_tasks_df = df_finalized
elif eco_run_2015:
    task_statements_df = pd.read_csv("../data/task_statements_v20.1.csv")
    eco_data_2015 = pd.read_csv(eco_data_2015)
    pct_onet_tasks_df = pct_to_onet_tasks(eco_data_2015, task_statements_df)
else:
    task_statements_df = pd.read_csv("../data/task_statements_v20.1.csv")
    pct_df = pd.read_csv(pct_df)
    pct_onet_tasks_df = pct_to_onet_tasks(pct_df, task_statements_df)

if save_files_each_step:
    pct_onet_tasks_df.to_csv("../data/merged_data_files/pct_onet_tasks.csv", index=False)

## Step 2: Add SOC Major Occupational Category And Broad Counts

In [51]:
if not (mcp_run or microsoft_run or eco_run_2025):
    soc_crosswalk_df = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")
    soc_crosswalk_df = soc_crosswalk_df.rename(
        columns={
            "O*NET-SOC 2010 Title": "title",
            "O*NET-SOC 2019 Code": "onet_soc_code_2019"
        }
    )
    soc_crosswalk_df['soc_code_2019'] = soc_crosswalk_df['onet_soc_code_2019']
    # titles_df = pct_onet_tasks_df[["title", "broad_counts"]].drop_duplicates(subset=["title"])
    pct_onet_tasks_df = pct_onet_tasks_df.merge(
        soc_crosswalk_df[["title", "soc_code_2019"]],
        on="title",
        how="left"
    )
else:
    pct_onet_tasks_df["soc_code_2019"] = pct_onet_tasks_df["soc_code_2019_full"]



In [52]:
def add_soc_structure(pct_onet_tasks_df, soc_structure_df, detailed_occ_2019) -> pd.DataFrame:
    """
    Adds major_occ_category, minor_occ_category, broad_occ labels, and broad_counts
    to each row based on SOC structure data, with hard-coded fixes for known SOC anomalies.
    """

    soc_struct = soc_structure_df.copy()
    soc_struct.rename(columns={"SOC or O*NET-SOC 2019 Title": "soc_title"}, inplace=True)

    # ==========================
    # 🔧 FIX MINOR GROUP ERRORS
    # ==========================

    # Explicitly overwrite incorrect minor group codes
    soc_struct.loc[soc_struct["Minor Group"] == "15-1200", "Minor Group"] = "15-1000"
    soc_struct.loc[soc_struct["Minor Group"] == "31-1100", "Minor Group"] = "31-1000"
    soc_struct.loc[soc_struct["Minor Group"] == "51-5100", "Minor Group"] = "51-5000"

    # ==========================
    # BUILD TASK DF
    # ==========================

    df = pct_onet_tasks_df.copy()
    df["soc6"] = df["soc_code_2019"].astype(str).str[:7]
    df["major_group_code"] = df["soc6"].str[:2]
    df["minor_group_code"] = df["soc6"].str[:4] + "000"
    df["broad_code"] = df["soc6"].str[:-1] + "0"

    # ==========================================
    # 🔧 HARD-CODE BROAD FIX FOR 29-122x / 29-123x
    # ==========================================

    mask_29 = df["soc6"].str.startswith(("29-122", "29-123"))
    df.loc[mask_29, "broad_code"] = "29-1210"

    # ==========================
    # BUILD LOOKUPS
    # ==========================

    major_map = (
        soc_struct.loc[soc_struct["Major Group"].notna()]
        .drop_duplicates(subset=["Major Group"])
        .assign(key=lambda x: x["Major Group"].str[:2])
        .set_index("key")["soc_title"]
        .to_dict()
    )

    minor_map = (
        soc_struct.loc[soc_struct["Minor Group"].notna()]
        .drop_duplicates(subset=["Minor Group"])
        .set_index("Minor Group")["soc_title"]
        .to_dict()
    )

    broad_map = (
        soc_struct.loc[soc_struct["Broad Occupation"].notna()]
        .drop_duplicates(subset=["Broad Occupation"])
        .set_index("Broad Occupation")["soc_title"]
        .to_dict()
    )

    # ==========================
    # MAP LABELS
    # ==========================

    df["major_occ_category"] = df["major_group_code"].map(major_map)
    df["minor_occ_category"] = df["minor_group_code"].map(minor_map)
    df["broad_occ"] = df["broad_code"].map(broad_map)

    # ==========================
    # BROAD COUNTS
    # ==========================

    det = detailed_occ_2019.copy()
    det["soc6"] = det["O*NET-SOC 2019 Code"].str[:7]
    det["broad_code"] = det["soc6"].str[:-1] + "0"

    # Apply same 29 fix to counts
    mask_29_det = det["soc6"].str.startswith(("29-122", "29-123"))
    det.loc[mask_29_det, "broad_code"] = "29-1210"

    broad_counts_map = (
        det.groupby("broad_code")["soc6"]
        .nunique()
        .astype("Int64")
        .to_dict()
    )

    df["broad_counts"] = df["broad_code"].map(broad_counts_map)

    # Cleanup
    df.drop(columns=["soc6", "major_group_code", "minor_group_code", "broad_code", "soc_code_2019"], inplace=True)

    return df.reset_index(drop=True)

soc_structure_df = pd.read_csv("../data/soc_structure_2019.csv")
detailed_occ_2019 = pd.read_csv("../data/detailed_occ_2019.csv")
pct_tasks_soc_structure_df = add_soc_structure(pct_onet_tasks_df, soc_structure_df, detailed_occ_2019)

master_df = pd.read_csv("../data/master_pct_normalized.csv")

# Merge master_pct_normalized column onto the soc structure df
pct_tasks_soc_structure_df = pct_tasks_soc_structure_df.merge(
    master_df[['title', 'avg_total_pct_normalized']], 
    on='title', 
    how='left'
)

# Rename to your specific requested name
pct_tasks_soc_structure_df.rename(columns={'avg_total_pct_normalized': 'master_pct_normalized'}, inplace=True)

if save_files_each_step:
    pct_tasks_soc_structure_df.to_csv("../data/merged_data_files/pct_tasks_soc_structure.csv", index=False)



## Step 3: Add 2025 Wage and Employment Data

In [53]:
if mcp_run or microsoft_run or eco_run_2025:
    
    pct_tasks_soc_structure_df["soc_code_2019"] = (
        pct_tasks_soc_structure_df["soc_code_2019_full"]
        .astype(str)
        .str.split(".")
        .str[0]
    )

### 3.1: Add Updated (2019) SOC Codes

In [54]:
# Get df of updated SOC codes to merge with up to date wage and employment data

def add_updated_soc_code(pct_tasks_soc_structure_df, soc_crosswalk_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles from our main df and their corresponding O*NET-SOC 2019 code (some titles are duplicated as they get split into different SOC codes)
    This is so we can merge the wage and employment data separate from our main df and merge all at once. 

    Args:
        pct_tasks_soc_structure_df (pd.DataFrame): DataFrame from previous step.
        soc_crosswalk_df (pd.DataFrame): DataFrame 2010 and 2019 occupation titles and SOC codes

    Returns:
        pd.DataFrame: DataFrame with an added 'soc_code_2019' column.
    """

    # Rename columns
    soc_crosswalk_df = soc_crosswalk_df.rename(
        columns={
            "O*NET-SOC 2010 Title": "title",
            "O*NET-SOC 2019 Code": "onet_soc_code_2019"
        }
    )

    soc_crosswalk_df['soc_code_2019'] = soc_crosswalk_df['onet_soc_code_2019'].str[:7]

    # Get unique titles from rolling DataFrame
    titles_df = pct_tasks_soc_structure_df[["title", "broad_counts"]].drop_duplicates(subset=["title"])

    # Merge to attach 2019 SOC codes
    merged = titles_df.merge(
        soc_crosswalk_df[["title", "soc_code_2019"]],
        on="title",
        how="left"
    )

    return merged


if mcp_run or microsoft_run or eco_run_2025:
    title_and_2019_soc_df = pct_tasks_soc_structure_df[["title", "soc_code_2019", "broad_counts"]]
else:
    soc_crosswalk_df = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")
    title_and_2019_soc_df = add_updated_soc_code(pct_tasks_soc_structure_df, soc_crosswalk_df)

if save_files_each_step:
    title_and_2019_soc_df.to_csv("../data/merged_data_files/title_and_2019_soc.csv", index=False)

In [55]:
title_and_2019_soc_df

,title,soc_code_2019,broad_counts
0,Geospatial Information Scientists and Technolo...,15-1299,1
1,Database Administrators,15-1243,4
2,Business Intelligence Analysts,15-2051,1
3,Computer Systems Engineers/Architects,15-1299,1
4,Computer Systems Engineers/Architects,15-1299,1
...,...,...,...
9678,Actors,27-2011,2
9679,Writers and Authors,27-3043,3
9680,"Mail Clerks and Mail Machine Operators, Except...",43-9051,1
9681,"Managers, All Other",11-9072,2


### 3.2: Add 2025 National Wage Data

In [56]:
def add_nat_wage_2025(title_and_2019_soc_df, nat_wage_df, scraped_wage_df, inflation_fac) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their national annual and hourly median salary from 2025. 
    It also includes a 6 (from previous df) & 5 digit SOC code for use in following merging. 

    Args:
        title_and_2019_soc_df (pd.DataFrame): DataFrame from previous step.
        nat_wage_df (pd.DataFrame): DataFrame of OEWS data from 2025.
        scraped_wage_df (pd.DataFrame): DataFrame containing scraped wage data from O*NET's website from Jan 2020 

    Returns:
        pd.DataFrame: DataFrame with national wage data from 2025 added
    """

     # Get only columns needed
    wage_df_trimmed = nat_wage_df[["OCC_CODE", "O_GROUP", "H_MEDIAN", "A_MEDIAN"]].copy()
    wage_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2019"}, inplace=True)

    # Change wage columns to floats
    for c in ["H_MEDIAN", "A_MEDIAN"]:
        wage_df_trimmed[c] = pd.to_numeric(wage_df_trimmed[c], errors="coerce")

    # Initial merge on detailed SOC codes
    merged = title_and_2019_soc_df.merge(
        wage_df_trimmed, 
        on="soc_code_2019", 
        how="left"
    )

    # Get 5 digit SOC codes for broad groups to merge on
    merged["5_digit_soc"] = merged["soc_code_2019"].astype(str).str[:6]     
    wage_df_trimmed["5_digit_soc"] = wage_df_trimmed["soc_code_2019"].astype(str).str[:6]

    #Create fallback DataFrames with only broad groups and where median values are missing
    wage_df_trimmed_fallback_1st = wage_df_trimmed[wage_df_trimmed["O_GROUP"] == "broad"]
    merged_fallback_1st = merged[merged["H_MEDIAN"].isna() | merged["A_MEDIAN"].isna()]

    # Create fallback df with broad group wages
    fallback_merge = merged_fallback_1st.merge(
        wage_df_trimmed_fallback_1st[["5_digit_soc", "H_MEDIAN", "A_MEDIAN"]],
        on="5_digit_soc", how="left",
        suffixes=("", "_fallback")
    )

    # Make titles unique so we don't create a Cartesian product when merging into main DataFrame
    fallback_merge_unique_titles = fallback_merge.drop_duplicates(subset="title")

    # Merge fallback data into the main dataframe
    merged = merged.merge(
        fallback_merge_unique_titles[["title", "H_MEDIAN_fallback", "A_MEDIAN_fallback"]],
        on="title",
        how="left"
    )

    # Fill missing median values from fallback columns
    merged["H_MEDIAN"] = merged["H_MEDIAN"].fillna(merged["H_MEDIAN_fallback"])
    merged["A_MEDIAN"] = merged["A_MEDIAN"].fillna(merged["A_MEDIAN_fallback"])

    # Create column to merge on and where annual median is missing
    scraped_wage_df["title"] = scraped_wage_df["JobName"]
    merged_fallback_2nd = merged[merged["H_MEDIAN"].isna() & merged["A_MEDIAN"].isna()]

    # Create 2nd fallback df with scraper wage data
    fallback_merge_2nd = merged_fallback_2nd.merge(
        scraped_wage_df[["title", "MedianSalary"]],
        on="title", how="left",
    )

    # Make titles unique so we don't create a Cartesian product when merging into main DataFrame
    fallback_merge_2nd_unique_titles = fallback_merge_2nd.drop_duplicates(subset="title")

    # Merge 2nd fallback data into the main dataframe
    merged = merged.merge(
        fallback_merge_2nd_unique_titles[["title", "MedianSalary"]],
        on="title",
        how="left"
    )

    # Fill missing median values from scraper median columns and make present value due to inflation
    inflation_factor = inflation_fac
    merged["A_MEDIAN"] = merged["A_MEDIAN"].fillna(merged["MedianSalary"] * inflation_factor)

    # Fill missing annual median using hourly median * 2080 (52 weeks * 40 hours)
    merged.loc[merged["A_MEDIAN"].isna() & merged["H_MEDIAN"].notna(), "A_MEDIAN"] = (
        merged["H_MEDIAN"] * 2080
    )

    # Fill missing hourly median using annual median / 2080
    merged.loc[merged["H_MEDIAN"].isna() & merged["A_MEDIAN"].notna(), "H_MEDIAN"] = (
        merged["A_MEDIAN"] / 2080
    )

    # Create final national wage columns by averaging for any duplicate titles and drop uneeded columns. 
    merged["h_median_national"] = merged.groupby("title")["H_MEDIAN"].transform("mean")
    merged["a_median_national"] = merged.groupby("title")["A_MEDIAN"].transform("mean")
    merged.drop(columns=["H_MEDIAN", "A_MEDIAN", "H_MEDIAN_fallback", "A_MEDIAN_fallback", "MedianSalary", "O_GROUP"], inplace=True)

    return merged.reset_index(drop=True)


nat_wage_2025_df = pd.read_csv("../data/oews_national_2025.csv")
scraped_wage_df = pd.read_csv("../data/scraped_wage_data.csv")
titles_and_nat_wage_2025_df = add_nat_wage_2025(title_and_2019_soc_df, nat_wage_2025_df, scraped_wage_df, jan_2020_inflation_factor)

if save_files_each_step:
    titles_and_nat_wage_2025_df.to_csv("../data/merged_data_files/titles_and_nat_wage_2025.csv", index=False)


### 3.3: Add 2025 State Wage Data

In [57]:
# Build state name -> abbreviation mapping from the OEWS data
_state_df = pd.read_csv("../data/oews_states_2025.csv", usecols=["AREA_TITLE", "PRIM_STATE"])
STATE_ABBREV = (
    _state_df.drop_duplicates(subset="AREA_TITLE")
    .set_index("AREA_TITLE")["PRIM_STATE"]
    .str.lower()
    .to_dict()
)
ALL_STATES = list(STATE_ABBREV.keys())   # full names used for filtering
del _state_df
print(f"{len(ALL_STATES)} states/territories: {list(STATE_ABBREV.values())}")

54 states/territories: ['al', 'ak', 'az', 'ar', 'ca', 'co', 'ct', 'de', 'dc', 'fl', 'ga', 'hi', 'id', 'il', 'in', 'ia', 'ks', 'ky', 'la', 'me', 'md', 'ma', 'mi', 'mn', 'ms', 'mo', 'mt', 'ne', 'nv', 'nh', 'nj', 'nm', 'ny', 'nc', 'nd', 'oh', 'ok', 'or', 'pa', 'ri', 'sc', 'sd', 'tn', 'tx', 'ut', 'vt', 'va', 'wa', 'wv', 'wi', 'wy', 'gu', 'pr', 'vi']


In [58]:
def add_state_wage_2025(titles_and_nat_wage_df, state_wage_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their state annual and hourly median salary from 2025
    for all states/territories.

    Args:
        titles_and_nat_wage_df (pd.DataFrame): DataFrame from previous step.
        state_wage_df (pd.DataFrame): DataFrame of OEWS data from 2025 with state level breakdown

    Returns:
        pd.DataFrame: DataFrame with state wage data from 2025 added (columns per state)
    """

    merged = titles_and_nat_wage_df.copy()

    for state_name, abbrev in STATE_ABBREV.items():
        # Get only columns needed, filtered to this state
        wage_df_trimmed = state_wage_df[["OCC_CODE", "H_MEDIAN", "A_MEDIAN", "AREA_TITLE"]].copy()
        wage_df_trimmed = wage_df_trimmed[wage_df_trimmed["AREA_TITLE"] == state_name]
        wage_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2019",
                                        "H_MEDIAN": "h_median_state",
                                        "A_MEDIAN": "a_median_state"}, inplace=True)

        # Change wage columns to floats
        for c in ["h_median_state", "a_median_state"]:
            wage_df_trimmed[c] = pd.to_numeric(wage_df_trimmed[c], errors="coerce")

        # Merge on detailed SOC codes
        merged = merged.merge(
            wage_df_trimmed,
            on="soc_code_2019",
            how="left"
        )

        # Fill missing annual median using hourly median * 2080 (52 weeks * 40 hours)
        merged.loc[merged["a_median_state"].isna() & merged["h_median_state"].notna(), "a_median_state"] = (
            merged["h_median_state"] * 2080
        )

        # Fill missing hourly median using annual median / 2080
        merged.loc[merged["h_median_state"].isna() & merged["a_median_state"].notna(), "h_median_state"] = (
            merged["a_median_state"] / 2080
        )

        # Fill remaining missing values with national data
        merged.loc[merged["a_median_state"].isna(), "a_median_state"] = merged["a_median_national"]
        merged.loc[merged["h_median_state"].isna(), "h_median_state"] = merged["h_median_national"]

        # Aggregate to title level and store as named columns
        merged[f"h_median_{abbrev}"] = merged.groupby("title")["h_median_state"].transform("mean")
        merged[f"a_median_{abbrev}"] = merged.groupby("title")["a_median_state"].transform("mean")
        merged.drop(columns=["h_median_state", "a_median_state", "AREA_TITLE"], inplace=True)

    return merged


state_wage_df_2025 = pd.read_csv("../data/oews_states_2025.csv")
titles_nat_and_state_wage_2025_df = add_state_wage_2025(titles_and_nat_wage_2025_df, state_wage_df_2025)

if save_files_each_step:
    titles_nat_and_state_wage_2025_df.to_csv("../data/merged_data_files/titles_nat_and_state_wage_2025.csv", index=False)


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3108992563.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged[f"a_median_{abbrev}"] = merged.groupby("title")["a_median_state"].transform("mean")
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3108992563.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged[f"h_median_{abbrev}"] = merged.groupby("title")["h_median_state"].transform("mean")
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3108992563.py:51: PerformanceWarning: DataFrame is highly fragmented.  Thi

### 3.4: Add 2025 National Employment Data

In [59]:
def add_nat_emp_2025(titles_nat_and_state_wage_df, nat_emp_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their national employment data from 2025.  

    Args:
        titles_nat_and_state_wage_df (pd.DataFrame): DataFrame from previous step.
        nat_emp_df (pd.DataFrame): DataFrame of OEWS data from 2025.

    Returns:
        pd.DataFrame: DataFrame with national employment data from 2025 added
    """

     # Get only columns needed
    emp_df_trimmed = nat_emp_df[["OCC_CODE", "TOT_EMP", "O_GROUP"]].copy()
    emp_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2019"}, inplace=True)

    # Change emp columns to floats
    emp_df_trimmed["TOT_EMP"] = pd.to_numeric(emp_df_trimmed["TOT_EMP"], errors="coerce")

    # Initial merge on detailed SOC codes
    merged = titles_nat_and_state_wage_df.merge(
        emp_df_trimmed, 
        on="soc_code_2019", 
        how="left"
    )

    # Get 5 digit SOC codes for broad groups to merge on  
    emp_df_trimmed["5_digit_soc"] = emp_df_trimmed["soc_code_2019"].astype(str).str[:6]

    #Create fallback DataFrames with only broad groups and where median values are missing
    emp_df_trimmed_fallback_1st = emp_df_trimmed[emp_df_trimmed["O_GROUP"] == "broad"]
    merged_fallback_1st = merged[merged["TOT_EMP"].isna()]

    # Create fallback df with broad group wages
    fallback_merge = merged_fallback_1st.merge(
        emp_df_trimmed_fallback_1st[["5_digit_soc", "TOT_EMP"]],
        on="5_digit_soc", how="left",
        suffixes=("", "_fallback")
    )

    fallback_merge["TOT_EMP_fallback"] = fallback_merge["TOT_EMP_fallback"] / fallback_merge["broad_counts"]

    # Make titles unique so we don't create a Cartesian product when merging into main DataFrame
    fallback_merge_unique_titles = fallback_merge.drop_duplicates(subset="title")

    # Merge fallback data into the main dataframe
    merged = merged.merge(
        fallback_merge_unique_titles[["title", "TOT_EMP_fallback"]],
        on="title",
        how="left"
    )

    # Fill missing emp values from fallback columns
    merged["TOT_EMP"] = merged["TOT_EMP"].fillna(merged["TOT_EMP_fallback"])

    # After the initial merge, count how many titles share each SOC code
    soc_title_counts = merged.groupby("soc_code_2019")["title"].transform("nunique")

    # Divide employment by number of titles sharing that SOC code
    merged["TOT_EMP"] = merged["TOT_EMP"] / soc_title_counts

    # Create final national emp columns by dividing by number of occurences for each soc code and summing per occupation. 
    title_counts = merged.groupby("title")["soc_code_2019"].transform("count")
    merged["TOT_EMP_adj"] = merged["TOT_EMP"] / title_counts
    merged["emp_total_national"] = merged.groupby("title")["TOT_EMP_adj"].transform("sum")

    merged.drop(columns=["TOT_EMP_fallback", "TOT_EMP", "O_GROUP", "TOT_EMP_adj", "broad_counts"], inplace=True)
    return merged.reset_index(drop=True)


nat_emp_df_2025 = pd.read_csv("../data/oews_national_2025.csv")
titles_wage_nat_emp_2025_df = add_nat_emp_2025(titles_nat_and_state_wage_2025_df, nat_emp_df_2025)

if save_files_each_step:
    titles_wage_nat_emp_2025_df.to_csv("../data/merged_data_files/titles_wage_nat_emp_2025.csv", index=False)

C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2390932461.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["TOT_EMP_adj"] = merged["TOT_EMP"] / title_counts
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2390932461.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["emp_total_national"] = merged.groupby("title")["TOT_EMP_adj"].transform("sum")


### 3.5: Add 2025 State Employment Data

In [60]:
def add_state_emp_2025(titles_wage_nat_emp_df, state_emp_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their state employment data from 2025
    for all states/territories.

    Args:
        titles_wage_nat_emp_df (pd.DataFrame): DataFrame from previous step.
        state_emp_df (pd.DataFrame): DataFrame of OEWS data from 2025.

    Returns:
        pd.DataFrame: DataFrame with state employment data from 2025 added (columns per state)
    """

    # Change emp column to float once
    state_emp_df["TOT_EMP"] = pd.to_numeric(state_emp_df["TOT_EMP"], errors="coerce")

    merged = titles_wage_nat_emp_df.copy()

    for state_name, abbrev in STATE_ABBREV.items():
        # Get only columns needed, filtered to this state
        emp_df_trimmed = state_emp_df[["OCC_CODE", "TOT_EMP", "AREA_TITLE"]].copy()
        emp_df_trimmed = emp_df_trimmed[emp_df_trimmed["AREA_TITLE"] == state_name]
        emp_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2019"}, inplace=True)

        # Merge on detailed SOC codes
        merged = merged.merge(
            emp_df_trimmed,
            on="soc_code_2019",
            how="left"
        )

        # Fill missing values with national data * state share of employment
        total_nat_emp = state_emp_df.loc[state_emp_df["OCC_CODE"] == "00-0000", "TOT_EMP"].sum()
        total_state_emp = state_emp_df.loc[
            (state_emp_df["OCC_CODE"] == "00-0000") & (state_emp_df["AREA_TITLE"] == state_name), "TOT_EMP"
        ]
        if total_state_emp.empty:
            state_share = 0.0
        else:
            state_share = float(total_state_emp.iloc[0]) / float(total_nat_emp)
        merged.loc[merged["TOT_EMP"].isna(), "TOT_EMP"] = (
            (merged["emp_total_national"] * state_share).round()
        )

        # Aggregate to title level
        title_counts = merged.groupby("title")["soc_code_2019"].transform("count")
        merged["TOT_EMP_adj"] = merged["TOT_EMP"] / title_counts
        merged[f"emp_total_{abbrev}"] = merged.groupby("title")["TOT_EMP_adj"].transform("sum")

        merged.drop(columns=["TOT_EMP", "AREA_TITLE", "TOT_EMP_adj"], inplace=True)

    return merged.reset_index(drop=True)


state_emp_2025_df = pd.read_csv("../data/oews_states_2025.csv")
titles_wage_all_emp_2025_df = add_state_emp_2025(titles_wage_nat_emp_2025_df, state_emp_2025_df)

if save_files_each_step:
    titles_wage_all_emp_2025_df.to_csv("../data/merged_data_files/titles_wage_all_emp_2025.csv", index=False)

In [61]:
titles_wage_all_emp_2025_df

,title,soc_code_2019,5_digit_soc,h_median_national,a_median_national,h_median_al,a_median_al,h_median_ak,a_median_ak,h_median_az,...,emp_total_ut,emp_total_vt,emp_total_va,emp_total_wa,emp_total_wv,emp_total_wi,emp_total_wy,emp_total_gu,emp_total_pr,emp_total_vi
0,Geospatial Information Scientists and Technolo...,15-1299,15-129,56.050000,116580.000000,51.120000,106330.000000,50.940000,105960.000000,50.600000,...,6680.000000,1320.000000,17000.000000,14360.000000,3380.000000,4680.000000,170.000000,280.000000,880.000000,40.000000
1,Database Administrators,15-1243,15-124,60.050000,124899.069767,48.372558,100615.116279,53.344884,110955.348837,64.622558,...,868.604651,105.465116,5544.651163,2462.558140,170.232558,813.023256,134.883721,21.000000,106.046512,10.000000
2,Business Intelligence Analysts,15-2051,15-205,57.800000,120230.000000,49.580000,103120.000000,40.440000,84110.000000,51.560000,...,4360.000000,200.000000,8920.000000,9600.000000,390.000000,4370.000000,157.000000,37.000000,100.000000,18.000000
3,Computer Systems Engineers/Architects,15-1299,15-129,56.050000,116580.000000,51.120000,106330.000000,50.940000,105960.000000,50.600000,...,6680.000000,1320.000000,17000.000000,14360.000000,3380.000000,4680.000000,170.000000,280.000000,880.000000,40.000000
4,Computer Systems Engineers/Architects,15-1299,15-129,56.050000,116580.000000,51.120000,106330.000000,50.940000,105960.000000,50.600000,...,6680.000000,1320.000000,17000.000000,14360.000000,3380.000000,4680.000000,170.000000,280.000000,880.000000,40.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9678,Actors,27-2011,27-201,29.050000,81670.000000,24.710000,51396.800000,29.050000,81670.000000,21.160000,...,630.000000,107.000000,1444.000000,1248.000000,247.000000,260.000000,99.000000,23.000000,330.000000,11.000000
9679,Writers and Authors,27-3043,27-304,36.980000,76910.000000,32.190000,66950.000000,36.980000,76910.000000,26.920000,...,410.000000,110.000000,1530.000000,770.000000,110.000000,710.000000,43.000000,10.000000,90.000000,5.000000
9680,"Mail Clerks and Mail Machine Operators, Except...",43-9051,43-905,18.890000,39280.000000,17.400000,36190.000000,18.890000,39280.000000,18.400000,...,460.000000,120.000000,1610.000000,1360.000000,270.000000,1310.000000,90.000000,23.000000,160.000000,11.000000
9681,"Managers, All Other",11-9072,11-907,44.101429,91734.285714,40.818571,84904.285714,41.431071,86171.428571,42.596429,...,8984.285714,762.857143,23640.357143,19706.785714,1396.428571,11655.714286,664.642857,330.821429,1303.928571,104.392857


### 3.6: Merge 2025 Wage and Employment Data Into Task Data

In [62]:
def wage_emp_to_tasks_2025(titles_wage_all_emp_df, pct_tasks_soc_structure_df) -> pd.DataFrame:
    """
    Returns DataFrame with our wage and employment data from 2025 added to our task data.  

    Args:
        titles_wage_all_emp_df (pd.DataFrame): DataFrame from previous step.
        pct_tasks_soc_structure_df (pd.DataFrame): DataFrame from step 2

    Returns:
        pd.DataFrame: DataFrame with wage and employment data from 2025 added to task data
    """

    titles_wage_all_emp_df = titles_wage_all_emp_df.drop_duplicates(subset="title").copy()

    titles_wage_all_emp_df.drop(columns=["5_digit_soc", "soc_code_2019"], inplace=True)

    merged = pct_tasks_soc_structure_df.merge(
        titles_wage_all_emp_df,
        on="title",
        how="left"
    )

    # Build rename dict: national + all states
    rename_map = {
        "h_median_national": "h_med_nat_2025",
        "a_median_national": "a_med_nat_2025",
        "emp_total_national": "emp_tot_nat_2025",
    }
    for abbrev in STATE_ABBREV.values():
        rename_map[f"h_median_{abbrev}"] = f"h_med_{abbrev}_2025"
        rename_map[f"a_median_{abbrev}"] = f"a_med_{abbrev}_2025"
        rename_map[f"emp_total_{abbrev}"] = f"emp_tot_{abbrev}_2025"

    merged.rename(columns=rename_map, inplace=True)
    
    return merged


task_wage_emp_2025_df = wage_emp_to_tasks_2025(titles_wage_all_emp_2025_df, pct_tasks_soc_structure_df)

if save_files_each_step:
    task_wage_emp_2025_df.to_csv("../data/merged_data_files/task_wage_emp_2025.csv", index=False)

In [63]:
task_wage_emp_2025_df

,task,task_normalized,soc_code_2019_full,soc_code_2010,title_current,title,task_id,task_type,n_responding,date,...,emp_tot_ut_2025,emp_tot_vt_2025,emp_tot_va_2025,emp_tot_wa_2025,emp_tot_wv_2025,emp_tot_wi_2025,emp_tot_wy_2025,emp_tot_gu_2025,emp_tot_pr_2025,emp_tot_vi_2025
0,"Document, design, code, or test Geographic Inf...",document design code or test geographic inform...,15-1299.02,15-1199.04,Geographic Information Systems Technologists a...,Geospatial Information Scientists and Technolo...,21730.0,Core,NaN,11/2020,...,6680.000000,1320.000000,17000.000000,14360.000000,3380.000000,4680.000000,170.000000,280.000000,880.000000,40.000000
1,Develop and document database architectures.,develop and document database architectures,15-1243.00,15-1141.00,Database Architects,Database Administrators,16115.0,Core,21.0,08/2023,...,868.604651,105.465116,5544.651163,2462.558140,170.232558,813.023256,134.883721,21.000000,106.046512,10.000000
2,Create or review technical design documentatio...,create or review technical design documentatio...,15-2051.01,15-1199.08,Business Intelligence Analysts,Business Intelligence Analysts,16137.0,Core,22.0,08/2024,...,4360.000000,200.000000,8920.000000,9600.000000,390.000000,4370.000000,157.000000,37.000000,100.000000,18.000000
3,Provide technical guidance or support for the ...,provide technical guidance or support for the ...,15-1299.08,15-1199.02,Computer Systems Engineers/Architects,Computer Systems Engineers/Architects,14672.0,Core,22.0,08/2024,...,6680.000000,1320.000000,17000.000000,14360.000000,3380.000000,4680.000000,170.000000,280.000000,880.000000,40.000000
4,"Document design specifications, installation i...",document design specifications installation in...,15-1299.08,15-1199.02,Computer Systems Engineers/Architects,Computer Systems Engineers/Architects,14668.0,Core,22.0,08/2024,...,6680.000000,1320.000000,17000.000000,14360.000000,3380.000000,4680.000000,170.000000,280.000000,880.000000,40.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9678,"Work closely with directors, other actors, and...",work closely with directors other actors and p...,27-2011.00,27-2011.00,Actors,Actors,7644.0,Core,50.0,07/2017,...,630.000000,107.000000,1444.000000,1248.000000,247.000000,260.000000,99.000000,23.000000,330.000000,11.000000
9679,"Work with staff to develop script, story, or a...",work with staff to develop script story or adv...,27-3043.00,27-3043.00,Writers and Authors,Writers and Authors,22669.0,NaN,NaN,11/2020,...,410.000000,110.000000,1530.000000,770.000000,110.000000,710.000000,43.000000,10.000000,90.000000,5.000000
9680,"Wrap packages or bundles by hand, or by using ...",wrap packages or bundles by hand or by using t...,43-9051.00,43-9051.00,"Mail Clerks and Mail Machine Operators, Except...","Mail Clerks and Mail Machine Operators, Except...",13268.0,Core,84.0,07/2016,...,460.000000,120.000000,1610.000000,1360.000000,270.000000,1310.000000,90.000000,23.000000,160.000000,11.000000
9681,Write and present strategies for recreational ...,write and present strategies for recreational ...,11-9072.00,11-9199.00,"Entertainment and Recreation Managers, Except ...","Managers, All Other",21415.0,Core,52.0,08/2025,...,8984.285714,762.857143,23640.357143,19706.785714,1396.428571,11655.714286,664.642857,330.821429,1303.928571,104.392857


## Step 4: Add 2015 Wage and Employment Data

### 4.1: Add 2015 National Wage Data

In [64]:
def add_nat_wage_2015(pct_tasks_soc_structure_df, nat_wage_df, inflation_fac) -> pd.DataFrame:
    """
    Creates a DataFrame of titles and their 2010 SOC codes
    Returns DataFrame with occupation titles along with their national annual and hourly median salary from 2015 in real and nominal terms merged with titles and SOC codes. 
    It also includes a 5 digit SOC code for use in following merging. 

    Args:
        pct_tasks_soc_structure_df (pd.DataFrame): DataFrame from Step 2.
        nat_wage_df (pd.DataFrame): DataFrame of OEWS data from 2015 

    Returns:
        pd.DataFrame: DataFrame with national wage data from 2015 added
    """

    # Make df with titles and SOC codes
    title_soc_code_2010_df = pct_tasks_soc_structure_df[["title", "soc_code_2010", "broad_counts"]].drop_duplicates(subset="title").copy()
    title_soc_code_2010_df.reset_index(drop=True, inplace=True)
    title_soc_code_2010_df['soc_code_2010'] = title_soc_code_2010_df['soc_code_2010'].str[:7]

    # Get only columns needed
    wage_df_trimmed = nat_wage_df[["OCC_CODE", "OCC_GROUP", "H_MEDIAN", "A_MEDIAN", "H_MEAN", "A_MEAN"]].copy()
    wage_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2010"}, inplace=True)

    # Change wage columns to floats
    for c in ["H_MEDIAN", "A_MEDIAN", "H_MEAN", "A_MEAN"]:
        wage_df_trimmed[c] = pd.to_numeric(wage_df_trimmed[c], errors="coerce")

    # Initial merge on detailed SOC codes
    merged = title_soc_code_2010_df.merge(
        wage_df_trimmed, 
        on="soc_code_2010", 
        how="left"
    )

    # Fill missing annual median using hourly median * 2080 (52 weeks * 40 hours)
    merged.loc[merged["A_MEDIAN"].isna() & merged["H_MEDIAN"].notna(), "A_MEDIAN"] = (
        merged["H_MEDIAN"] * 2080
    )

    # Fill missing hourly median using annual median / 2080
    merged.loc[merged["H_MEDIAN"].isna() & merged["A_MEDIAN"].notna(), "H_MEDIAN"] = (
        merged["A_MEDIAN"] / 2080
    )

    # Get 5 digit SOC codes for broad groups to merge on
    merged["5_digit_soc"] = merged["soc_code_2010"].astype(str).str[:6]     
    wage_df_trimmed["5_digit_soc"] = wage_df_trimmed["soc_code_2010"].astype(str).str[:6]

    #Create fallback DataFrames with only broad groups and where median values are missing
    wage_df_trimmed_fallback_1st = wage_df_trimmed[wage_df_trimmed["OCC_GROUP"] == "broad"]
    merged_fallback_1st = merged[merged["H_MEDIAN"].isna() | merged["A_MEDIAN"].isna()]

    # Create fallback df with broad group wages
    fallback_merge = merged_fallback_1st.merge(
        wage_df_trimmed_fallback_1st[["5_digit_soc", "H_MEDIAN", "A_MEDIAN"]],
        on="5_digit_soc", how="left",
        suffixes=("", "_fallback")
    )

    # Make titles unique so we don't create a Cartesian product when merging into main DataFrame
    fallback_merge_unique_titles = fallback_merge.drop_duplicates(subset="title")

    # Merge fallback data into the main dataframe
    merged = merged.merge(
        fallback_merge_unique_titles[["title", "H_MEDIAN_fallback", "A_MEDIAN_fallback"]],
        on="title",
        how="left"
    )

    # Fill missing median values from fallback columns
    merged["H_MEDIAN"] = merged["H_MEDIAN"].fillna(merged["H_MEDIAN_fallback"])
    merged["A_MEDIAN"] = merged["A_MEDIAN"].fillna(merged["A_MEDIAN_fallback"])

    # Fill missing median values from mean columns
    merged["H_MEDIAN"] = merged["H_MEDIAN"].fillna(merged["H_MEAN"])
    merged["A_MEDIAN"] = merged["A_MEDIAN"].fillna(merged["A_MEAN"])

    # Rename and drop columns for cleanup 
    merged.rename(columns={"H_MEDIAN": "h_med_nat_nominal"}, inplace=True)
    merged.rename(columns={"A_MEDIAN": "a_med_nat_nominal"}, inplace=True)
    merged.drop(columns=["H_MEDIAN_fallback", "A_MEDIAN_fallback", "H_MEAN", "A_MEAN", "OCC_GROUP"], inplace=True)

    # Make present value column for inflation
    inflation_factor = inflation_fac
    merged["h_med_nat_real"] = merged["h_med_nat_nominal"] * inflation_factor
    merged["a_med_nat_real"] = merged["a_med_nat_nominal"] * inflation_factor

    return merged.reset_index(drop=True)


nat_wage_df_2015 = pd.read_csv("../data/oews_national_2015.csv")
titles_and_nat_wage_2015_df = add_nat_wage_2015(pct_tasks_soc_structure_df, nat_wage_df_2015, may_2015_inflation_factor)

if save_files_each_step:
    titles_and_nat_wage_2015_df.to_csv("../data/merged_data_files/titles_and_nat_wage_2015.csv", index=False)

### 4.2: Add 2015 State Wage Data

In [65]:
def add_state_wage_2015(titles_and_nat_wage_df, state_wage_df, inflation_fac) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their state annual and hourly median salary from 2015 in nominal and real terms. 

    Args:
        titles_and_nat_wage_df (pd.DataFrame): DataFrame from previous step.
        state_wage_df (pd.DataFrame): DataFrame of OEWS data from 2015 with state level breakdown

    Returns:
        pd.DataFrame: DataFrame with state wage data from 2015 added
    """

    # Get only columns needed
    wage_df_trimmed = state_wage_df[["OCC_CODE", "H_MEDIAN", "A_MEDIAN", "ST"]].copy()
    wage_df_trimmed = wage_df_trimmed[wage_df_trimmed["ST"] == "UT"]
    wage_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2010",
                                    "H_MEDIAN": "h_median_state",
                                    "A_MEDIAN": "a_median_state"}, inplace=True)

    # Change wage columns to floats
    for c in ["h_median_state", "a_median_state"]:
        wage_df_trimmed[c] = pd.to_numeric(wage_df_trimmed[c], errors="coerce")

    # Initial merge on detailed SOC codes
    merged = titles_and_nat_wage_df.merge(
        wage_df_trimmed, 
        on="soc_code_2010", 
        how="left"
    )

    # Fill missing annual median using hourly median * 2080 (52 weeks * 40 hours)
    merged.loc[merged["a_median_state"].isna() & merged["h_median_state"].notna(), "a_median_state"] = (
        merged["h_median_state"] * 2080
    )

    # Fill missing hourly median using annual median / 2080
    merged.loc[merged["h_median_state"].isna() & merged["a_median_state"].notna(), "h_median_state"] = (
        merged["a_median_state"] / 2080
    )

    # Fill remaining missing values with national data
    merged.loc[merged["a_median_state"].isna(), "a_median_state"] = (
        merged["a_med_nat_nominal"]
    )
    merged.loc[merged["h_median_state"].isna(), "h_median_state"] = (
        merged["h_med_nat_nominal"]
    )

    # Rename and drop columns for cleanup
    merged.rename(columns={"h_median_state": "h_med_utah_nominal",
                                    "a_median_state": "a_med_utah_nominal"}, inplace=True)
    merged.drop(columns=["ST"], inplace=True)

    # Make present value column for inflation
    inflation_factor = inflation_fac
    merged["h_med_utah_real"] = merged["h_med_utah_nominal"] * inflation_factor
    merged["a_med_utah_real"] = merged["a_med_utah_nominal"] * inflation_factor

    return merged.reset_index(drop=True)


state_wage_df_2015 = pd.read_csv("../data/oews_states_2015.csv")
titles_nat_and_state_wage_2015_df = add_state_wage_2015(titles_and_nat_wage_2015_df, state_wage_df_2015, may_2015_inflation_factor)

if save_files_each_step:
    titles_nat_and_state_wage_2015_df.to_csv("../data/merged_data_files/titles_nat_and_state_wage_2015.csv", index=False)

### 4.3: Add 2015 National Employment Data

In [66]:
def add_nat_emp_2015(titles_nat_and_state_wage_df, nat_emp_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their national employment data from 2015.  

    Args:
        titles_nat_and_state_wage_df (pd.DataFrame): DataFrame from previous step.
        nat_emp_df (pd.DataFrame): DataFrame of OEWS data from 2015.

    Returns:
        pd.DataFrame: DataFrame with national employment data from 2015 added
    """

    # Get only columns needed
    emp_df_trimmed = nat_emp_df[["OCC_CODE", "TOT_EMP", "OCC_GROUP"]].copy()
    emp_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2010"}, inplace=True)

    # Change emp columns to floats
    emp_df_trimmed["TOT_EMP"] = pd.to_numeric(emp_df_trimmed["TOT_EMP"], errors="coerce")

    # Initial merge on detailed SOC codes
    merged = titles_nat_and_state_wage_df.merge(
        emp_df_trimmed, 
        on="soc_code_2010", 
        how="left"
    )

    # Get 5 digit SOC codes for broad groups to merge on  
    emp_df_trimmed["5_digit_soc"] = emp_df_trimmed["soc_code_2010"].astype(str).str[:6]

    #Create fallback DataFrames with only broad groups and where median values are missing
    emp_df_trimmed_fallback_1st = emp_df_trimmed[emp_df_trimmed["OCC_GROUP"] == "broad"]
    merged_fallback_1st = merged[merged["TOT_EMP"].isna()]

    # Create fallback df with broad group wages
    fallback_merge = merged_fallback_1st.merge(
        emp_df_trimmed_fallback_1st[["5_digit_soc", "TOT_EMP"]],
        on="5_digit_soc", how="left",
        suffixes=("", "_fallback")
    )

    fallback_merge["TOT_EMP_fallback"] = fallback_merge["TOT_EMP_fallback"] / fallback_merge["broad_counts"]

    # Create fallback df with broad group wages
    fallback_merge = merged_fallback_1st.merge(
        emp_df_trimmed_fallback_1st[["5_digit_soc", "TOT_EMP"]],
        on="5_digit_soc", how="left",
        suffixes=("", "_fallback")
    )

    # Make titles unique so we don't create a Cartesian product when merging into main DataFrame
    fallback_merge_unique_titles = fallback_merge.drop_duplicates(subset="title")

    # Merge fallback data into the main dataframe
    merged = merged.merge(
        fallback_merge_unique_titles[["title", "TOT_EMP_fallback"]],
        on="title",
        how="left"
    )

    # Fill missing emp values from fallback columns
    merged["TOT_EMP"] = merged["TOT_EMP"].fillna(merged["TOT_EMP_fallback"])

    # Rename and drop columns for cleanup
    merged.rename(columns={"TOT_EMP": "emp_tot_nat"}, inplace=True)
    merged.drop(columns=["TOT_EMP_fallback", "OCC_GROUP", "broad_counts"], inplace=True)

    return merged.reset_index(drop=True)


nat_emp_df_2015 = pd.read_csv("../data/oews_national_2015.csv")
titles_wage_nat_emp_2015_df = add_nat_emp_2015(titles_nat_and_state_wage_2015_df, nat_emp_df_2015)

if save_files_each_step:
    titles_wage_nat_emp_2015_df.to_csv("../data/merged_data_files/titles_wage_nat_emp_2015.csv", index=False)

### 4.4: Add 2015 State Employment Data

In [67]:
def add_state_emp_2015(titles_wage_nat_emp_df, state_emp_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their state employment data from 2015.  

    Args:
        titles_wage_nat_emp_df (pd.DataFrame): DataFrame from previous step.
        state_emp_df (pd.DataFrame): DataFrame of OEWS data from 2015.

    Returns:
        pd.DataFrame: DataFrame with state employment data from 2015 added
    """

    # Change emp columns to floats
    state_emp_df["TOT_EMP"] = pd.to_numeric(state_emp_df["TOT_EMP"], errors="coerce")

    # Get only columns needed
    emp_df_trimmed = state_emp_df[["OCC_CODE", "TOT_EMP", "ST"]].copy()
    emp_df_trimmed = emp_df_trimmed[emp_df_trimmed["ST"] == "UT"]
    emp_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2010"}, inplace=True)

    # Initial merge on detailed SOC codes
    merged = titles_wage_nat_emp_2015_df.merge(
        emp_df_trimmed, 
        on="soc_code_2010", 
        how="left"
    )

    # Fill remaining missing values with national data by multiplying by the proportion of state employment to national employment
    total_nat_emp = state_emp_df.loc[state_emp_df["OCC_CODE"] == "00-0000", "TOT_EMP"].sum()
    total_utah_emp = state_emp_df.loc[(state_emp_df["OCC_CODE"] == "00-0000") & (state_emp_df["ST"] == "UT"), "TOT_EMP"].iloc[0]
    utah_share = float(total_utah_emp) / float(total_nat_emp)
    merged.loc[merged["TOT_EMP"].isna(), "TOT_EMP"] = (
    (merged["emp_tot_nat"] * utah_share).round())

    # Rename and drop columns for cleanup
    merged.rename(columns={"TOT_EMP": "emp_tot_utah"}, inplace=True)
    merged.drop(columns=["ST"], inplace=True)

    return merged.reset_index(drop=True)


state_emp_2015_df = pd.read_csv("../data/oews_states_2015.csv")
titles_wage_all_emp_2015_df = add_state_emp_2015(titles_wage_nat_emp_2015_df, state_emp_2015_df)

if save_files_each_step:
    titles_wage_all_emp_2015_df.to_csv("../data/merged_data_files/titles_wage_all_emp_2015.csv", index=False)

In [68]:
titles_wage_all_emp_2015_df

,title,soc_code_2010,h_med_nat_nominal,a_med_nat_nominal,5_digit_soc,h_med_nat_real,a_med_nat_real,h_med_utah_nominal,a_med_utah_nominal,h_med_utah_real,a_med_utah_real,emp_tot_nat,emp_tot_utah
0,Geospatial Information Scientists and Technolo...,15-1199,40.98,85240.0,15-119,55.7328,115926.4,34.88,72540.0,47.4368,98654.4,223370.0,2620.0
1,Database Administrators,15-1141,39.29,81710.0,15-114,53.4344,111125.6,39.67,82520.0,53.9512,112227.2,113770.0,900.0
2,Business Intelligence Analysts,15-1199,40.98,85240.0,15-119,55.7328,115926.4,34.88,72540.0,47.4368,98654.4,223370.0,2620.0
3,Computer Systems Engineers/Architects,15-1199,40.98,85240.0,15-119,55.7328,115926.4,34.88,72540.0,47.4368,98654.4,223370.0,2620.0
4,Document Management Specialists,15-1199,40.98,85240.0,15-119,55.7328,115926.4,34.88,72540.0,47.4368,98654.4,223370.0,2620.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
871,Helpers--Carpenters,47-3012,13.41,27890.0,47-301,18.2376,37930.4,12.96,26960.0,17.6256,36665.6,37820.0,270.0
872,Tree Trimmers and Pruners,37-3013,16.10,33500.0,37-301,21.8960,45560.0,14.37,29890.0,19.5432,40650.4,40160.0,383.0
873,Excavating and Loading Machine and Dragline Op...,53-7032,19.26,40050.0,53-703,26.1936,54468.0,19.11,39740.0,25.9896,54046.4,49880.0,430.0
874,Ophthalmic Medical Technicians,29-2057,16.99,35350.0,29-205,23.1064,48076.0,16.20,33690.0,22.0320,45818.4,39160.0,270.0


### 4.5: Merge 2015 Wage and Employment Data Into Task Data

In [69]:
def wage_emp_to_tasks_2015(titles_wage_all_emp_df, pct_tasks_soc_structure_df) -> pd.DataFrame:
    """
    Returns DataFrame with our wage and employment data from 2015 added to our task data.  

    Args:
        titles_wage_all_emp_df (pd.DataFrame): DataFrame from previous step.
        pct_tasks_soc_structure_df (pd.DataFrame): DataFrame from step 2

    Returns:
        pd.DataFrame: DataFrame with wage and employment data from 2015 added to task data
    """

    titles_wage_all_emp_df = titles_wage_all_emp_df.drop_duplicates(subset="title")

    titles_wage_all_emp_df.drop(columns=["soc_code_2010", "5_digit_soc"], inplace=True)

    merged = pct_tasks_soc_structure_df.merge(
        titles_wage_all_emp_df,
        on="title",
        how="left"
    )

    merged.rename(columns={"h_med_nat_nominal": "h_med_nat_nominal_2015",
                            "a_med_nat_nominal": "a_med_nat_nominal_2015",
                            "h_med_nat_real": "h_med_nat_real_2015",
                            "a_med_nat_real": "a_med_nat_real_2015",
                            "h_med_utah_nominal": "h_med_ut_nominal_2015",
                            "a_med_utah_nominal": "a_med_ut_nominal_2015",
                            "h_med_utah_real": "h_med_ut_real_2015",
                            "a_med_utah_real": "a_med_ut_real_2015",
                            "emp_tot_nat": "emp_tot_nat_2015",
                            "emp_tot_utah": "emp_tot_ut_2015"}, inplace=True)
    
    return merged
    

tasks_all_wage_emp_df = wage_emp_to_tasks_2015(titles_wage_all_emp_2015_df, task_wage_emp_2025_df)

if save_files_each_step:
    tasks_all_wage_emp_df.to_csv("../data/merged_data_files/tasks_all_wage_emp.csv", index=False)

In [70]:
tasks_all_wage_emp_df

,task,task_normalized,soc_code_2019_full,soc_code_2010,title_current,title,task_id,task_type,n_responding,date,...,h_med_nat_nominal_2015,a_med_nat_nominal_2015,h_med_nat_real_2015,a_med_nat_real_2015,h_med_ut_nominal_2015,a_med_ut_nominal_2015,h_med_ut_real_2015,a_med_ut_real_2015,emp_tot_nat_2015,emp_tot_ut_2015
0,"Document, design, code, or test Geographic Inf...",document design code or test geographic inform...,15-1299.02,15-1199.04,Geographic Information Systems Technologists a...,Geospatial Information Scientists and Technolo...,21730.0,Core,NaN,11/2020,...,40.98,85240.0,55.7328,115926.40,34.88,72540.0,47.4368,98654.40,223370.0,2620.0
1,Develop and document database architectures.,develop and document database architectures,15-1243.00,15-1141.00,Database Architects,Database Administrators,16115.0,Core,21.0,08/2023,...,39.29,81710.0,53.4344,111125.60,39.67,82520.0,53.9512,112227.20,113770.0,900.0
2,Create or review technical design documentatio...,create or review technical design documentatio...,15-2051.01,15-1199.08,Business Intelligence Analysts,Business Intelligence Analysts,16137.0,Core,22.0,08/2024,...,40.98,85240.0,55.7328,115926.40,34.88,72540.0,47.4368,98654.40,223370.0,2620.0
3,Provide technical guidance or support for the ...,provide technical guidance or support for the ...,15-1299.08,15-1199.02,Computer Systems Engineers/Architects,Computer Systems Engineers/Architects,14672.0,Core,22.0,08/2024,...,40.98,85240.0,55.7328,115926.40,34.88,72540.0,47.4368,98654.40,223370.0,2620.0
4,"Document design specifications, installation i...",document design specifications installation in...,15-1299.08,15-1199.02,Computer Systems Engineers/Architects,Computer Systems Engineers/Architects,14668.0,Core,22.0,08/2024,...,40.98,85240.0,55.7328,115926.40,34.88,72540.0,47.4368,98654.40,223370.0,2620.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9678,"Work closely with directors, other actors, and...",work closely with directors other actors and p...,27-2011.00,27-2011.00,Actors,Actors,7644.0,Core,50.0,07/2017,...,18.80,39104.0,25.5680,53181.44,18.35,38168.0,24.9560,51908.48,50570.0,720.0
9679,"Work with staff to develop script, story, or a...",work with staff to develop script story or adv...,27-3043.00,27-3043.00,Writers and Authors,Writers and Authors,22669.0,NaN,NaN,11/2020,...,28.97,60250.0,39.3992,81940.00,18.15,37750.0,24.6840,51340.00,43380.0,640.0
9680,"Wrap packages or bundles by hand, or by using ...",wrap packages or bundles by hand or by using t...,43-9051.00,43-9051.00,"Mail Clerks and Mail Machine Operators, Except...","Mail Clerks and Mail Machine Operators, Except...",13268.0,Core,84.0,07/2016,...,13.74,28570.0,18.6864,38855.20,14.01,29140.0,19.0536,39630.40,95640.0,1620.0
9681,Write and present strategies for recreational ...,write and present strategies for recreational ...,11-9072.00,11-9199.00,"Entertainment and Recreation Managers, Except ...","Managers, All Other",21415.0,Core,52.0,08/2025,...,50.41,104850.0,68.5576,142596.00,44.06,91630.0,59.9216,124616.80,376440.0,2380.0


## Step 5: Adjust Employment Columns

In [71]:
def adjust_emp_old(tasks_all_wage_emp_df, state_emp_2015_df, state_emp_2025_df, nat_emp_df_2015, nat_emp_df_2025) -> pd.DataFrame:
    """
    Reallocates employment numbers based on the relative percent of Claude conversations, as we have some duplicate
    6 digit SOC codes but different titles  

    Args:
        tasks_all_wage_emp_df (pd.DataFrame): DataFrame from previous 4.5.

    Returns:
        pd.DataFrame: DataFrame with correct employment numbers
    """

    df = tasks_all_wage_emp_df

    # 6-digit SOC to remove decimals (e.g., '11-1011.03' -> '11-1011')
    df["soc6"] = df["soc_code_2010"].astype(str).str[:7]

    # 1. Fill NaNs with the group average (or 0 if the whole group is NaN)
    df["master_pct_normalized"] = df.groupby("soc6")["master_pct_normalized"].transform(lambda x: x.fillna(x.mean())).fillna(0)

    # 2. Dampen extreme values (Square Root) to bring values closer to the mean
    # This prevents outliers from "eating" all the employment in a SOC6 group
    df["master_pct_adj"] = np.sqrt(df["master_pct_normalized"])

    # 3. Calculate shares with a fallback to equal split
    title_val = df.groupby(["soc6", "title"])["master_pct_adj"].transform("first")
    soc6_total = df.groupby("soc6")["master_pct_adj"].transform(lambda x: x.unique().sum())
    title_counts = df.groupby("soc6")["title"].transform("nunique")

    # If the group total is 0, split equally. Otherwise, use the dampened shares.
    df["soc6_share"] = np.where(
        soc6_total > 0,
        title_val / soc6_total,
        1 / title_counts
    )

    # columns to allocate � national + all states for both years
    emp_cols = [c for c in df.columns if c.startswith("emp_tot_") and c.endswith(("_2025", "_2015"))]

    # For each employment column, check if reallocation is needed
    for c in emp_cols:
        # Get one employment value per title within each soc6
        emp_per_title = df.groupby(["soc6", "title"])[c].first().reset_index()
        
        # For each soc6, check if all titles have the same employment (duplicates from fallback)
        def has_duplicate_emp(group):
            return (group[c].nunique() == 1) and (len(group) > 1)
        
        soc6_needs_adjustment = emp_per_title.groupby("soc6").apply(has_duplicate_emp, include_groups=False)
        soc6_to_adjust = soc6_needs_adjustment[soc6_needs_adjustment].index.tolist()
        
        # Create mask for rows that need adjustment
        needs_adjustment = df["soc6"].isin(soc6_to_adjust)
        
        # Get the soc6 total by SUMMING across titles (not max!)
        emp_sum_per_title = df.groupby(["soc6", "title"])[c].first()
        soc6_sum = emp_sum_per_title.groupby("soc6").sum()
        soc6_tot_corrected = soc6_sum 
        soc6_tot = df["soc6"].map(soc6_tot_corrected)  # Map back to dataframe
        
        # Only reallocate where needed, otherwise keep original
        df[c] = np.where(
            needs_adjustment,
            round(soc6_tot * df["soc6_share"]),
            df[c]
        )

    # Create percent-of-workforce columns from the reallocated totals
    # Build pct_map and df_map dynamically for national + all states
    abbrev_to_name = {v: k for k, v in STATE_ABBREV.items()}
    pct_map = {}
    df_map = {}
    for col in emp_cols:
        pct_col = col.replace("emp_tot_", "emp_pct_")
        pct_map[col] = pct_col
        if col == "emp_tot_nat_2025":
            df_map[col] = nat_emp_df_2025
        elif col == "emp_tot_nat_2015":
            df_map[col] = nat_emp_df_2015
        elif col.endswith("_2025"):
            df_map[col] = state_emp_2025_df
        elif col.endswith("_2015"):
            df_map[col] = state_emp_2015_df

    for tot_col, pct_col in pct_map.items():
        if tot_col in df.columns:
            source_df = df_map[tot_col]
            # For national columns, use the single "All Occupations" row
            if "_nat_" in tot_col:
                total_sum = source_df[source_df["OCC_TITLE"] == "All Occupations"]["TOT_EMP"].iloc[0]
            else:
                # Extract state abbrev from column name (e.g. emp_tot_ca_2025 -> ca)
                st_abbrev = tot_col.split("_")[2]
                # 2025 data uses AREA_TITLE (full name), 2015 data uses ST (abbreviation)
                if "AREA_TITLE" in source_df.columns:
                    st_name = abbrev_to_name.get(st_abbrev, "")
                    state_rows = source_df[
                        (source_df["OCC_TITLE"] == "All Occupations") & (source_df["AREA_TITLE"] == st_name)
                    ]["TOT_EMP"]
                else:
                    state_rows = source_df[
                        (source_df["OCC_TITLE"] == "All Occupations") & (source_df["ST"] == st_abbrev.upper())
                    ]["TOT_EMP"]
                if state_rows.empty:
                    continue
                total_sum = state_rows.iloc[0]
            df[pct_col] = (df.groupby("title")[tot_col].transform("first") / total_sum) * 100

    df.drop(columns=["soc6","soc6_share"], inplace=True)
    return df


In [72]:
def adjust_emp_new(tasks_all_wage_emp_df, state_emp_2015_df, state_emp_2025_df, nat_emp_df_2015, nat_emp_df_2025) -> pd.DataFrame:
    """
    Reallocates employment numbers based on the relative percent of Claude conversations, as we have some duplicate
    6 digit SOC codes but different titles  

    Args:
        tasks_all_wage_emp_df (pd.DataFrame): DataFrame from previous 4.5.

    Returns:
        pd.DataFrame: DataFrame with correct employment numbers
    """

    df = tasks_all_wage_emp_df

    # 6-digit SOC to remove decimals (e.g., '11-1011.03' -> '11-1011')
    df["soc6"] = df["soc_code_2019_full"].astype(str).str[:7]

    # 1. Fill NaNs with the group average (or 0 if the whole group is NaN)
    df["master_pct_normalized"] = df.groupby("soc6")["master_pct_normalized"].transform(lambda x: x.fillna(x.mean())).fillna(0)

    # 2. Dampen extreme values (Square Root) to bring values closer to the mean
    df["master_pct_adj"] = np.sqrt(df["master_pct_normalized"])

    # 3. Calculate shares with a fallback to equal split
    title_val = df.groupby(["soc6", "title_current"])["master_pct_adj"].transform("first")
    soc6_total = df.groupby("soc6")["master_pct_adj"].transform(lambda x: x.unique().sum())
    title_counts = df.groupby("soc6")["title_current"].transform("nunique")

    # If the group total is 0, split equally. Otherwise, use the dampened shares.
    df["soc6_share"] = np.where(
        soc6_total > 0,
        title_val / soc6_total,
        1 / title_counts
    )

    # columns to allocate � national + all states for both years
    emp_cols = [c for c in df.columns if c.startswith("emp_tot_") and c.endswith(("_2025", "_2015"))]

    # For each employment column, check if reallocation is needed
    for c in emp_cols:
        # Get one employment value per title within each soc6
        emp_per_title = df.groupby(["soc6", "title_current"])[c].first().reset_index()
        
        # For each soc6, check if all titles have the same employment (duplicates from fallback)
        def has_duplicate_emp(group):
            return (group[c].nunique() == 1) and (len(group) > 1)
        
        soc6_needs_adjustment = emp_per_title.groupby("soc6").apply(has_duplicate_emp, include_groups=False)
        soc6_to_adjust = soc6_needs_adjustment[soc6_needs_adjustment].index.tolist()
        
        # Create mask for rows that need adjustment
        needs_adjustment = df["soc6"].isin(soc6_to_adjust)
        
        # Get the soc6 total by SUMMING across titles (not max!)
        emp_sum_per_title = df.groupby(["soc6", "title_current"])[c].first()
        soc6_sum = emp_sum_per_title.groupby("soc6").sum()
        soc6_tot_corrected = soc6_sum 
        soc6_tot = df["soc6"].map(soc6_tot_corrected)  # Map back to dataframe
        
        # Only reallocate where needed, otherwise keep original
        df[c] = np.where(
            needs_adjustment,
            round(soc6_tot * df["soc6_share"]),
            df[c]
        )

    # Create percent-of-workforce columns from the reallocated totals
    # Build pct_map and df_map dynamically for national + all states
    abbrev_to_name = {v: k for k, v in STATE_ABBREV.items()}
    pct_map = {}
    df_map = {}
    for col in emp_cols:
        pct_col = col.replace("emp_tot_", "emp_pct_")
        pct_map[col] = pct_col
        if col == "emp_tot_nat_2025":
            df_map[col] = nat_emp_df_2025
        elif col == "emp_tot_nat_2015":
            df_map[col] = nat_emp_df_2015
        elif col.endswith("_2025"):
            df_map[col] = state_emp_2025_df
        elif col.endswith("_2015"):
            df_map[col] = state_emp_2015_df

    for tot_col, pct_col in pct_map.items():
        if tot_col in df.columns:
            source_df = df_map[tot_col]
            # For national columns, use the single "All Occupations" row
            if "_nat_" in tot_col:
                total_sum = source_df[source_df["OCC_TITLE"] == "All Occupations"]["TOT_EMP"].iloc[0]
            else:
                # Extract state abbrev from column name (e.g. emp_tot_ca_2025 -> ca)
                st_abbrev = tot_col.split("_")[2]
                # 2025 data uses AREA_TITLE (full name), 2015 data uses ST (abbreviation)
                if "AREA_TITLE" in source_df.columns:
                    st_name = abbrev_to_name.get(st_abbrev, "")
                    state_rows = source_df[
                        (source_df["OCC_TITLE"] == "All Occupations") & (source_df["AREA_TITLE"] == st_name)
                    ]["TOT_EMP"]
                else:
                    state_rows = source_df[
                        (source_df["OCC_TITLE"] == "All Occupations") & (source_df["ST"] == st_abbrev.upper())
                    ]["TOT_EMP"]
                if state_rows.empty:
                    continue
                total_sum = state_rows.iloc[0]
            df[pct_col] = (df.groupby("title_current")[tot_col].transform("first") / total_sum) * 100

    df.drop(columns=["soc6","soc6_share"], inplace=True)
    return df


In [73]:

if aei_run or eco_run_2015: 
    tasks_wage_emp_final_df = adjust_emp_old(tasks_all_wage_emp_df, state_emp_2015_df, state_emp_2025_df, nat_emp_df_2015, nat_emp_df_2025)
else:
    tasks_wage_emp_final_df = adjust_emp_new(tasks_all_wage_emp_df, state_emp_2015_df, state_emp_2025_df, nat_emp_df_2015, nat_emp_df_2025)

if save_files_each_step:
    tasks_wage_emp_final_df.to_csv("../data/merged_data_files/tasks_wage_emp_final.csv", index=False)


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\39107627.py:106: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[pct_col] = (df.groupby("title_current")[tot_col].transform("first") / total_sum) * 100
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\39107627.py:106: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[pct_col] = (df.groupby("title_current")[tot_col].transform("first") / total_sum) * 100
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\39107627.py:106: PerformanceWarning: DataFrame is highly fragmented.  This is 

## 7: Cleanup On Main Data

In [74]:
def final_cleanup(df) -> pd.DataFrame:
    """
    Description:
        Does some final cleanup and imputing for our data
    
    Args:
        tasks_wage_emp_final_df (pd.DataFrame): DataFrame from Step 5
    
    Returns:
        pd.DataFrame: Final merged DataFrame with all necessary information cleaned.
    """

    # Drop None task row
    # Normalize column names
    df["task_normalized"] = df["task"].apply(normalize_text)
    df["title_normalized"] = df["title"].apply(normalize_text)

    df = df[df["task_normalized"].notna()].copy()
    df["pct_weighted"] = 100 * df["pct_weighted"] / df["pct_weighted"].sum()

    # Get current column order
    cols = df.columns.tolist()

    # Find the index of 'title' and insert 'title_normalized' right after it
    title_idx = cols.index('title')
    cols.insert(title_idx + 1, cols.pop(cols.index('title_normalized')))

    # Reorder the dataframe
    df = df[cols]

    # Fill missing n_responding values with the median within the same occupation
    df["n_responding"] = df.groupby("soc_code_2010")["n_responding"].transform(
        lambda x: x.fillna(x.median()) if not x.isna().all() else x
    )

    # Drop specific confidence interval and bound columns
    cols_to_drop = [
        "task_id", 
        "date",
        "task_type",
        "n_responding",
        "domain_source",
        "n_occurrences",
        "broad_counts",
        "pct_weighted",
        # 2015 Data Columns
        "h_med_nat_nominal_2015",
        "a_med_nat_nominal_2015",
        "h_med_nat_real_2015",
        "a_med_nat_real_2015",
        "h_med_ut_nominal_2015",
        "a_med_ut_nominal_2015",
        "h_med_ut_real_2015",
        "a_med_ut_real_2015",
        "emp_tot_nat_2015",
        "emp_tot_ut_2015",
        "emp_pct_nat_2015",
        "emp_pct_ut_2015",
        # Hourly Wage Columns (2025) � national
        "h_med_nat_2025",
        # emp pct columns � national
        "emp_pct_nat_2025",
    ]
    # Add hourly wage and emp_pct columns for all states (2025)
    for _abbrev in STATE_ABBREV.values():
        cols_to_drop.append(f"h_med_{_abbrev}_2025")
        cols_to_drop.append(f"emp_pct_{_abbrev}_2025")

    df = df.drop(columns=cols_to_drop)

    placeholder_values = ["#", "*", "", "n/a", "na", "--"]
    df.replace(placeholder_values, pd.NA, inplace=True)

    return df


tasks_final_df = final_cleanup(tasks_wage_emp_final_df)

if mcp_run:

    # Load original MCP task results
    mcp_df = pd.read_csv(mcp_data)

    # Normalize merge keys
    mcp_df["task_normalized"] = mcp_df["task"].apply(normalize_text)
    mcp_df["title_current_normalized"] = mcp_df["occupation"].apply(normalize_text)

    tasks_final_df["title_current_normalized"] = (
        tasks_final_df["title_current"].apply(normalize_text)
    )

    # Merge MCP numeric columns back in
    mcp_numeric_cols = [
        "task_normalized",
        "title_current_normalized",
        "n_ratings",
        "mean_rating",
        "median_rating",
        "max_rating",
        "min_rating",
        "p25_rating",
        "p75_rating",
        "top_mcps",
        "top_mcp_urls",
        # "mcp_titles",
    ]

    tasks_final_df = tasks_final_df.merge(
        mcp_df[mcp_numeric_cols],
        on=["task_normalized", "title_current_normalized"],
        how="left"
    )

    tasks_final_df.to_csv(output_file_main_1, index=False)
else:
    tasks_final_df.to_csv(output_file_main_1, index=False)

if save_files_each_step:
    tasks_final_df.to_csv(output_file_main_1, index=False)

C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3174195199.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["title_normalized"] = df["title"].apply(normalize_text)


In [75]:
# import pandas as pd

# file_path = "../data/TEST_1_tasks_v1.csv"

# # Read only the header
# cols = pd.read_csv(file_path, nrows=0).columns.tolist()

# print(f"Total Columns: {len(cols)}")
# print(cols)

# Main Data Part 2

### Dates, Taxonomy, Physical

In [76]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data")

file_date_map = {
    "first_pass_microsoft.csv": "2024-09-30",
    "first_pass_aei_v1.csv": "2024-12-23",
    "first_pass_aei_v2.csv": "2025-03-06",
    "first_pass_mcp_v1.csv": "2025-04-24",
    "first_pass_mcp_v2.csv": "2025-05-24",
    "first_pass_mcp_v3.csv": "2025-07-23",
    "first_pass_aei_v3.csv": "2025-08-11",
    "first_pass_aei_api_v3.csv": "2025-08-11",
    "first_pass_aei_v4.csv": "2025-11-13",
    "first_pass_aei_api_v4.csv": "2025-11-13",
    "first_pass_aei_v5.csv": "2026-02-12",
    "first_pass_aei_api_v5.csv": "2026-02-12",
    "first_pass_mcp_v4.csv": "2026-02-18",
    "first_pass_eco_tasks_2015.csv": "2015-01-01",
    "first_pass_eco_tasks_2025.csv": "2025-01-01",
}

dfs = {}

for filename, date in file_date_map.items():
    df = pd.read_csv(data_dir / filename)
    df["date"] = pd.to_datetime(date)
    dfs[filename] = df


data_dir = Path("../data")

# --- LOAD BOTH TAXONOMY FILES ---

# AEI uses v20.1
taxonomy_v20 = pd.read_csv(data_dir / "tasks_dwa_iwa_gwa_v20.1.csv")
taxonomy_v20["task_normalized"] = taxonomy_v20["task"].apply(normalize_text)
taxonomy_v20["title_normalized"] = taxonomy_v20["Title"].apply(normalize_text)

taxonomy_v20_subset = taxonomy_v20[
    ["task_normalized", "title_normalized", "dwa_title", "iwa_title", "gwa_title"]
].copy()


# MCP + Microsoft use v30.1
taxonomy_v30 = pd.read_csv(data_dir / "tasks_dwa_iwa_gwa_v30.1.csv")
taxonomy_v30["task_normalized"] = taxonomy_v30["task"].apply(normalize_text)
taxonomy_v30["title_current"] = taxonomy_v30["Title"]

taxonomy_v30_subset = taxonomy_v30[
    ["task_normalized", "title_current", "dwa_title", "iwa_title", "gwa_title"]
].copy()


# --- PROCESS EACH DATASET ---
for filename, df in dfs.items():

    print(f"\nProcessing: {filename}")
    df.drop(columns=["task_normalized"], inplace=True, errors="ignore")  # drop if exists
    df["task_normalized"] = df["task"].apply(normalize_text)

    if "aei" in filename.lower() or "2015" in filename.lower():
        # AEI merge logic
        df = df.merge(
            taxonomy_v20_subset,
            on=["task_normalized", "title_normalized"],
            how="left"
        )
    elif "microsoft" in filename.lower():
        df = df
    else:
        # MCP  merge logic
        df = df.merge(
            taxonomy_v30_subset,
            on=["task_normalized", "title_current"],
            how="left"
        )

    total = len(df)
    print(filename)
    print(total)

    print(f"DWA missing %: {df['dwa_title'].isna().mean() * 100:.2f}%")
    print(f"IWA missing %: {df['iwa_title'].isna().mean() * 100:.2f}%")
    print(f"GWA missing %: {df['gwa_title'].isna().mean() * 100:.2f}%")

    # Save with suffix
    # new_name = filename.replace(".csv", "_taxonomy.csv")
    # df.to_csv(data_dir / new_name, index=False)

    dfs[filename] = df



data_dir = Path("../data")

phys = pd.read_csv(data_dir / "tasks_dwa_iwa_gwa_v30.1_physical.csv")
phys["task_normalized"] = phys["task"].apply(normalize_text)
phys["title_normalized"] = phys["Title"].apply(normalize_text)
phys["title_current"] = phys["Title"]

def mode_bool(s: pd.Series):
    s = s.dropna()
    if s.empty:
        return pd.NA
    vc = s.value_counts()
    if len(vc) == 1:
        return vc.index[0]
    if vc.iloc[0] == vc.iloc[1]:
        return True
    return vc.idxmax()

phys_pair_aei = (
    phys.groupby(["task_normalized", "title_normalized"], as_index=False)["physical"]
    .agg(mode_bool)
)

phys_pair_non = (
    phys.groupby(["task_normalized", "title_current"], as_index=False)["physical"]
    .agg(mode_bool)
)

task_lookup = phys.groupby("task_normalized")["physical"].agg(mode_bool)

for filename, df in dfs.items():
    print(f"\nProcessing: {filename}")

    # 1) primary merge
    if "aei" in filename.lower() or "2015" in filename.lower():
        df = df.merge(
            phys_pair_aei,
            on=["task_normalized", "title_normalized"],
            how="left",
            validate="m:1",
        )
    else:
        df = df.merge(
            phys_pair_non,
            on=["task_normalized", "title_current"],
            how="left",
            validate="m:1",
        )

    total = len(df)
    print(f"After pair-merge missing %: {df['physical'].isna().mean() * 100:.2f}%")

    df["physical_imputed"] = False
    

    # 2) task-only
    mask = df["physical"].isna()
    if mask.any():
        df.loc[mask, "physical"] = df.loc[mask, "task_normalized"].map(task_lookup)
    print(f"After task-only missing %: {df['physical'].isna().mean() * 100:.2f}%")

    mask_after_task = df["physical"].isna()

    # 3) DWA-majority
    mask = df["physical"].isna()
    if mask.any():
        dwa_majority = (
            df.dropna(subset=["physical", "dwa_title"])
            .groupby("dwa_title")["physical"]
            .agg(lambda s: bool(s.mean() >= 0.5))
        )
        df.loc[mask, "physical"] = df.loc[mask, "dwa_title"].map(dwa_majority)
    print(f"After DWA-majority missing %: {df['physical'].isna().mean() * 100:.2f}%")

    # 4) OCCUPATION majority
    mask = df["physical"].isna()
    if mask.any():
        if "aei" in filename.lower() or "2015" in filename.lower():
            occ_col = "title_normalized"
        else:
            occ_col = "title_current"
        occ_majority = (
            df.dropna(subset=["physical", occ_col])
            .groupby(occ_col)["physical"]
            .agg(lambda s: bool(s.mean() >= 0.5))
        )
        df.loc[mask, "physical"] = df.loc[mask, occ_col].map(occ_majority)
    print(f"After occupation-majority missing %: {df['physical'].isna().mean() * 100:.2f}%")

    # 5) BROAD majority
    mask = df["physical"].isna()
    if mask.any():
        broad_majority = (
            df.dropna(subset=["physical", "broad_occ"])
            .groupby("broad_occ")["physical"]
            .agg(lambda s: bool(s.mean() >= 0.5))
        )
        df.loc[mask, "physical"] = df.loc[mask, "broad_occ"].map(broad_majority)
    print(f"After broad-majority missing %: {df['physical'].isna().mean() * 100:.2f}%")

    # 6) MINOR majority
    mask = df["physical"].isna()
    if mask.any() and "minor_occ_category" in df.columns:
        minor_majority = (
            df.dropna(subset=["physical", "minor_occ_category"])
            .groupby("minor_occ_category")["physical"]
            .agg(lambda s: bool(s.mean() >= 0.5))
        )
        df.loc[mask, "physical"] = df.loc[mask, "minor_occ_category"].map(minor_majority)
    print(f"After minor-majority missing %: {df['physical'].isna().mean() * 100:.2f}%")

    # 7) MAJOR majority
    mask = df["physical"].isna()
    if mask.any() and "major_occ_category" in df.columns:
        major_majority = (
            df.dropna(subset=["physical", "major_occ_category"])
            .groupby("major_occ_category")["physical"]
            .agg(lambda s: bool(s.mean() >= 0.5))
        )
        df.loc[mask, "physical"] = df.loc[mask, "major_occ_category"].map(major_majority)
    print(f"After major-majority missing %: {df['physical'].isna().mean() * 100:.2f}%")

    df.loc[mask_after_task & df["physical"].notna(), "physical_imputed"] = True

    remaining = df["physical"].isna().sum()
    if remaining:
        raise ValueError(f"{filename}: still {remaining} missing physical values after all imputations.")
    else:
        print("All physical values resolved.")
    
    out_name = filename.replace(".csv", "_dates_taxonomy_physical.csv")
    df.to_csv(data_dir / out_name, index=False)

C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4243987091.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["date"] = pd.to_datetime(date)



Processing: first_pass_microsoft.csv
first_pass_microsoft.csv
10438
DWA missing %: 0.00%
IWA missing %: 0.00%
GWA missing %: 0.00%

Processing: first_pass_aei_v1.csv
first_pass_aei_v1.csv
5481
DWA missing %: 3.72%
IWA missing %: 3.72%
GWA missing %: 3.72%

Processing: first_pass_aei_v2.csv
first_pass_aei_v2.csv
5264
DWA missing %: 3.76%
IWA missing %: 3.76%
GWA missing %: 3.76%

Processing: first_pass_mcp_v1.csv
first_pass_mcp_v1.csv
9575
DWA missing %: 0.00%
IWA missing %: 0.00%
GWA missing %: 0.00%

Processing: first_pass_mcp_v2.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["task_normalized"] = df["task"].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["task_normalized"] = df["task"].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which 

first_pass_mcp_v2.csv
11123
DWA missing %: 0.01%
IWA missing %: 0.01%
GWA missing %: 0.01%

Processing: first_pass_mcp_v3.csv
first_pass_mcp_v3.csv
12500
DWA missing %: 0.01%
IWA missing %: 0.01%
GWA missing %: 0.01%

Processing: first_pass_aei_v3.csv
first_pass_aei_v3.csv
4278
DWA missing %: 3.65%
IWA missing %: 3.65%
GWA missing %: 3.65%

Processing: first_pass_aei_api_v3.csv
first_pass_aei_api_v3.csv
3325
DWA missing %: 3.76%
IWA missing %: 3.76%
GWA missing %: 3.76%

Processing: first_pass_aei_v4.csv
first_pass_aei_v4.csv
5143
DWA missing %: 3.66%
IWA missing %: 3.66%
GWA missing %: 3.66%

Processing: first_pass_aei_api_v4.csv
first_pass_aei_api_v4.csv
3585
DWA missing %: 3.77%
IWA missing %: 3.77%
GWA missing %: 3.77%

Processing: first_pass_aei_v5.csv
first_pass_aei_v5.csv
5217
DWA missing %: 3.64%
IWA missing %: 3.64%
GWA missing %: 3.64%

Processing: first_pass_aei_api_v5.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["task_normalized"] = df["task"].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["task_normalized"] = df["task"].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which 

...

C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4243987091.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["physical_imputed"] = False



Processing: first_pass_eco_tasks_2015.csv
After pair-merge missing %: 28.14%
After task-only missing %: 12.69%
After DWA-majority missing %: 0.93%
After occupation-majority missing %: 0.00%
After broad-majority missing %: 0.00%
After minor-majority missing %: 0.00%
After major-majority missing %: 0.00%
All physical values resolved.


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4243987091.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["physical_imputed"] = False



Processing: first_pass_eco_tasks_2025.csv
After pair-merge missing %: 0.00%
After task-only missing %: 0.00%
After DWA-majority missing %: 0.00%
After occupation-majority missing %: 0.00%
After broad-majority missing %: 0.00%
After minor-majority missing %: 0.00%
After major-majority missing %: 0.00%
All physical values resolved.


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4243987091.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["physical_imputed"] = False


### Microsoft cleanup: AI-dominant physical filter + Eloundou exposure trim


In [77]:

# Two-step trim applied to the Microsoft intermediate produced above, before the rest of Part 2 continues. Filters are baked in at the per-version stage so every downstream consumer (cumulative buckets, dashboard) inherits the cleanup — there are no separate filtered output files.

# 1. **Physical filter under AI-dominant IWAs.** Drops Microsoft rows where `physical == True` AND `iwa_title` is in the dynamically-computed set of IWAs where `share_ai >= 2 * share_user` AND `share_ai >= 0.0005` in `iwa_metrics.csv`. Conceptually closest to the paper's `w_nonphys` adjustment — only filters physical instances where Copilot's AI side dominates the user side.

# 2. **Eloundou no-exposure filter.** Temporarily merges Eloundou's `eloundu_human` and `eloundu_gpt4` exposure labels from `full_labelset.tsv`, drops rows where both are `"E0"`, then strips the columns. The columns get re-added by the later final-stage Eloundou cell after cumulative builds.

# Only `first_pass_microsoft_dates_taxonomy_physical.csv` is touched here. AEI/MCP/ECO files pass through unchanged.


"""
Microsoft cleanup: AI-dominant physical filter + Eloundou exposure trim.

Operates on first_pass_microsoft_dates_taxonomy_physical.csv (produced by the
taxonomy/physical cell above). Two filters applied in sequence; file overwritten
in place so every downstream cell inherits the cleanup.

Step 1 — AI-dominant physical filter:
  Drop rows where physical == True AND iwa_title is in the AI-dominant set,
  defined dynamically from iwa_metrics.csv as IWAs where
    share_ai >= 2 * share_user AND share_ai >= 0.0005.

Step 2 — Eloundou no-exposure filter:
  Merge eloundu_human / eloundu_gpt4 from full_labelset.tsv (pair lookup with
  task-only fallback). Drop rows where both labels are "E0". Strip the columns
  afterward — the later final-stage Eloundou cell reattaches them.
"""

import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

# 1. AI-dominant 2x IWAs computed live from iwa_metrics.csv
iwa_metrics = pd.read_csv(DATA_DIR / "iwa_metrics.csv")
ai_dominant_mask = (
    (iwa_metrics["share_ai"] >= 2 * iwa_metrics["share_user"])
    & (iwa_metrics["share_ai"] >= 0.0005)
)
ai_dominant_iwas = frozenset(iwa_metrics.loc[ai_dominant_mask, "title"])
print(f"AI-dominant 2x IWAs from iwa_metrics: {len(ai_dominant_iwas)}")

# 2. Load Microsoft intermediate
ms_path = DATA_DIR / "first_pass_microsoft_dates_taxonomy_physical.csv"
df = pd.read_csv(ms_path)
before_rows = len(df)
before_pairs = df[["title_current", "task_normalized"]].drop_duplicates().shape[0]

# 3. Drop physical rows under AI-dominant IWAs
phys_drop = (df["physical"] == True) & (df["iwa_title"].isin(ai_dominant_iwas))
df = df[~phys_drop].copy()
print(f"After AI-dominant physical filter: {len(df):,} of {before_rows:,} rows "
      f"(dropped {phys_drop.sum():,})")

# 4. Build Eloundou lookups (matches the final-stage cell's pair + task-only fallback)
eloundu = pd.read_csv(DATA_DIR / "full_labelset.tsv", sep="\t")[
    ["Title", "Task", "human_exposure_agg", "gpt4_exposure"]
].rename(columns={"human_exposure_agg": "eloundu_human",
                  "gpt4_exposure": "eloundu_gpt4"})
eloundu["title_norm"] = eloundu["Title"].apply(normalize_text)
eloundu["task_normalized"] = eloundu["Task"].apply(normalize_text)

def _mode_or_na(s):
    s = s.dropna()
    if s.empty:
        return pd.NA
    counts = s.value_counts()
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        return pd.NA
    return counts.index[0]

pair_lookup = (
    eloundu.groupby(["title_norm", "task_normalized"], dropna=False)
    .agg(eloundu_human=("eloundu_human", _mode_or_na),
         eloundu_gpt4=("eloundu_gpt4", _mode_or_na))
    .reset_index()
)
task_lookup = (
    eloundu.groupby("task_normalized", dropna=False)
    .agg(eloundu_human_t=("eloundu_human", _mode_or_na),
         eloundu_gpt4_t=("eloundu_gpt4", _mode_or_na))
    .reset_index()
)

# 5. Temporarily merge Eloundou onto Microsoft
df = df.drop(columns=[c for c in ["eloundu_human", "eloundu_gpt4"] if c in df.columns])
df["_match_title_norm"] = df["title_current"].apply(normalize_text)
df = (df.merge(pair_lookup,
               left_on=["_match_title_norm", "task_normalized"],
               right_on=["title_norm", "task_normalized"],
               how="left")
        .drop(columns=["title_norm"])
        .merge(task_lookup, on="task_normalized", how="left"))
h_fill = df["eloundu_human"].isna() & df["eloundu_human_t"].notna()
g_fill = df["eloundu_gpt4"].isna() & df["eloundu_gpt4_t"].notna()
df.loc[h_fill, "eloundu_human"] = df.loc[h_fill, "eloundu_human_t"]
df.loc[g_fill, "eloundu_gpt4"] = df.loc[g_fill, "eloundu_gpt4_t"]
df = df.drop(columns=["_match_title_norm", "eloundu_human_t", "eloundu_gpt4_t"])

# 6. Drop rows where both Eloundou labels are E0
both_e0 = (df["eloundu_human"] == "E0") & (df["eloundu_gpt4"] == "E0")
n_before = len(df)
df = df[~both_e0].copy()
print(f"After Eloundou E0+E0 filter:       {len(df):,} of {n_before:,} rows "
      f"(dropped {both_e0.sum():,})")

# 7. Drop Eloundou columns (re-added later by final-stage cell)
df = df.drop(columns=["eloundu_human", "eloundu_gpt4"])

# 8. Renormalize pct_normalized so unique (title_current, task_normalized)
#    pairs sum back to 100. After filtering, surviving rows still hold their
#    pre-drop pct magnitudes, so the unique-pair sum is < 100. Rescale
#    uniformly: pct *= 100 / current_unique_pair_sum. Preserves all
#    proportional relationships; loses the ability to back-derive raw
#    conversation counts (already an approximation after the IWA->pair
#    distribution step).
pair_key = ["title_current", "task_normalized"]
pct_sum_before = df.drop_duplicates(subset=pair_key)["pct_normalized"].sum()
if pct_sum_before > 0:
    df["pct_normalized"] = df["pct_normalized"] * 100.0 / pct_sum_before
pct_sum_after = df.drop_duplicates(subset=pair_key)["pct_normalized"].sum()
print(f"Renormalized pct_normalized: unique-pair sum {pct_sum_before:.4f} -> {pct_sum_after:.4f}")

# 9. Save back in place
after_pairs = df[pair_key].drop_duplicates().shape[0]
df.to_csv(ms_path, index=False)
print(f"\nMicrosoft cleanup complete:")
print(f"  rows: {before_rows:,} -> {len(df):,}  (dropped {before_rows - len(df):,})")
print(f"  unique (title_current, task_normalized) pairs: {before_pairs:,} -> {after_pairs:,}")
print(f"  Saved: {ms_path}")


AI-dominant 2x IWAs from iwa_metrics: 32
After AI-dominant physical filter: 9,950 of 10,438 rows (dropped 488)
After Eloundou E0+E0 filter:       7,635 of 9,950 rows (dropped 2,315)
Renormalized pct_normalized: unique-pair sum 85.4520 -> 100.0000

Microsoft cleanup complete:
  rows: 10,438 -> 7,635  (dropped 2,803)
  unique (title_current, task_normalized) pairs: 9,047 -> 6,613
  Saved: ..\data\first_pass_microsoft_dates_taxonomy_physical.csv


### Microsoft Standardized Auto/Aug Column 

In [78]:
"""
Compute Microsoft automation/augmentation mean score (CONT method).
auto_aug_mean = 5 * impact_scope_avg, mapping the 0-1 impact scope
linearly to the 0-5 rating scale.
Merges back to task level for the main pipeline.

NOTE: The previous distribution-based method (MCP conditional rating
distributions binned by scope) is preserved below in a commented-out
block for reference.
"""
# ============================================================
# CONFIG
# ============================================================
data_dir = Path("../data")
N_TOTAL = 100_000  # used for iwa_n diagnostic

# ============================================================
# CONT scoring: auto_aug_mean = 5 * impact_scope_avg
# ============================================================

ms_df = pd.read_csv(data_dir / "first_pass_microsoft_dates_taxonomy_physical.csv")

iwa_scored = ms_df.groupby('iwa_title').agg(
    impact_scope_ai=('impact_scope_ai', 'first'),
    impact_scope_user=('impact_scope_user', 'first'),
    impact_scope_ai_adj=('impact_scope_ai_adj', 'first'),
    impact_scope_user_adj=('impact_scope_user_adj', 'first'),
    pct_normalized_sum=('pct_normalized', 'sum'),
    n_tasks=('task', 'count'),
    gwa_title=('gwa_title', 'first'),
).reset_index()

iwa_scored['impact_scope_avg'] = iwa_scored['impact_scope_ai_adj'] + iwa_scored['impact_scope_user_adj']
iwa_scored['iwa_n'] = N_TOTAL * iwa_scored['pct_normalized_sum']
iwa_scored['auto_aug_mean'] = 5 * iwa_scored['impact_scope_avg']

# Diagnostics
print(f"\nauto_aug_mean at IWA level (CONT = 5 * impact_scope_avg):")
print(iwa_scored['auto_aug_mean'].describe())

# ============================================================
# PREVIOUS METHOD (MCP conditional distributions) -- KEPT FOR REFERENCE
# ============================================================
# INCLUDE_EXTRA_STATS = False  # Set True to include max, min, p75, p25, pct_1-5 in output
#
# # ----- PART 1: Build conditional distributions from MCP data -----
# mcp_df = pd.read_csv(data_dir / "mcp_results_2026-02-18.csv")
#
# mcp_dists = []
# for idx, cell in mcp_df["task_rating_response_raw"].dropna().items():
#     cell = cell.replace("<ratings>", "").replace("</ratings>", "").strip()
#     ratings = list(map(int, re.findall(r":(\d+)", cell)))
#     if len(ratings) == 0:
#         continue
#     counts = Counter(ratings)
#     n = len(ratings)
#     n_mod_plus = counts.get(3, 0) + counts.get(4, 0) + counts.get(5, 0)
#     mcp_dists.append({
#         'mcp_idx': idx,
#         'n_ratings': n,
#         'pct_mod_plus': n_mod_plus / n if n > 0 else 0,
#         'count_1': counts.get(1, 0),
#         'count_2': counts.get(2, 0),
#         'count_3': counts.get(3, 0),
#         'count_4': counts.get(4, 0),
#         'count_5': counts.get(5, 0),
#     })
#
# mcp_dist_df = pd.DataFrame(mcp_dists)
#
# bins = np.arange(0, 1.05, 0.05)
# bin_labels = [f"{b:.2f}-{b+0.05:.2f}" for b in bins[:-1]]
# mcp_dist_df['scope_bin'] = pd.cut(mcp_dist_df['pct_mod_plus'], bins=bins, labels=bin_labels, include_lowest=True)
#
# conditional_dists = {}
# for bin_label, group in mcp_dist_df.groupby('scope_bin', observed=True):
#     total_counts = group[['count_1', 'count_2', 'count_3', 'count_4', 'count_5']].sum()
#     total = total_counts.sum()
#     if total > 0:
#         conditional_dists[bin_label] = {
#             'dist': {
#                 1: total_counts['count_1'] / total,
#                 2: total_counts['count_2'] / total,
#                 3: total_counts['count_3'] / total,
#                 4: total_counts['count_4'] / total,
#                 5: total_counts['count_5'] / total,
#             },
#             'n_mcps': len(group),
#             'n_ratings': int(total),
#         }
#
# print(f"MCP conditional distributions built: {len(conditional_dists)} bins from {len(mcp_dist_df)} MCPs")
#
# # ----- PART 2: Apply to Microsoft data -----
# iwa_agg = ms_df.groupby('iwa_title').agg(
#     impact_scope_ai=('impact_scope_ai', 'first'),
#     impact_scope_user=('impact_scope_user', 'first'),
#     impact_scope_ai_adj=('impact_scope_ai_adj', 'first'),
#     impact_scope_user_adj=('impact_scope_user_adj', 'first'),
#     pct_normalized_sum=('pct_normalized', 'sum'),
#     n_tasks=('task', 'count'),
#     gwa_title=('gwa_title', 'first'),
# ).reset_index()
#
# iwa_agg['impact_scope_avg'] = iwa_agg['impact_scope_ai_adj'] + iwa_agg['impact_scope_user_adj']
# iwa_agg['iwa_n'] = N_TOTAL * iwa_agg['pct_normalized_sum']
#
# def find_matching_bin(scope_val, conditional_dists, bin_labels):
#     bin_idx = int(scope_val / 0.05)
#     bin_idx = min(bin_idx, len(bin_labels) - 1)
#     bin_idx = max(bin_idx, 0)
#     target_label = bin_labels[bin_idx]
#     if target_label in conditional_dists:
#         return target_label
#     for offset in range(1, len(bin_labels)):
#         for direction in [1, -1]:
#             check_idx = bin_idx + offset * direction
#             if 0 <= check_idx < len(bin_labels):
#                 check_label = bin_labels[check_idx]
#                 if check_label in conditional_dists:
#                     return check_label
#     return None
#
# def compute_stats(scope, n, conditional_dists, bin_labels, include_extra=False):
#     bin_label = find_matching_bin(scope, conditional_dists, bin_labels)
#     if bin_label is None:
#         result = {'auto_aug_mean': np.nan}
#         if include_extra:
#             result.update({
#                 'auto_aug_max': np.nan, 'auto_aug_min': np.nan,
#                 'auto_aug_p75': np.nan, 'auto_aug_p25': np.nan,
#                 'auto_aug_bin': None,
#                 'auto_aug_pct_1': np.nan, 'auto_aug_pct_2': np.nan,
#                 'auto_aug_pct_3': np.nan, 'auto_aug_pct_4': np.nan,
#                 'auto_aug_pct_5': np.nan,
#             })
#         return result
#     dist = conditional_dists[bin_label]['dist']
#     ratings = sorted(dist.keys())
#     mean_score = round(sum(r * dist[r] for r in ratings), 3)
#     result = {'auto_aug_mean': mean_score}
#     if include_extra:
#         max_score = np.nan
#         for r in reversed(ratings):
#             if n * dist[r] >= 0.5:
#                 max_score = r
#                 break
#         if pd.isna(max_score):
#             for r in reversed(ratings):
#                 if n * dist[r] >= 0.1:
#                     max_score = r
#                     break
#         min_score = np.nan
#         for r in ratings:
#             if n * dist[r] >= 0.5:
#                 min_score = r
#                 break
#         if pd.isna(min_score):
#             for r in ratings:
#                 if n * dist[r] >= 0.1:
#                     min_score = r
#                     break
#         cumsum = 0
#         p25 = np.nan
#         p75 = np.nan
#         for r in ratings:
#             cumsum += dist[r]
#             if pd.isna(p25) and cumsum >= 0.25:
#                 p25 = r
#             if pd.isna(p75) and cumsum >= 0.75:
#                 p75 = r
#                 break
#         result.update({
#             'auto_aug_max': max_score,
#             'auto_aug_min': min_score,
#             'auto_aug_p75': p75,
#             'auto_aug_p25': p25,
#             'auto_aug_bin': bin_label,
#             'auto_aug_pct_1': round(dist.get(1, 0), 4),
#             'auto_aug_pct_2': round(dist.get(2, 0), 4),
#             'auto_aug_pct_3': round(dist.get(3, 0), 4),
#             'auto_aug_pct_4': round(dist.get(4, 0), 4),
#             'auto_aug_pct_5': round(dist.get(5, 0), 4),
#         })
#     return result
#
# # Apply at IWA level
# results = iwa_agg.apply(
#     lambda row: pd.Series(compute_stats(
#         row['impact_scope_avg'], row['iwa_n'],
#         conditional_dists, bin_labels, include_extra=INCLUDE_EXTRA_STATS
#     )), axis=1
# )
#
# iwa_scored = pd.concat([iwa_agg, results], axis=1)
#
# # Diagnostics
# print(f"\nauto_aug_mean at IWA level:")
# print(iwa_scored['auto_aug_mean'].describe())
# if INCLUDE_EXTRA_STATS:
#     for col in ['auto_aug_max', 'auto_aug_min', 'auto_aug_p75', 'auto_aug_p25']:
#         print(f"\n{col}:")
#         print(iwa_scored[col].describe())

# ============================================================
# PART 3: Map back to task level and save
# ============================================================

merge_cols = ['iwa_title', 'iwa_n', 'impact_scope_avg', 'auto_aug_mean']

df_final = ms_df.merge(iwa_scored[merge_cols], on='iwa_title', how='left')

print(f"\nFinal task-level shape: {df_final.shape}")
print(f"Tasks with scores: {df_final['auto_aug_mean'].notna().sum()}")

df_final.to_csv('../data/second_pass_microsoft.csv', index=False)
print("Saved.")


auto_aug_mean at IWA level (CONT = 5 * impact_scope_avg):
count    134.000000
mean       2.657230
std        0.571810
min        1.338933
25%        2.281259
50%        2.676736
75%        3.000209
max        4.244627
Name: auto_aug_mean, dtype: float64

Final task-level shape: (7635, 137)
Tasks with scores: 7635
Saved.


### AEI Standardized Auto/Aug Columns

In [79]:
import pandas as pd

DATA = "../data/"

pairs = [
    (
        "first_pass_aei_v2_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v2.csv",
        "task_name",
    ),
    (
        "first_pass_aei_api_v3_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_api_v3.csv",
        "task",
    ),
    (
        "first_pass_aei_v3_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v3.csv",
        "task",
    ),
    (
        "first_pass_aei_api_v4_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_api_v4.csv",
        "task",
    ),
    (
        "first_pass_aei_v4_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v4.csv",
        "task",
    ),
    (   
        "first_pass_aei_v5_dates_taxonomy_physical.csv", 
        "automation_vs_augmentation_by_task_v5.csv", 
        "task"
    ),
    (   
        "first_pass_aei_api_v5_dates_taxonomy_physical.csv", 
        "automation_vs_augmentation_by_task_api_v5.csv", 
        "task"
    )
]

for master_file, collab_file, collab_task_col in pairs:

    master_df = pd.read_csv(DATA + master_file)
    collab_df = pd.read_csv(DATA + collab_file)

    # normalize task text in collab file
    collab_df["task_normalized"] = collab_df[collab_task_col].apply(normalize_text)
    # master_df.drop(columns=["task_normalized"], inplace=True, errors="ignore")  # drop if exists
    # master_df["task_normalized"] = master_df["task"].apply(normalize_text)

    # find numeric columns for averaging duplicates
    numeric_cols = collab_df.select_dtypes(include="number").columns.tolist()

    # collapse duplicate tasks
    collab_df = (
        collab_df
        .groupby("task_normalized", as_index=False)[numeric_cols]
        .mean()
    )

    merged_df = master_df.merge(
        collab_df,
        on="task_normalized",
        how="left",
        # indicator=True
    )

    # merged_df["collab_merge_match"] = merged_df["_merge"] == "both"
    # merged_df = merged_df.drop(columns="_merge")

    # output_name = "test_" + master_file
    # merged_df.to_csv(DATA + output_name, index=False)

    # print(f"Saved: {output_name}")

In [80]:
empty_cutoff = 0.875
number_needed = 2
replacement_cutoff = 0.99

DATA = "../data/"


pairs = [
    (
        "first_pass_aei_v2_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v2.csv",
        "task_name",
    ),
    (
        "first_pass_aei_api_v3_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_api_v3.csv",
        "task",
    ),
    (
        "first_pass_aei_v3_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v3.csv",
        "task",
    ),
    (
        "first_pass_aei_api_v4_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_api_v4.csv",
        "task",
    ),
    (
        "first_pass_aei_v4_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v4.csv",
        "task",
    ),
    (   
        "first_pass_aei_v5_dates_taxonomy_physical.csv", 
        "automation_vs_augmentation_by_task_v5.csv", 
        "task"
    ),
    (   
        "first_pass_aei_api_v5_dates_taxonomy_physical.csv", 
        "automation_vs_augmentation_by_task_api_v5.csv", 
        "task"
    )
]

for master_file, collab_file, collab_task_col in pairs:

    master_df = pd.read_csv(DATA + master_file)
    collab_df = pd.read_csv(DATA + collab_file)

    # normalize task text in collab file
    collab_df["task_normalized"] = collab_df[collab_task_col].apply(normalize_text)
    # master_df.drop(columns=["task_normalized"], inplace=True, errors="ignore")  # drop if exists
    # master_df["task_normalized"] = master_df["task"].apply(normalize_text)

    # find numeric columns for averaging duplicates
    numeric_cols = collab_df.select_dtypes(include="number").columns.tolist()

    # collapse duplicate tasks
    collab_df = (
        collab_df
        .groupby("task_normalized", as_index=False)[numeric_cols]
        .mean()
    )

    merged_df = master_df.merge(
        collab_df,
        on="task_normalized",
        how="left",
        #indicator=True
    )

    # ---------------------------------------------
    # COLLAB NORMALIZATION + IMPUTATION
    # ---------------------------------------------

    method_cols = ["directive","feedback_loop","learning","validation","task_iteration"]
    

    # convert percentages to proportions
    merged_df["collab_missing"] = merged_df[method_cols].isna().all(axis=1)
    merged_df[method_cols] = merged_df[method_cols].fillna(0)
    # merged_df[method_cols] = merged_df[method_cols] / 100
    

    print("Rows with no collaboration data:", merged_df["collab_missing"].sum())    

    # create empty column
    if "filtered" in merged_df.columns:
        merged_df["empty"] = merged_df["filtered"]
    else:
        merged_df["empty"] = merged_df[["none","not_classified"]].sum(axis=1)

    # ------------------------------------------------
    # normalize rows where empty < empty_cutoff
    # ------------------------------------------------

    merged_df["method_sum"] = merged_df[method_cols].sum(axis=1)

    low_mask = merged_df["empty"] < empty_cutoff

    merged_df.loc[low_mask, method_cols] = merged_df.loc[low_mask, method_cols].div(
        merged_df.loc[low_mask, "method_sum"], axis=0
    )

    merged_df.drop(columns="method_sum", inplace=True)

    # ------------------------------------------------
    # build lookup tables from low-empty tasks
    # ------------------------------------------------

    low_tasks = merged_df[merged_df["empty"] < empty_cutoff]

    dwa_means = (
        low_tasks
        .groupby("dwa_title")[method_cols]
        .mean()
    )

    dwa_means = dwa_means.div(dwa_means.sum(axis=1), axis=0).fillna(0)

    iwa_means = (
        low_tasks
        .groupby("iwa_title")[method_cols]
        .mean()
    )
    gwa_means = (
        low_tasks
        .groupby("gwa_title")[method_cols]
        .mean()
    )

    iwa_means = iwa_means.div(iwa_means.sum(axis=1), axis=0).fillna(0)
    gwa_means = gwa_means.div(gwa_means.sum(axis=1), axis=0).fillna(0)

    dwa_counts = low_tasks.groupby("dwa_title").size()
    iwa_counts = low_tasks.groupby("iwa_title").size()
    gwa_counts = low_tasks.groupby("gwa_title").size()


    dwa_dict = dwa_means.to_dict("index")
    iwa_dict = iwa_means.to_dict("index")
    gwa_dict = gwa_means.to_dict("index")

    # ------------------------------------------------
    # occupation-level lookup tables
    # ------------------------------------------------

    occ_cols = [
        "soc_code_2010",
        "broad_occ",
        "minor_occ_category",
        "major_occ_category"
    ]

    occ_dicts = {}
    occ_counts = {}

    for col in occ_cols:

        means = (
            low_tasks
            .groupby(col)[method_cols]
            .mean()
        )

        means = means.div(means.sum(axis=1), axis=0).fillna(0)

        counts = low_tasks.groupby(col).size()

        occ_dicts[col] = means.to_dict("index")
        occ_counts[col] = counts

    # ------------------------------------------------
    # IMPUTATION FUNCTION
    # ------------------------------------------------

    iwa_flag = []


    def impute_row(row):

        task = row["task"]
        empty = row["empty"]
        source = None

        dwas = task_dwa_map.get(task, [])
        iwas = task_iwa_map.get(task, [])
        gwas = task_gwa_map.get(task, [])

        current_sum = row[method_cols].sum()

        proportions = None

        # -------------------------------
        # DWA fallback
        # -------------------------------
        for dwa in dwas:
            if dwa in dwa_dict and dwa_counts[dwa] >= number_needed:
                proportions = dwa_dict[dwa]
                source = "dwa"
                break

        # -------------------------------
        # IWA fallback
        # -------------------------------
        if proportions is None:
            for iwa in iwas:
                if iwa in iwa_dict and iwa_counts[iwa] >= number_needed:
                    proportions = iwa_dict[iwa]
                    source = "iwa"
                    break

        # -------------------------------
        # GWA fallback
        # -------------------------------
        if proportions is None:
            for gwa in gwas:
                if gwa in gwa_dict and gwa_counts[gwa] >= number_needed:
                    proportions = gwa_dict[gwa]
                    source = "gwa"
                    break
        
        # -------------------------------
        # OCCUPATION FALLBACKS
        # -------------------------------
        if proportions is None:

            for col in occ_cols:

                key = row[col]

                if key in occ_dicts[col] and occ_counts[col][key] >= number_needed:
                    proportions = occ_dicts[col][key]
                    source = col
                    break

        # -------------------------------
        # final failure case
        # -------------------------------
        if proportions is None:
            iwa_flag.append(task)
            return row


        # ------------------------------------------------
        # mid-empty case (.5–.9)
        # ------------------------------------------------
        if empty_cutoff <= empty < replacement_cutoff:

            remaining = 1 - current_sum

            for col in method_cols:
                row[col] += remaining * proportions.get(col, 0)

        # ------------------------------------------------
        # high-empty case (≥ .9)
        # ------------------------------------------------
        elif empty >= replacement_cutoff:

            for col in method_cols:
                row[col] = proportions.get(col, 0)

        row["imputation_source"] = source
        if empty < empty_cutoff:
            row["imputation_source"] = "none_needed"
            return row
        return row

    task_dwa_map = merged_df.groupby("task")["dwa_title"].unique().to_dict()
    task_iwa_map = merged_df.groupby("task")["iwa_title"].unique().to_dict()
    task_gwa_map = merged_df.groupby("task")["gwa_title"].unique().to_dict()
    merged_df["imputation_source"] = None
    merged_df = merged_df.apply(impute_row, axis=1)

    # ------------------------------------------------
    # VALIDATION
    # ------------------------------------------------

    check = merged_df[method_cols].sum(axis=1)

    print("\nRows summing to 1:", (check.round(5)==1).sum(), "/", len(check))

    # ------------------------------------------------
    # cleanup columns
    # ------------------------------------------------

    drop_cols = [c for c in ["none","not_classified","filtered", "collab_missing", "imputation_source"] if c in merged_df.columns]

    merged_df.drop(columns=drop_cols, inplace=True)


    # ------------------------------------------------
    # AEI 1-5 SCALE COLUMNS
    # ------------------------------------------------

    score_map = {
        "learning": 1,
        "validation": 2,
        "task_iteration": 3,
        "feedback_loop": 4,
        "directive": 5,
    }

    method_cols_ordered = ["learning", "validation", "task_iteration", "feedback_loop", "directive"]

    # mean: weighted average
    merged_df["aei_mean"] = sum(
        merged_df[col] * score for col, score in score_map.items()
    )

    # max: highest category with nonzero presence
    def get_max(row):
        for col in reversed(method_cols_ordered):  # high to low
            if row[col] > 0:
                return score_map[col]
        return 1

    # min: lowest category with nonzero presence
    def get_min(row):
        for col in method_cols_ordered:  # low to high
            if row[col] > 0:
                return score_map[col]
        return 1

    # percentile: cumulative from top, find where cumulative first exceeds threshold
    def get_percentile(row, threshold):
        cumulative = 0
        for col in reversed(method_cols_ordered):
            cumulative += row[col]
            if cumulative >= threshold:
                return score_map[col]
        return score_map[method_cols_ordered[0]]

    merged_df["aei_max"] = merged_df.apply(get_max, axis=1)
    merged_df["aei_min"] = merged_df.apply(get_min, axis=1)
    merged_df["aei_p75"] = merged_df.apply(lambda r: get_percentile(r, 0.25), axis=1)
    merged_df["aei_p25"] = merged_df.apply(lambda r: get_percentile(r, 0.75), axis=1)

    # drop source columns
    drop_cols = method_cols_ordered + ["empty", "aei_max", "aei_min", "aei_p75", "aei_p25"] 
    merged_df.drop(columns=[c for c in drop_cols if c in merged_df.columns], inplace=True)
    merged_df.rename(columns={"aei_mean": "auto_aug_mean"}, inplace=True)

    # ------------------------------------------------
    # save output
    # ------------------------------------------------

    output_name = master_file.replace("first", "second").replace("_dates_taxonomy_physical", "")
    
    merged_df.to_csv(DATA + output_name, index=False)
    print(f"Saved: {output_name}")

C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["collab_missing"] = merged_df[method_cols].isna().all(axis=1)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["empty"] = merged_df["filtered"]
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

Rows with no collaboration data: 0


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:273: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["imputation_source"] = None



Rows summing to 1: 5264 / 5264


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:308: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["aei_mean"] = sum(
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:335: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["aei_max"] = merged_df.apply(get_max, axis=1)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:336: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Con

...

Saved: second_pass_aei_v5.csv
Rows with no collaboration data: 643


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["collab_missing"] = merged_df[method_cols].isna().all(axis=1)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["empty"] = merged_df[["none","not_classified"]].sum(axis=1)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram


Rows summing to 1: 3737 / 3737


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:308: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["aei_mean"] = sum(
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:335: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["aei_max"] = merged_df.apply(get_max, axis=1)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\2177193359.py:336: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Con

Saved: second_pass_aei_api_v5.csv


### Cleanup

In [81]:
DATA_DIR = Path("../data")  # Adjust this path if your data is elsewhere

# 1. Process "second_pass_aei" files: Drop physical_imputed
aei_seconds = [f for f in os.listdir(DATA_DIR) if f.startswith("second_pass_aei") and f.endswith(".csv")]
for file in aei_seconds:
    df = pd.read_csv(DATA_DIR / file)
    if "physical_imputed" in df.columns:
        df = df.drop(columns=["physical_imputed", "master_pct_normalized", "master_pct_adj"])
    df.to_csv(DATA_DIR / file, index=False)
    print(f"Updated: {file}")

# 2. Process "first_pass_aei_v1": Drop physical_imputed, add empty auto_aug_mean
v1_file = "first_pass_aei_v1_dates_taxonomy_physical.csv"
if os.path.exists(DATA_DIR / v1_file):
    df_v1 = pd.read_csv(DATA_DIR / v1_file)
    df_v1 = df_v1.drop(columns=["physical_imputed", "master_pct_normalized", "master_pct_adj"], errors='ignore')
    df_v1["auto_aug_mean"] = None
    df_v1.to_csv(DATA_DIR / "second_pass_aei_v1.csv", index=False)
    print("Created: second_pass_aei_v1.csv")

# 2.5. Process Eco 2015 and 2025
eco_files = [
    ("first_pass_eco_tasks_2015_dates_taxonomy_physical.csv", "second_pass_eco_2015.csv"),
    ("first_pass_eco_tasks_2025_dates_taxonomy_physical.csv", "second_pass_eco_2025.csv")
]

for input_file, output_file in eco_files:
    if os.path.exists(DATA_DIR / input_file):
        df_eco = pd.read_csv(DATA_DIR / input_file)
        
        # Drop the physical imputation column as requested
        df_eco = df_eco.drop(columns=["physical_imputed", "master_pct_normalized", "master_pct_adj"], errors='ignore')
        
        # Add the empty auto_aug_mean column to match the AEI structure
        df_eco["auto_aug_mean"] = None
        
        df_eco.to_csv(DATA_DIR / output_file, index=False)
        print(f"Created: {output_file}")
    else:
        print(f"Warning: {input_file} not found.")

# 3. Process "first_pass_mcp" files: Drop/Rename and save as second_pass
mcp_firsts = [f for f in os.listdir(DATA_DIR) if f.startswith("first_pass_mcp") and "_dates_taxonomy_physical" in f]
for file in mcp_firsts:
    df_mcp = pd.read_csv(DATA_DIR / file)
    
    # Drops
    cols_to_drop = ["physical_imputed", "n_ratings", "median_rating", "max_rating", 
                    "min_rating", "p25_rating", "p75_rating", "master_pct_normalized", "master_pct_adj"]
    df_mcp = df_mcp.drop(columns=[c for c in cols_to_drop if c in df_mcp.columns])
    
    # Rename
    if "mean_rating" in df_mcp.columns:
        df_mcp = df_mcp.rename(columns={"mean_rating": "auto_aug_mean"})
    
    # Save as second_pass
    new_name = file.replace("first", "second").replace("_dates_taxonomy_physical", "")
    df_mcp.to_csv(DATA_DIR / new_name, index=False)
    print(f"Created: {new_name}")

# 4. Process "second_pass_microsoft": Drop specific columns
ms_second = "second_pass_microsoft.csv"
if os.path.exists(DATA_DIR / ms_second):
    df_ms = pd.read_csv(DATA_DIR / ms_second)
    df_ms["title_current_normalized"] = df_ms["title_current"].apply(normalize_text)
    df_ms = df_ms.drop(columns=["physical_imputed", "iwa_n", "impact_scope_avg",
                                'impact_scope_user', 'impact_scope_ai', 'impact_scope_user_adj', 'impact_scope_ai_adj', "master_pct_normalized", "master_pct_adj"], errors='ignore')
    df_ms.to_csv(DATA_DIR / ms_second, index=False)
    print(f"Updated: {ms_second}")

# 5. DELETE all first_pass CSVs with "dates_taxonomy_physical"
# --- WARNING: This permanently deletes files ---
for file in os.listdir(DATA_DIR):
    if file.startswith("first_pass") and "_dates_taxonomy_physical.csv" in file:
        os.remove(DATA_DIR / file)
        print(f"Deleted: {file}")

print("\nCleanup Complete.")

Updated: second_pass_aei_api_v3.csv
Updated: second_pass_aei_api_v4.csv
Updated: second_pass_aei_api_v5.csv
Updated: second_pass_aei_v1.csv
Updated: second_pass_aei_v2.csv
Updated: second_pass_aei_v3.csv
Updated: second_pass_aei_v4.csv
Updated: second_pass_aei_v5.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\1247284016.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_v1["auto_aug_mean"] = None


Created: second_pass_aei_v1.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\1247284016.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eco["auto_aug_mean"] = None


Created: second_pass_eco_2015.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\1247284016.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eco["auto_aug_mean"] = None


Created: second_pass_eco_2025.csv
Created: second_pass_mcp_v1.csv
Created: second_pass_mcp_v2.csv
Created: second_pass_mcp_v3.csv
Created: second_pass_mcp_v4.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\1247284016.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ms["title_current_normalized"] = df_ms["title_current"].apply(normalize_text)


Updated: second_pass_microsoft.csv
Deleted: first_pass_aei_api_v3_dates_taxonomy_physical.csv
Deleted: first_pass_aei_api_v4_dates_taxonomy_physical.csv
Deleted: first_pass_aei_api_v5_dates_taxonomy_physical.csv
Deleted: first_pass_aei_v1_dates_taxonomy_physical.csv
Deleted: first_pass_aei_v2_dates_taxonomy_physical.csv
Deleted: first_pass_aei_v3_dates_taxonomy_physical.csv
Deleted: first_pass_aei_v4_dates_taxonomy_physical.csv
Deleted: first_pass_aei_v5_dates_taxonomy_physical.csv
Deleted: first_pass_eco_tasks_2015_dates_taxonomy_physical.csv
Deleted: first_pass_eco_tasks_2025_dates_taxonomy_physical.csv
Deleted: first_pass_mcp_v1_dates_taxonomy_physical.csv
Deleted: first_pass_mcp_v2_dates_taxonomy_physical.csv
Deleted: first_pass_mcp_v3_dates_taxonomy_physical.csv
Deleted: first_pass_mcp_v4_dates_taxonomy_physical.csv
Deleted: first_pass_microsoft_dates_taxonomy_physical.csv

Cleanup Complete.


In [82]:
DATA_DIR = Path("../data")

# Define the master desired order
# We split this into segments to make the logic clear
COLS_BASE = ['task', 'task_normalized']
COLS_TAXONOMY = ['dwa_title', 'iwa_title', 'gwa_title']
COLS_TITLES_SOC = [
    'title', 'title_normalized', 'soc_code_2010', 
    'title_current', 'title_current_normalized', 
    'soc_code_2019', 'soc_code_2019_full'
]
COLS_CATEGORIES = ['broad_occ','minor_occ_category', 'major_occ_category']
COLS_METRICS = ['date', 'pct_normalized', 'auto_aug_mean', 'physical']
COLS_ECON = ['a_med_nat_2025']
for _abbrev in STATE_ABBREV.values():
    COLS_ECON.append(f'a_med_{_abbrev}_2025')
COLS_ECON.append('emp_tot_nat_2025')
for _abbrev in STATE_ABBREV.values():
    COLS_ECON.append(f'emp_tot_{_abbrev}_2025')

# Combine into one master list
MASTER_ORDER = COLS_BASE + COLS_TAXONOMY + COLS_TITLES_SOC + COLS_CATEGORIES + COLS_METRICS + COLS_ECON

def reorder_columns(file_path):
    df = pd.read_csv(file_path)
    
    # Filter the master order to only include columns that actually exist in this file
    existing_cols = [c for c in MASTER_ORDER if c in df.columns]
    
    # Check if there are any columns in the dataframe NOT in our master list 
    # (to avoid losing data accidentally)
    extra_cols = [c for c in df.columns if c not in existing_cols]
    
    # Final list: standard order + anything left over
    final_col_order = existing_cols + extra_cols
    
    df = df[final_col_order]
    df.to_csv(file_path, index=False)
    print(f"Reordered: {file_path.name}")

# Process all second_pass files
target_files = [f for f in os.listdir(DATA_DIR) if f.startswith("second_pass") and f.endswith(".csv")]

for filename in target_files:
    reorder_columns(DATA_DIR / filename)

print("\nAll files reordered successfully.")

Reordered: second_pass_aei_api_v3.csv
Reordered: second_pass_aei_api_v4.csv
Reordered: second_pass_aei_api_v5.csv
Reordered: second_pass_aei_v1.csv
Reordered: second_pass_aei_v2.csv
Reordered: second_pass_aei_v3.csv
Reordered: second_pass_aei_v4.csv
Reordered: second_pass_aei_v5.csv
Reordered: second_pass_eco_2015.csv
Reordered: second_pass_eco_2025.csv
Reordered: second_pass_eco_2025_with_task_prop.csv
Reordered: second_pass_mcp_v1.csv
Reordered: second_pass_mcp_v2.csv
Reordered: second_pass_mcp_v3.csv
Reordered: second_pass_mcp_v4.csv
Reordered: second_pass_microsoft.csv

All files reordered successfully.


### Check Columns

In [83]:
print(pd.read_csv("../data/second_pass_microsoft.csv").columns.tolist())
print(pd.read_csv("../data/second_pass_mcp_v2.csv").columns.tolist())
print(pd.read_csv("../data/second_pass_aei_v1.csv").columns.tolist())
print(pd.read_csv("../data/second_pass_eco_2015.csv").columns.tolist())
print(pd.read_csv("../data/second_pass_eco_2025.csv").columns.tolist())


['task', 'task_normalized', 'dwa_title', 'iwa_title', 'gwa_title', 'title', 'title_normalized', 'soc_code_2010', 'title_current', 'title_current_normalized', 'soc_code_2019', 'soc_code_2019_full', 'broad_occ', 'minor_occ_category', 'major_occ_category', 'date', 'pct_normalized', 'auto_aug_mean', 'physical', 'a_med_nat_2025', 'a_med_al_2025', 'a_med_ak_2025', 'a_med_az_2025', 'a_med_ar_2025', 'a_med_ca_2025', 'a_med_co_2025', 'a_med_ct_2025', 'a_med_de_2025', 'a_med_dc_2025', 'a_med_fl_2025', 'a_med_ga_2025', 'a_med_hi_2025', 'a_med_id_2025', 'a_med_il_2025', 'a_med_in_2025', 'a_med_ia_2025', 'a_med_ks_2025', 'a_med_ky_2025', 'a_med_la_2025', 'a_med_me_2025', 'a_med_md_2025', 'a_med_ma_2025', 'a_med_mi_2025', 'a_med_mn_2025', 'a_med_ms_2025', 'a_med_mo_2025', 'a_med_mt_2025', 'a_med_ne_2025', 'a_med_nv_2025', 'a_med_nh_2025', 'a_med_nj_2025', 'a_med_nm_2025', 'a_med_ny_2025', 'a_med_nc_2025', 'a_med_nd_2025', 'a_med_oh_2025', 'a_med_ok_2025', 'a_med_or_2025', 'a_med_pa_2025', 'a_med_ri_

# Main Data Part 3

## Ratings

### Parameter Adjustment

In [84]:
frequency_weights = {
    1: 1 / 260,
    2: 4 / 260,
    3: 24 / 260,
    4: 104 / 260,
    5: 1,
    6: 3,
    7: 8
}


#Helper Functions

def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()                   # lowercase + trim
    text = re.sub(r"[^\w\s]", "", text)            # remove punctuation
    text = re.sub(r"\s+", " ", text)               # collapse multiple spaces
    return text


### Create Task Prop

In [85]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

# 1. Load and unique-ify Eco 2025
eco_2025 = pd.read_csv(DATA_DIR / "second_pass_eco_2025.csv")
eco_2025_unique = eco_2025[["title_current", "task_normalized"]].drop_duplicates()

# 2. Load and unique-ify Eco 2015
eco_2015 = pd.read_csv(DATA_DIR / "second_pass_eco_2015.csv")
eco_2015_unique = eco_2015[["title", "task_normalized"]].drop_duplicates()

# 3. Load Crosswalk
crosswalk = pd.read_csv(DATA_DIR / "2010_to_2019_soc_crosswalk.csv")

# 4. Count tasks in 2025 (The Denominator)
counts_2025 = eco_2025_unique.groupby("title_current").size().reset_index(name="count_2025")

# 5. Map 2015 tasks to 2025 titles using Crosswalk (The Numerator)
# First, link 2015 unique tasks to the crosswalk via the 2010 title
cw_mapped = crosswalk.merge(
    eco_2015_unique, 
    left_on="O*NET-SOC 2010 Title", 
    right_on="title", 
    how="inner"
)

# Now group by the 2019 Title (which matches title_current) to get the 2015 task sum
counts_2015_mapped = (
    cw_mapped.groupby("O*NET-SOC 2019 Title")
    .size()
    .reset_index(name="count_2015")
)

# 6. Merge counts and calculate the proportion
final_counts = counts_2025.merge(
    counts_2015_mapped, 
    left_on="title_current", 
    right_on="O*NET-SOC 2019 Title", 
    how="left"
)

# Fill 0 for cases where a 2019 title has no 2010 ancestor in the crosswalk
final_counts["count_2015"] = final_counts["count_2015"].fillna(0)
final_counts["task_prop"] = final_counts["count_2015"] / final_counts["count_2025"]

# 7. Map the task_prop back to the original Eco 2025 dataframe
eco_2025 = eco_2025.merge(
    final_counts[["title_current", "task_prop"]], 
    on="title_current", 
    how="left"
)

# eco_2025 = eco_2025[["title_current", "task_prop"]].drop_duplicates().sort_values("task_prop", ascending=False)
# # Print a few examples
# print(eco_2025[["title_current", "task_prop"]].drop_duplicates().sort_values("task_prop", ascending=False).head(10))
eco_2025.to_csv(DATA_DIR / "second_pass_eco_2025_with_task_prop.csv", index=False)
eco_2025

,task,task_normalized,dwa_title,iwa_title,gwa_title,title,title_normalized,soc_code_2010,title_current,soc_code_2019,...,emp_tot_vt_2025,emp_tot_va_2025,emp_tot_wa_2025,emp_tot_wv_2025,emp_tot_wi_2025,emp_tot_wy_2025,emp_tot_gu_2025,emp_tot_pr_2025,emp_tot_vi_2025,task_prop
0,Direct or coordinate an organization's financi...,direct or coordinate an organizations financia...,Direct financial operations.,Manage budgets or finances.,"Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,586.0,7929.0,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0,1.0
1,"Confer with board members, organization offici...",confer with board members organization officia...,Confer with organizational members to accompli...,Communicate with others about operational plan...,"Communicating with Supervisors, Peers, or Subo...",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,586.0,7929.0,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0,1.0
2,"Prepare budgets for approval, including those ...",prepare budgets for approval including those f...,Prepare operational budgets.,Manage budgets or finances.,"Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,586.0,7929.0,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0,1.0
3,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,Implement organizational process or policy cha...,Implement procedures or processes.,Making Decisions and Solving Problems,Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,586.0,7929.0,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0,1.0
4,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,Develop organizational policies or programs.,"Develop organizational policies, systems, or p...",Developing Objectives and Strategies,Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,586.0,7929.0,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23845,"Clean interiors of tank cars or tank trucks, u...",clean interiors of tank cars or tank trucks us...,Clean vessels or marine equipment.,"Clean tools, equipment, facilities, or work ar...",Performing General Physical Activities,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,21.0,60.0,243.0,90.0,170.0,19.0,5.0,64.0,2.0,1.0
23846,Lower gauge rods into tanks or read meters to ...,lower gauge rods into tanks or read meters to ...,Measure the level or depth of water or other l...,"Measure physical characteristics of materials,...",Estimating the Quantifiable Characteristics of...,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,21.0,60.0,243.0,90.0,170.0,19.0,5.0,64.0,2.0,1.0
23847,Operate conveyors and equipment to transfer gr...,operate conveyors and equipment to transfer gr...,Operate conveyors or other industrial material...,Operate industrial processing or production eq...,Controlling Machines and Processes,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,21.0,60.0,243.0,90.0,170.0,19.0,5.0,64.0,2.0,1.0
23848,"Perform general warehouse activities, such as ...",perform general warehouse activities such as o...,Weigh materials to ensure compliance with spec...,"Measure physical characteristics of materials,...",Estimating the Quantifiable Characteristics of...,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,21.0,60.0,243.0,90.0,170.0,19.0,5.0,64.0,2.0,1.0


### Eco Dfs

In [86]:
DATA_DIR = Path("../data")

# ---------------------------------------------------------
# 1. PREP RATINGS FROM v30.1
# ---------------------------------------------------------
tr_df = pd.read_csv(DATA_DIR / "task_ratings_v30.1.csv")
tr_df["task_normalized"] = tr_df["Task"].apply(normalize_text)
tr_df["title_normalized"] = tr_df["Title"].apply(normalize_text)

# Frequency: weighted mean from category distribution
freq_df = tr_df[tr_df["Scale ID"] == "FT"].copy()
freq_df = freq_df[pd.to_numeric(freq_df["Category"], errors="coerce").notnull()]
freq_df["Category"] = freq_df["Category"].astype(int)
freq_df["weighted_val"] = freq_df["Data Value"] * freq_df["Category"].map(frequency_weights) / 100

freq_agg = (
    freq_df.groupby(["title_normalized", "task_normalized"])["weighted_val"]
    .sum()
    .reset_index()
    .rename(columns={"weighted_val": "freq_mean"})
)

# Importance & Relevance (one row per task already)
imp_df = (
    tr_df[tr_df["Scale ID"] == "IM"][["title_normalized", "task_normalized", "Data Value"]]
    .rename(columns={"Data Value": "importance"})
)
rel_df = (
    tr_df[tr_df["Scale ID"] == "RT"][["title_normalized", "task_normalized", "Data Value"]]
    .rename(columns={"Data Value": "relevance"})
)

ratings = freq_agg.merge(imp_df, on=["title_normalized", "task_normalized"], how="outer")
ratings = ratings.merge(rel_df, on=["title_normalized", "task_normalized"], how="outer")

# Task-only averages for fallback
ratings_task_avg = (
    ratings.groupby("task_normalized")[["freq_mean", "importance", "relevance"]]
    .mean()
    .reset_index()
    .rename(columns={"freq_mean": "freq_mean_tavg", "importance": "importance_tavg", "relevance": "relevance_tavg"})
)

print(f"Ratings prepared: {len(ratings)} (title, task) pairs")

# ---------------------------------------------------------
# 2. PROCESS EACH ECO FILE
# ---------------------------------------------------------
RATING_COLS = ["freq_mean", "importance", "relevance"]
MIN_FOR_IMPUTE = 5

eco_configs = [
    ("second_pass_eco_2025_with_task_prop.csv", "title_current"),
    ("second_pass_eco_2015.csv", "title"),
]

for eco_file, title_col in eco_configs:
    print(f"\n{'='*60}")
    print(f"Processing: {eco_file} (title col: {title_col})")

    df = pd.read_csv(DATA_DIR / eco_file)
    
    df["task_normalized"] = df["task"].apply(normalize_text)
    df["_merge_title_norm"] = df[title_col].apply(normalize_text)

    total = len(df)
    print(f"Total rows: {total}")

    # --- MERGE: (title, task) ---
    merged = df.merge(
        ratings,
        left_on=["_merge_title_norm", "task_normalized"],
        right_on=["title_normalized", "task_normalized"],
        how="left",
    )

    # Drop the ratings title_normalized to avoid confusion
    merged.drop(columns=["title_normalized"], inplace=True, errors="ignore")

    matched_title_task = merged["freq_mean"].notna().sum()
    print(f"After (title+task) merge: {matched_title_task}/{total} ({matched_title_task/total:.1%})")

    # --- FALLBACK: task-only average for still-missing ---
    merged = merged.merge(ratings_task_avg, on="task_normalized", how="left")
    for col in RATING_COLS:
        mask = merged[col].isna()
        merged.loc[mask, col] = merged.loc[mask, f"{col}_tavg"]
    merged.drop(columns=[f"{c}_tavg" for c in RATING_COLS], inplace=True)

    matched_after_task = merged["freq_mean"].notna().sum()
    print(f"After task-only fallback: {matched_after_task}/{total} ({matched_after_task/total:.1%})")

    # --- IMPUTATION: Occupation-Level Average ---
    merged["rating_imputed"] = "none"
    
    # Use deduplicated (title, task) pairs for occupation-level stats
    # to avoid taxonomy duplicates inflating counts/means
    deduped_for_occ = merged.drop_duplicates(subset=[title_col, "task_normalized"])
    
    for col in RATING_COLS:
        occ_stats = (
            deduped_for_occ[deduped_for_occ[col].notna()]
            .groupby(title_col)[col]
            .agg(["mean", "count"])
            .rename(columns={"mean": f"{col}_occ_lvl", "count": f"{col}_occ_cnt"})
        )
        merged = merged.merge(occ_stats, on=title_col, how="left")
        
        occ_mask = merged[col].isna() & (merged[f"{col}_occ_cnt"] >= MIN_FOR_IMPUTE)
        merged.loc[occ_mask, col] = merged.loc[occ_mask, f"{col}_occ_lvl"]
        merged.loc[occ_mask & (merged["rating_imputed"] == "none"), "rating_imputed"] = "occupation"
        
        merged.drop(columns=[f"{col}_occ_lvl", f"{col}_occ_cnt"], inplace=True)

    matched_after_occ = merged["freq_mean"].notna().sum()
    print(f"After occupation-level average: {matched_after_occ}/{total} ({matched_after_occ/total:.1%})")

    # --- IMPUTATION: DWA -> IWA -> GWA ---
    for level_col, level_label in [("dwa_title", "dwa"), ("iwa_title", "iwa"), ("gwa_title", "gwa")]:
        for col in RATING_COLS:
            still_missing = merged[col].isna()
            if not still_missing.any():
                continue

            # Compute means and counts from rows that HAVE values at this level
            level_stats = (
                merged[merged[col].notna()]
                .groupby(level_col)[col]
                .agg(["mean", "count"])
                .rename(columns={"mean": f"{col}_lvl", "count": f"{col}_cnt"})
            )

            merged = merged.merge(level_stats, on=level_col, how="left", suffixes=("", f"_{level_label}"))

            fill_mask = merged[col].isna() & (merged[f"{col}_cnt"] >= MIN_FOR_IMPUTE)
            merged.loc[fill_mask, col] = merged.loc[fill_mask, f"{col}_lvl"]

            # Update imputation label (only upgrade, don't overwrite a more specific one)
            newly_filled = fill_mask & merged["rating_imputed"].isin(["none", level_label])
            merged.loc[newly_filled, "rating_imputed"] = level_label

            merged.drop(columns=[f"{col}_lvl", f"{col}_cnt"], inplace=True)

    # --- FALLBACK: major_occ_category -> global mean ---
    for col in RATING_COLS:
        mask = merged[col].isna()
        if not mask.any():
            continue

        major_means = merged[merged[col].notna()].groupby("major_occ_category")[col].mean()
        fill_mask = mask & merged["major_occ_category"].map(major_means).notna()
        merged.loc[fill_mask, col] = merged.loc[fill_mask, "major_occ_category"].map(major_means)
        merged.loc[fill_mask & (merged["rating_imputed"].isin(["none", "major"])), "rating_imputed"] = "major"

    for col in RATING_COLS:
        mask = merged[col].isna()
        if mask.any():
            merged.loc[mask, col] = merged[col].mean()
            merged.loc[mask, "rating_imputed"] = "global"

    # Mark rows that got ratings from the initial merge as "none" (no imputation)
    # Rows still missing after everything stay "none" but with NaN values
    final_missing = merged[RATING_COLS].isna().any(axis=1).sum()
    print(f"After DWA/IWA/GWA imputation: {total - final_missing}/{total} ({(total-final_missing)/total:.1%})")
    print(f"Still missing: {final_missing}")

    # Imputation level distribution
    print(f"\nImputation levels:")
    print(merged["rating_imputed"].value_counts().to_string())

    # --- PENALTY: Halve freq_mean for non-occupation imputations in high-impute titles ---
    # 1. Calculate the percentage of tasks per occupation that have ANY imputation (not 'none')
    # Use deduped pairs for imputation percentage calculation
    deduped_impute = merged.drop_duplicates(subset=[title_col, "task_normalized"])
    title_impute_pct = (
        deduped_impute.groupby(title_col)["rating_imputed"]
        .apply(lambda x: (x != "none").mean())
        .to_dict()
    )
    titles_total_impute_pct = merged[title_col].map(title_impute_pct)
    
    # 2. Identify rows that are specifically NOT 'none' and NOT 'occupation'
    non_occ_impute_mask = ~merged["rating_imputed"].isin(["none", "occupation"])
    
    # 3. Apply penalty: If title is > 50% imputed overall, halve freq for the non-occ imputed rows
    penalty_mask = (titles_total_impute_pct > 0.5) & non_occ_impute_mask
    merged.loc[penalty_mask, "freq_mean"] = merged.loc[penalty_mask, "freq_mean"] / 2
    
    # 4. Diagnostics
    n_penalized = penalty_mask.sum()
    print(f"Penalized {n_penalized} tasks in occupations with > 50% total imputation.")

    # --- CLEANUP & SAVE ---
    merged.drop(columns=["_merge_title_norm", "title_normalized_y", "rating_imputed"], inplace=True, errors="ignore")
    merged.rename(columns={"title_normalized_x": "title_normalized"}, inplace=True)

    if eco_file == "second_pass_eco_2025_with_task_prop.csv":
        # 1. Reload the original task_prop source to ensure we have the clean column
        tp_source = pd.read_csv(DATA_DIR / "second_pass_eco_2025_with_task_prop.csv")
        
        # 2. Get only unique title_current and the task_prop column
        tp_lookup = tp_source[["title_current", "task_prop"]].drop_duplicates()
        
        # 3. Force merge it back into the final 'merged' dataframe
        # We drop task_prop from 'merged' first if it exists to avoid suffixes like _x/_y
        if "task_prop" in merged.columns:
            merged.drop(columns=["task_prop"], inplace=True)
            
        merged = merged.merge(tp_lookup, on="title_current", how="left")
        
        # 4. Save
        merged.to_csv(DATA_DIR / "third_pass_eco_2025.csv", index=False)
    else:
        output_name = eco_file.replace("second_pass_", "third_pass_")
        merged.to_csv(DATA_DIR / output_name, index=False)
    print(f"\nSaved: {output_name} ({len(merged)} rows)")

Ratings prepared: 17951 (title, task) pairs

Processing: second_pass_eco_2025_with_task_prop.csv (title col: title_current)


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3871387620.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_merge_title_norm"] = df[title_col].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3871387620.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["rating_imputed"] = "none"


Total rows: 23850
After (title+task) merge: 22784/23850 (95.5%)
After task-only fallback: 22925/23850 (96.1%)
After occupation-level average: 23323/23850 (97.8%)
After DWA/IWA/GWA imputation: 23850/23850 (100.0%)
Still missing: 0

Imputation levels:
rating_imputed
none          22925
dwa             482
occupation      398
iwa              44
gwa               1
Penalized 527 tasks in occupations with > 50% total imputation.

Saved: second_pass_aei_api_v5.csv (23850 rows)

Processing: second_pass_eco_2015.csv (title col: title)


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3871387620.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_merge_title_norm"] = df[title_col].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3871387620.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["rating_imputed"] = "none"


Total rows: 24631
After (title+task) merge: 17660/24631 (71.7%)
After task-only fallback: 21098/24631 (85.7%)
After occupation-level average: 23795/24631 (96.6%)
After DWA/IWA/GWA imputation: 24631/24631 (100.0%)
Still missing: 0

Imputation levels:
rating_imputed
none          21098
occupation     2697
dwa             666
iwa             125
major            44
gwa               1
Penalized 829 tasks in occupations with > 50% total imputation.

Saved: third_pass_eco_2015.csv (24631 rows)


### Merge into second passes

In [87]:
# ---------------------------------------------------------
# Load third pass eco files (with ratings already merged)
# ---------------------------------------------------------
eco_2015 = pd.read_csv(DATA_DIR / "third_pass_eco_2015.csv")
eco_2025 = pd.read_csv(DATA_DIR / "third_pass_eco_2025.csv")

# Build lookup tables: (title/title_current, task_normalized) -> ratings
# Deduplicate to avoid blowing up rows on merge
rating_cols = ["freq_mean", "importance", "relevance"]

lookup_2015 = (
    eco_2015[["title_normalized", "task_normalized"] + rating_cols]
    .drop_duplicates(subset=["title_normalized", "task_normalized"])
)

lookup_2025 = (
    eco_2025[["title_current", "task_normalized"] + rating_cols]
    .drop_duplicates(subset=["title_current", "task_normalized"])
)

# ---------------------------------------------------------
# Process all second pass files
# ---------------------------------------------------------
# second_pass_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("second_pass_") and f.endswith(".csv") and "eco" not in f])
second_pass_files = ["second_pass_aei_api_v3.csv", "second_pass_aei_api_v4.csv", "second_pass_aei_v1.csv", 
                     "second_pass_aei_v2.csv", "second_pass_aei_v3.csv", "second_pass_aei_v4.csv", 
                     "second_pass_mcp_v1.csv", "second_pass_mcp_v2.csv", "second_pass_mcp_v3.csv", 
                     "second_pass_mcp_v4.csv", "second_pass_microsoft.csv", "second_pass_aei_api_v5.csv", "second_pass_aei_v5.csv"]

for filename in second_pass_files:
    df = pd.read_csv(DATA_DIR / filename)
    
    is_aei = "aei" in filename
    
    if is_aei:
        # AEI: merge on title_normalized + task_normalized using 2015 ratings
        df = df.merge(
            lookup_2015,
            on=["title_normalized", "task_normalized"],
            how="left",
        )
    else:
        # MCP, Microsoft: merge on title_current + task_normalized using 2025 ratings
        df = df.merge(
            lookup_2025,
            on=["title_current", "task_normalized"],
            how="left",
        )
    
    # Reorder: insert freq_mean, importance, relevance after auto_aug_mean
    cols = list(df.columns)
    
    # Remove rating cols from wherever they landed
    for rc in rating_cols:
        cols.remove(rc)
    
    # Find insertion point (right before 'physical')
    phys_idx = cols.index("physical")
    for i, rc in enumerate(rating_cols):
        cols.insert(phys_idx + i, rc)
    
    df = df[cols]
    
    # Report
    matched = df["freq_mean"].notna().sum()
    total = len(df)
    print(f"{filename}: {matched}/{total} ({matched/total:.1%}) have ratings")
    
    output_name = filename.replace("second_pass_", "third_pass_")
    df.to_csv(DATA_DIR / output_name, index=False)
    print(f"  -> Saved: {output_name}")

second_pass_aei_api_v3.csv: 3325/3325 (100.0%) have ratings
  -> Saved: third_pass_aei_api_v3.csv
second_pass_aei_api_v4.csv: 3585/3585 (100.0%) have ratings
  -> Saved: third_pass_aei_api_v4.csv
second_pass_aei_v1.csv: 5481/5481 (100.0%) have ratings
  -> Saved: third_pass_aei_v1.csv
second_pass_aei_v2.csv: 5264/5264 (100.0%) have ratings
  -> Saved: third_pass_aei_v2.csv
second_pass_aei_v3.csv: 4278/4278 (100.0%) have ratings
  -> Saved: third_pass_aei_v3.csv
second_pass_aei_v4.csv: 5143/5143 (100.0%) have ratings
  -> Saved: third_pass_aei_v4.csv
second_pass_mcp_v1.csv: 9575/9575 (100.0%) have ratings
  -> Saved: third_pass_mcp_v1.csv
second_pass_mcp_v2.csv: 11122/11123 (100.0%) have ratings
  -> Saved: third_pass_mcp_v2.csv
second_pass_mcp_v3.csv: 12499/12500 (100.0%) have ratings
  -> Saved: third_pass_mcp_v3.csv
second_pass_mcp_v4.csv: 12792/12793 (100.0%) have ratings
  -> Saved: third_pass_mcp_v4.csv
second_pass_microsoft.csv: 7635/7635 (100.0%) have ratings
  -> Saved: third_p

### Check Columns

In [88]:
DATA_DIR = Path("../data")
pass_segment = "third_pass"  # Change this to "third_pass", "final", etc.

# Define the specific file identifiers you want to check
files_to_check = [
    f"{pass_segment}_microsoft.csv",
    f"{pass_segment}_mcp_v2.csv",
    f"{pass_segment}_aei_v1.csv",
    f"{pass_segment}_eco_2015.csv",
    f"{pass_segment}_eco_2025.csv"
]

for filename in files_to_check:
    file_path = DATA_DIR / filename
    if file_path.exists():
        cols = pd.read_csv(file_path, nrows=0).columns.tolist()
        print(f"--- {filename} ---")
        print(f"{cols}\n")
    else:
        print(f"File not found: {filename}")

--- third_pass_microsoft.csv ---
['task', 'task_normalized', 'dwa_title', 'iwa_title', 'gwa_title', 'title', 'title_normalized', 'soc_code_2010', 'title_current', 'title_current_normalized', 'soc_code_2019', 'soc_code_2019_full', 'broad_occ', 'minor_occ_category', 'major_occ_category', 'date', 'pct_normalized', 'auto_aug_mean', 'freq_mean', 'importance', 'relevance', 'physical', 'a_med_nat_2025', 'a_med_al_2025', 'a_med_ak_2025', 'a_med_az_2025', 'a_med_ar_2025', 'a_med_ca_2025', 'a_med_co_2025', 'a_med_ct_2025', 'a_med_de_2025', 'a_med_dc_2025', 'a_med_fl_2025', 'a_med_ga_2025', 'a_med_hi_2025', 'a_med_id_2025', 'a_med_il_2025', 'a_med_in_2025', 'a_med_ia_2025', 'a_med_ks_2025', 'a_med_ky_2025', 'a_med_la_2025', 'a_med_me_2025', 'a_med_md_2025', 'a_med_ma_2025', 'a_med_mi_2025', 'a_med_mn_2025', 'a_med_ms_2025', 'a_med_mo_2025', 'a_med_mt_2025', 'a_med_ne_2025', 'a_med_nv_2025', 'a_med_nh_2025', 'a_med_nj_2025', 'a_med_nm_2025', 'a_med_ny_2025', 'a_med_nc_2025', 'a_med_nd_2025', 'a_me

### Rename to final

In [89]:
DATA_DIR = Path("../data")
FINAL_DIR = DATA_DIR / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Easy to change these for future runs
current_pass_prefix = "third_pass_" 
new_file_prefix = "final_"

# Find all files starting with the current prefix
files_to_process = [f for f in os.listdir(DATA_DIR) 
                    if f.startswith(current_pass_prefix) and f.endswith(".csv")]

if not files_to_process:
    print(f"No files found starting with '{current_pass_prefix}'")
else:
    for filename in sorted(files_to_process):
        df = pd.read_csv(DATA_DIR / filename)
        
        # Swap out the prefix for the new one
        new_name = filename.replace(current_pass_prefix, new_file_prefix)
        
        df.to_csv(FINAL_DIR / new_name, index=False)
        print(f"Transferred: {filename} -> {new_name}")

    print(f"\nSuccessfully moved {len(files_to_process)} files to {FINAL_DIR}")

Transferred: third_pass_aei_api_v3.csv -> final_aei_api_v3.csv
Transferred: third_pass_aei_api_v4.csv -> final_aei_api_v4.csv
Transferred: third_pass_aei_api_v5.csv -> final_aei_api_v5.csv
Transferred: third_pass_aei_v1.csv -> final_aei_v1.csv
Transferred: third_pass_aei_v2.csv -> final_aei_v2.csv
Transferred: third_pass_aei_v3.csv -> final_aei_v3.csv
Transferred: third_pass_aei_v4.csv -> final_aei_v4.csv
Transferred: third_pass_aei_v5.csv -> final_aei_v5.csv
Transferred: third_pass_eco_2015.csv -> final_eco_2015.csv
Transferred: third_pass_eco_2025.csv -> final_eco_2025.csv
Transferred: third_pass_mcp_v1.csv -> final_mcp_v1.csv
Transferred: third_pass_mcp_v2.csv -> final_mcp_v2.csv
Transferred: third_pass_mcp_v3.csv -> final_mcp_v3.csv
Transferred: third_pass_mcp_v4.csv -> final_mcp_v4.csv
Transferred: third_pass_microsoft.csv -> final_microsoft.csv

Successfully moved 15 files to ..\data\final


## Make Cumulative AEI, Add MCPs To Tasks for MCP Data

In [90]:
"""
Cumulative dataset builder — generates all cumulative dataset variants.

7 buckets across Usage / Agentic / All categories:

Usage:
  - all_confirmed_usage:    AEI Both + Microsoft
  - confirmed_human_usage:  AEI Conv + Microsoft
  - aei_all_usage:          AEI Conv + AEI API
  - aei_human_usage:        AEI Conv only
  - aei_agentic_usage:      AEI API only

Agentic:
  - all_agentic_usage:      MCP + AEI API

All:
  - all_usage:              AEI Both + MCP + Microsoft

Each bucket has multiple cumulative versions — one per new dataset arrival date.

2015 task set buckets (aei_all_usage, aei_human_usage, aei_agentic_usage):
  Merged on full 5-key (title, task_normalized, dwa_title, iwa_title, gwa_title).
  Uses native AEI row structure.

2025 task set buckets (all others):
  Scores extracted per (title_current, task_normalized), combined, then joined to
  ECO 2025 backbone. MCP/Microsoft match on 'title_current' directly. AEI/API
  match their 'title' against ECO 2025's 'title_current' first; unmatched AEI
  rows are then run through a title-fix recovery step that crosswalks the 2010
  SOC code to the corresponding ECO 2025 title_current(s) and re-checks the pair
  against ECO 2025. If the crosswalk yields K valid pair matches, the AEI row's
  pct_normalized is split evenly (pct / K) across them. This only recovers pairs
  that already exist in ECO 2025 — no synthetic (title, task) pairs are injected.

Microsoft physical/Eloundou cleanup is applied upstream (per-version stage), so
Microsoft sources here are already filtered. No bucket-level filter flags.

AEI v1 ships with no auto/aug data. Before combining, v1's auto_aug_mean is
imputed for v1-exclusive tasks from a DWA -> IWA -> GWA mean over the other AEI
sources in the same build (conv v2-v5 and, in API-containing buckets, API v3-v5).
See _impute_aei_v1_auto_aug. Imputation runs in native v20.1 space; v1-exclusive
tasks with no work-activity mapping are dropped.

Combining logic (same for all):
  - auto_aug_mean: max across sources
  - pct_normalized: summed across sources then renormalized so unique (occ, task) pairs sum to 100
  - date: set to latest dataset's date
  - other columns: from latest source (2015) or ECO 2025 backbone (2025)

Output naming: final_{bucket_name}_{end_date}.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ── Source file references ────────────────────────────────────────────────────────────────

AEI_CONV = {
    1: "final_aei_v1.csv",
    2: "final_aei_v2.csv",
    3: "final_aei_v3.csv",
    4: "final_aei_v4.csv",
    5: "final_aei_v5.csv",
}

AEI_API = {
    3: "final_aei_api_v3.csv",
    4: "final_aei_api_v4.csv",
    5: "final_aei_api_v5.csv",
}

MCP = {
    1: "final_mcp_v1.csv",
    2: "final_mcp_v2.csv",
    3: "final_mcp_v3.csv",
    4: "final_mcp_v4.csv",
}

MICROSOFT = "final_microsoft.csv"

AEI_V1_FILE = "final_aei_v1.csv"   # AEI v1 ships with no auto/aug data


# ── Bucket definitions ───────────────────────────────────────────────────────────────────────────────

def _cumul(steps):
    """Build cumulative version list from chronological dataset arrival steps.

    Args:
        steps: list of (end_date, [new_files_added_at_this_date])
    Returns:
        list of {"end_date": str, "files": list[str]} with cumulative file lists
    """
    versions = []
    all_files = []
    for end_date, new_files in steps:
        all_files = all_files + new_files
        versions.append({"end_date": end_date, "files": list(all_files)})
    return versions


BUCKETS = {
    # ── Usage ──
    "all_confirmed_usage": {
        "task_set": "2025",
        "versions": _cumul([
            ("2024-09-30", [MICROSOFT]),
            ("2024-12-23", [AEI_CONV[1]]),
            ("2025-03-06", [AEI_CONV[2]]),
            ("2025-08-11", [AEI_CONV[3], AEI_API[3]]),
            ("2025-11-13", [AEI_CONV[4], AEI_API[4]]),
            ("2026-02-12", [AEI_CONV[5], AEI_API[5]]),
        ]),
    },
    "confirmed_human_usage": {
        "task_set": "2025",
        "versions": _cumul([
            ("2024-09-30", [MICROSOFT]),
            ("2024-12-23", [AEI_CONV[1]]),
            ("2025-03-06", [AEI_CONV[2]]),
            ("2025-08-11", [AEI_CONV[3]]),
            ("2025-11-13", [AEI_CONV[4]]),
            ("2026-02-12", [AEI_CONV[5]]),
        ]),
    },
    "aei_all_usage": {
        "task_set": "2015",
        "versions": _cumul([
            ("2024-12-23", [AEI_CONV[1]]),
            ("2025-03-06", [AEI_CONV[2]]),
            ("2025-08-11", [AEI_CONV[3], AEI_API[3]]),
            ("2025-11-13", [AEI_CONV[4], AEI_API[4]]),
            ("2026-02-12", [AEI_CONV[5], AEI_API[5]]),
        ]),
    },
    "aei_human_usage": {
        "task_set": "2015",
        "versions": _cumul([
            ("2024-12-23", [AEI_CONV[1]]),
            ("2025-03-06", [AEI_CONV[2]]),
            ("2025-08-11", [AEI_CONV[3]]),
            ("2025-11-13", [AEI_CONV[4]]),
            ("2026-02-12", [AEI_CONV[5]]),
        ]),
    },
    "aei_agentic_usage": {
        "task_set": "2015",
        "versions": _cumul([
            ("2025-08-11", [AEI_API[3]]),
            ("2025-11-13", [AEI_API[4]]),
            ("2026-02-12", [AEI_API[5]]),
        ]),
    },
    # ── Agentic ──
    "all_agentic_usage": {
        "task_set": "2025",
        "versions": _cumul([
            ("2025-04-24", [MCP[1]]),
            ("2025-05-24", [MCP[2]]),
            ("2025-07-23", [MCP[3]]),
            ("2025-08-11", [AEI_API[3]]),
            ("2025-11-13", [AEI_API[4]]),
            ("2026-02-12", [AEI_API[5]]),
            ("2026-02-18", [MCP[4]]),
        ]),
    },
    # ── All ──
    "all_usage": {
        "task_set": "2025",
        "versions": _cumul([
            ("2024-09-30", [MICROSOFT]),
            ("2024-12-23", [AEI_CONV[1]]),
            ("2025-03-06", [AEI_CONV[2]]),
            ("2025-04-24", [MCP[1]]),
            ("2025-05-24", [MCP[2]]),
            ("2025-07-23", [MCP[3]]),
            ("2025-08-11", [AEI_CONV[3], AEI_API[3]]),
            ("2025-11-13", [AEI_CONV[4], AEI_API[4]]),
            ("2026-02-12", [AEI_CONV[5], AEI_API[5]]),
            ("2026-02-18", [MCP[4]]),
        ]),
    },
}


# ── Build functions ───────────────────────────────────────────────────────────────────────────────

MERGE_KEY_2015 = ["title", "task_normalized", "dwa_title", "iwa_title", "gwa_title"]


def _load(path: Path) -> pd.DataFrame:
    """Load a CSV and defragment to avoid PerformanceWarning on wide DataFrames."""
    return pd.read_csv(path).copy()


def _soc6(s):
    """Return the 6-digit SOC code (drop the decimal suffix)."""
    if pd.isna(s):
        return None
    return str(s).split(".")[0]


def _build_2010_soc_to_eco_titles(crosswalk_path: Path, eco_2025: pd.DataFrame) -> dict:
    """Build {soc_code_2010 -> set of ECO 2025 title_current values reachable via 2019 crosswalk}.

    Match is via the 6-digit SOC prefix on both sides, which avoids decimal-code mismatches.
    Both the full 2010 code and its 6-digit prefix become keys in the returned dict.
    """
    soc_col = "soc_code_2019_full" if "soc_code_2019_full" in eco_2025.columns else "soc_code_2019"
    eco_tmp = eco_2025.dropna(subset=[soc_col, "title_current"]).copy()
    eco_tmp["_soc6"] = eco_tmp[soc_col].map(_soc6)
    soc6_to_titles = eco_tmp.groupby("_soc6")["title_current"].apply(set).to_dict()

    cx = pd.read_csv(crosswalk_path, dtype=str)
    cx = cx.rename(columns={
        "O*NET-SOC 2010 Code": "soc_2010",
        "O*NET-SOC 2019 Code": "soc_2019",
    })
    cx["soc_2010_6"] = cx["soc_2010"].map(_soc6)
    cx["soc_2019_6"] = cx["soc_2019"].map(_soc6)
    cx_full_to_19_6 = cx.groupby("soc_2010")["soc_2019_6"].apply(set).to_dict()
    cx_6_to_19_6 = cx.groupby("soc_2010_6")["soc_2019_6"].apply(set).to_dict()

    lookup = {}
    for code_2010, codes_2019_6 in cx_full_to_19_6.items():
        titles = set()
        for c6 in codes_2019_6:
            titles |= soc6_to_titles.get(c6, set())
        if titles:
            lookup[code_2010] = titles
    # Also key by 6-digit prefix so source rows with bare 6-digit SOCs still resolve.
    for code_6, codes_2019_6 in cx_6_to_19_6.items():
        titles = set()
        for c6 in codes_2019_6:
            titles |= soc6_to_titles.get(c6, set())
        if titles:
            lookup.setdefault(code_6, titles)
    return lookup


def _recover_aei_via_title_fix(unmatched: pd.DataFrame, eco_pairs: set, cx_lookup: dict) -> pd.DataFrame:
    """For unmatched AEI rows, crosswalk soc_code_2010 -> ECO 2025 title_current(s),
    and keep only crosswalk targets where (target_title, task_normalized) is already
    a pair in ECO 2025. If K targets survive, emit K rows with pct_normalized/K.

    Returns a DataFrame with columns [title_current, task_normalized, pct_normalized,
    auto_aug_mean, date]. No new (title_current, task_normalized) pairs are introduced
    beyond what ECO 2025 already contains.
    """
    recov_rows = []
    for _, row in unmatched.iterrows():
        soc = row.get("soc_code_2010")
        targets = cx_lookup.get(soc)
        if targets is None and pd.notna(soc):
            targets = cx_lookup.get(_soc6(soc))
        if not targets:
            continue
        task = row["task_normalized"]
        valid = [t for t in targets if (t, task) in eco_pairs]
        if not valid:
            continue
        k = len(valid)
        split_pct = row["pct_normalized"] / k if pd.notna(row["pct_normalized"]) else row["pct_normalized"]
        for t in valid:
            recov_rows.append({
                "title_current": t,
                "task_normalized": task,
                "pct_normalized": split_pct,
                "auto_aug_mean": row["auto_aug_mean"],
                "date": row["date"],
            })
    return pd.DataFrame(recov_rows, columns=["title_current", "task_normalized",
                                             "pct_normalized", "auto_aug_mean", "date"])


def _aei_pool_frames(data_dir: Path, files: list[str]) -> list[pd.DataFrame]:
    """Load the non-v1 AEI source frames in `files` -- the pool used to impute
    AEI v1's missing auto_aug_mean. Conv (v2-v5) and API (v3-v5) AEI files both
    qualify; Microsoft and MCP never do (not AEI, different auto/aug scale)."""
    frames = []
    for f in files:
        if "aei" in f and f != AEI_V1_FILE:
            p = data_dir / f
            if p.exists():
                frames.append(_load(p))
    return frames


def _impute_aei_v1_auto_aug(v1_df: pd.DataFrame,
                            pool_frames: list[pd.DataFrame]) -> pd.DataFrame:
    """Impute auto_aug_mean for v1-exclusive tasks in AEI v1, which ships with no
    auto/aug data from Anthropic.

    Runs in native v20.1 taxonomy space (before any 2010->2019 crosswalk), so the
    DWA/IWA/GWA titles line up exactly with the pool -- the imputed value then
    rides through the title-fix crosswalk into the 2025 backbone untouched.

    pool_frames: the other AEI sources participating in THIS build (see
      _aei_pool_frames). Their union, restricted to non-null auto_aug_mean, is the
      reference pool, so the pool grows as later AEI versions arrive.

    Only v1 (title, task_normalized) pairs ABSENT from every pool frame are
    imputed (v1-exclusive). Shared pairs are left null so the real pool score wins
    the downstream max-combine -- an imputed mean must never override an observed
    score. Fallback chain: pool mean auto_aug over DWA -> IWA -> GWA. v1-exclusive
    rows with no DWA/IWA/GWA mapping at all are dropped (nothing to borrow from).

    Returns the modified v1 DataFrame.
    """
    if not pool_frames:
        return v1_df

    pool = pd.concat(pool_frames, ignore_index=True)
    pool = pool[pool["auto_aug_mean"].notna()]
    if pool.empty:
        return v1_df

    dwa_mean = pool.dropna(subset=["dwa_title"]).groupby("dwa_title")["auto_aug_mean"].mean()
    iwa_mean = pool.dropna(subset=["iwa_title"]).groupby("iwa_title")["auto_aug_mean"].mean()
    gwa_mean = pool.dropna(subset=["gwa_title"]).groupby("gwa_title")["auto_aug_mean"].mean()

    v1 = v1_df.copy()
    pool_pairs = set(zip(pool["title"], pool["task_normalized"]))
    is_excl = pd.Series(
        [(t, tn) not in pool_pairs for t, tn in zip(v1["title"], v1["task_normalized"])],
        index=v1.index,
    )

    imputed = (v1["dwa_title"].map(dwa_mean)
               .fillna(v1["iwa_title"].map(iwa_mean))
               .fillna(v1["gwa_title"].map(gwa_mean)))

    fill_mask = is_excl & v1["auto_aug_mean"].isna() & imputed.notna()
    v1.loc[fill_mask, "auto_aug_mean"] = imputed[fill_mask]

    drop_mask = is_excl & v1["auto_aug_mean"].isna()
    n_filled, n_dropped = int(fill_mask.sum()), int(drop_mask.sum())
    v1 = v1[~drop_mask].copy()
    print(f"    AEI v1 auto_aug impute: filled {n_filled} v1-exclusive rows "
          f"(DWA/IWA/GWA), dropped {n_dropped} (no work-activity mapping)")
    return v1


def build_cumulative_2015(data_dir: Path, files: list[str]) -> pd.DataFrame:
    """Build cumulative dataset for 2015 task set (AEI-only sources).

    Merges on full 5-key. For matched rows across versions:
      auto_aug_mean: max, pct_normalized: sum then renormalized to 100
      across unique (title, task_normalized) pairs, date: latest.
    """
    pool_frames = _aei_pool_frames(data_dir, files)
    frames = []
    for f in files:
        p = data_dir / f
        if not p.exists():
            print(f"    WARNING: {p} not found, skipping")
            continue
        df = _load(p)
        if f == AEI_V1_FILE:
            df = _impute_aei_v1_auto_aug(df, pool_frames)
        df["_date_parsed"] = pd.to_datetime(df["date"], errors="coerce")
        frames.append(df)

    if not frames:
        raise FileNotFoundError(f"No source files found: {files}")

    combined = pd.concat(frames, ignore_index=True).sort_values("_date_parsed", ascending=False)

    # Build aggregation: max auto_aug, max pct, latest for everything else
    special = {"auto_aug_mean", "pct_normalized", "date", "_date_parsed"}
    other_cols = [c for c in combined.columns if c not in set(MERGE_KEY_2015) | special]

    agg_dict = {c: "first" for c in other_cols}
    agg_dict["auto_aug_mean"] = "max"
    agg_dict["pct_normalized"] = "sum"
    agg_dict["_date_parsed"] = "max"

    grouped = combined.groupby(MERGE_KEY_2015, sort=False).agg(agg_dict).reset_index()

    # Date: latest across all sources
    latest_date = grouped["_date_parsed"].max()
    grouped["date"] = latest_date.strftime("%Y-%m-%d") if pd.notna(latest_date) else combined["date"].iloc[0]

    # Renormalize so unique (title, task_normalized) pairs sum to 100.
    pair_key = ["title", "task_normalized"]
    pct_total = grouped.drop_duplicates(subset=pair_key)["pct_normalized"].sum()
    if pct_total > 0:
        grouped["pct_normalized"] = grouped["pct_normalized"] * 100.0 / pct_total

    return grouped.drop(columns=["_date_parsed"], errors="ignore")


def build_cumulative_2025(data_dir: Path, files: list[str], eco_2025: pd.DataFrame,
                          cx_lookup: dict) -> pd.DataFrame:
    """Build cumulative dataset for 2025 task set (cross-source, ECO 2025 backbone).

    Extracts scores per (title_current, task_normalized) from each source, combines
    them, then joins to ECO 2025 for structural columns.

    AEI/API sources: match their 'title' against ECO 2025's 'title_current'.
      Unmatched rows go through the title-fix recovery step (crosswalk-based,
      only keeps recoveries that land on pairs already in ECO 2025; pct is split
      evenly across K crosswalk targets).
    MCP/Microsoft sources: match on 'title_current' directly. Microsoft has
    already been filtered upstream (Part-2 cleanup cell) — no source-side filter
    is applied here.
    """
    eco_pairs = set(zip(eco_2025["title_current"], eco_2025["task_normalized"]))
    pool_frames = _aei_pool_frames(data_dir, files)

    all_scores = []
    for f in files:
        p = data_dir / f
        if not p.exists():
            print(f"    WARNING: {p} not found, skipping")
            continue

        df = _load(p)
        if f == AEI_V1_FILE:
            df = _impute_aei_v1_auto_aug(df, pool_frames)
        is_aei = "aei" in f

        if is_aei:
            cols = ["title", "task_normalized", "pct_normalized", "auto_aug_mean", "date", "soc_code_2010"]
            scores = (df.drop_duplicates(subset=["title", "task_normalized"])[cols].copy())
            scores = scores.rename(columns={"title": "title_current"})
        else:
            cols = ["title_current", "task_normalized", "pct_normalized", "auto_aug_mean", "date"]
            scores = (df.drop_duplicates(subset=["title_current", "task_normalized"])[cols].copy())

        mask = pd.Series(
            [(t, tn) in eco_pairs for t, tn in zip(scores["title_current"], scores["task_normalized"])],
            index=scores.index,
        )
        before = len(scores)
        matched = scores.loc[mask].drop(columns=["soc_code_2010"], errors="ignore")

        if is_aei:
            unmatched = scores.loc[~mask]
            recovered = _recover_aei_via_title_fix(unmatched, eco_pairs, cx_lookup)
            scores = pd.concat([matched, recovered], ignore_index=True) if not recovered.empty else matched
            print(f"    {f}: {before} pairs -> {len(matched)} matched + {len(recovered)} recovered "
                  f"(from {len(unmatched)} unmatched via title-fix crosswalk)")
        else:
            scores = matched
            if len(scores) < before:
                print(f"    {f}: filtered {before} -> {len(scores)} pairs (ECO 2025 match)")

        scores["_date_parsed"] = pd.to_datetime(scores["date"], errors="coerce")
        all_scores.append(scores)

    if not all_scores:
        raise FileNotFoundError(f"No source files found: {files}")

    combined = pd.concat(all_scores, ignore_index=True)

    # Combine scores: max auto_aug, sum pct per (title_current, task_normalized)
    grouped = (combined.groupby(["title_current", "task_normalized"], sort=False)
               .agg(auto_aug_mean=("auto_aug_mean", "max"),
                    pct_normalized=("pct_normalized", "sum"),
                    _date_parsed=("_date_parsed", "max"))
               .reset_index())

    # Date: latest across all sources
    latest_date = grouped["_date_parsed"].max()
    grouped["date"] = latest_date.strftime("%Y-%m-%d") if pd.notna(latest_date) else "unknown"
    grouped = grouped.drop(columns=["_date_parsed"])

    # Renormalize so unique (title_current, task_normalized) pairs sum to 100.
    pct_total = grouped["pct_normalized"].sum()
    if pct_total > 0:
        grouped["pct_normalized"] = grouped["pct_normalized"] * 100.0 / pct_total

    # Join to ECO 2025 backbone for structural columns
    eco_base = eco_2025.drop(columns=["pct_normalized", "auto_aug_mean", "date"], errors="ignore")
    result = eco_base.merge(
        grouped[["title_current", "task_normalized", "pct_normalized", "auto_aug_mean", "date"]],
        on=["title_current", "task_normalized"],
        how="inner"
    )

    return result


def print_stats(df: pd.DataFrame, task_set: str) -> None:
    """Print summary stats for a built cumulative dataset."""
    n_rows = len(df)
    title_col = "title_current" if task_set == "2025" and "title_current" in df.columns else "title"
    dedup_key = [title_col, "task_normalized"]
    n_tasks = df.drop_duplicates(subset=dedup_key).shape[0]
    n_occs = df[title_col].nunique()
    pct_total = df.drop_duplicates(subset=dedup_key)["pct_normalized"].sum()
    print(f"    Rows: {n_rows:,}  |  Pairs: {n_tasks:,}  |  Occs: {n_occs:,}  |  pct total: {pct_total:.1f}")


# ── Main execution ───────────────────────────────────────────────────────────────────────────────────

data_dir = Path("../data/final")
output_dir = Path("../data/final")
output_dir.mkdir(parents=True, exist_ok=True)

# Load ECO 2025 once for all 2025-task-set builds
eco_2025 = _load(data_dir / "final_eco_2025.csv")
eco_pairs_count = eco_2025.drop_duplicates(subset=["title_current", "task_normalized"]).shape[0]
print(f"Loaded ECO 2025 backbone: {len(eco_2025):,} rows, {eco_pairs_count:,} unique (title_current, task) pairs")

# Build 2010 SOC -> ECO 2025 title_current lookup once for AEI title-fix recovery
cx_lookup = _build_2010_soc_to_eco_titles(Path("../data/2010_to_2019_soc_crosswalk.csv"), eco_2025)
print(f"Loaded SOC crosswalk: {len(cx_lookup):,} 2010 SOC keys -> ECO 2025 titles\n")

total_files = sum(len(b["versions"]) for b in BUCKETS.values())
file_count = 0

for bucket_name, bucket_config in BUCKETS.items():
    task_set = bucket_config["task_set"]
    versions = bucket_config["versions"]

    print(f"{'=' * 60}")
    print(f"  {bucket_name}  (task set: {task_set}, {len(versions)} versions)")
    print(f"{'=' * 60}")

    for i, version in enumerate(versions):
        end_date = version["end_date"]
        files = version["files"]
        out_name = f"final_{bucket_name}_{end_date}.csv"
        file_count += 1

        print(f"\n  [{file_count}/{total_files}] {out_name}")
        print(f"    Sources: {files}")

        if task_set == "2015":
            df = build_cumulative_2015(data_dir, files)
        else:
            df = build_cumulative_2025(data_dir, files, eco_2025, cx_lookup)

        print_stats(df, task_set)

        out_path = output_dir / out_name
        df.to_csv(out_path, index=False)
        print(f"    Saved: {out_path}")

    print()

print(f"\nDone. Built {file_count} cumulative datasets across {len(BUCKETS)} buckets.")


Loaded ECO 2025 backbone: 23,850 rows, 18,796 unique (title_current, task) pairs
Loaded SOC crosswalk: 1,827 2010 SOC keys -> ECO 2025 titles

  all_confirmed_usage  (task set: 2025, 6 versions)

  [1/42] final_all_confirmed_usage_2024-09-30.csv
    Sources: ['final_microsoft.csv']
    Rows: 8,662  |  Pairs: 6,613  |  Occs: 844  |  pct total: 100.0
    Saved: ..\data\final\final_all_confirmed_usage_2024-09-30.csv

  [2/42] final_all_confirmed_usage_2024-12-23.csv
    Sources: ['final_microsoft.csv', 'final_aei_v1.csv']
    final_aei_v1.csv: 4250 pairs -> 3152 matched + 696 recovered (from 1098 unmatched via title-fix crosswalk)
    Rows: 10,475  |  Pairs: 8,121  |  Occs: 863  |  pct total: 100.0
    Saved: ..\data\final\final_all_confirmed_usage_2024-12-23.csv

  [3/42] final_all_confirmed_usage_2025-03-06.csv
    Sources: ['final_microsoft.csv', 'final_aei_v1.csv', 'final_aei_v2.csv']
    AEI v1 auto_aug impute: filled 950 v1-exclusive rows (DWA/IWA/GWA), dropped 48 (no work-activity 

C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3242766989.py:379: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grouped = combined.groupby(MERGE_KEY_2015, sort=False).agg(agg_dict).reset_index()
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3242766989.py:383: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grouped["date"] = latest_date.strftime("%Y-%m-%d") if pd.notna(latest_date) else combined["date"].iloc[0]


    Saved: ..\data\final\final_aei_all_usage_2024-12-23.csv

  [14/42] final_aei_all_usage_2025-03-06.csv
    Sources: ['final_aei_v1.csv', 'final_aei_v2.csv']
    AEI v1 auto_aug impute: filled 950 v1-exclusive rows (DWA/IWA/GWA), dropped 48 (no work-activity mapping)


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3242766989.py:379: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grouped = combined.groupby(MERGE_KEY_2015, sort=False).agg(agg_dict).reset_index()
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3242766989.py:383: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grouped["date"] = latest_date.strftime("%Y-%m-%d") if pd.notna(latest_date) else combined["date"].iloc[0]


    Rows: 5,771  |  Pairs: 4,639  |  Occs: 781  |  pct total: 100.0
    Saved: ..\data\final\final_aei_all_usage_2025-03-06.csv

  [15/42] final_aei_all_usage_2025-08-11.csv
    Sources: ['final_aei_v1.csv', 'final_aei_v2.csv', 'final_aei_v3.csv', 'final_aei_api_v3.csv']
    AEI v1 auto_aug impute: filled 568 v1-exclusive rows (DWA/IWA/GWA), dropped 26 (no work-activity mapping)


...

    Saved: ..\data\final\final_aei_agentic_usage_2025-08-11.csv

  [24/42] final_aei_agentic_usage_2025-11-13.csv
    Sources: ['final_aei_api_v3.csv', 'final_aei_api_v4.csv']


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3242766989.py:379: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grouped = combined.groupby(MERGE_KEY_2015, sort=False).agg(agg_dict).reset_index()
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3242766989.py:383: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grouped["date"] = latest_date.strftime("%Y-%m-%d") if pd.notna(latest_date) else combined["date"].iloc[0]


    Rows: 3,763  |  Pairs: 2,947  |  Occs: 663  |  pct total: 100.0
    Saved: ..\data\final\final_aei_agentic_usage_2025-11-13.csv

  [25/42] final_aei_agentic_usage_2026-02-12.csv
    Sources: ['final_aei_api_v3.csv', 'final_aei_api_v4.csv', 'final_aei_api_v5.csv']


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3242766989.py:379: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grouped = combined.groupby(MERGE_KEY_2015, sort=False).agg(agg_dict).reset_index()
C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3242766989.py:383: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grouped["date"] = latest_date.strftime("%Y-%m-%d") if pd.notna(latest_date) else combined["date"].iloc[0]


    Rows: 4,110  |  Pairs: 3,219  |  Occs: 683  |  pct total: 100.0
    Saved: ..\data\final\final_aei_agentic_usage_2026-02-12.csv

  all_agentic_usage  (task set: 2025, 7 versions)

  [26/42] final_all_agentic_usage_2025-04-24.csv
    Sources: ['final_mcp_v1.csv']
    Rows: 9,575  |  Pairs: 7,171  |  Occs: 867  |  pct total: 100.0
    Saved: ..\data\final\final_all_agentic_usage_2025-04-24.csv

  [27/42] final_all_agentic_usage_2025-05-24.csv
    Sources: ['final_mcp_v1.csv', 'final_mcp_v2.csv']
    final_mcp_v2.csv: filtered 8359 -> 8358 pairs (ECO 2025 match)
    Rows: 11,122  |  Pairs: 8,358  |  Occs: 886  |  pct total: 100.0
    Saved: ..\data\final\final_all_agentic_usage_2025-05-24.csv

  [28/42] final_all_agentic_usage_2025-07-23.csv
    Sources: ['final_mcp_v1.csv', 'final_mcp_v2.csv', 'final_mcp_v3.csv']
    final_mcp_v2.csv: filtered 8359 -> 8358 pairs (ECO 2025 match)
    final_mcp_v3.csv: filtered 9436 -> 9435 pairs (ECO 2025 match)
    Rows: 12,499  |  Pairs: 9,435  |  O

## Latest-date AEI Usage on ECO 2025 Backbone

In [91]:
# ─────────────────────────────────────────────────────────────────────────
# Latest-date AEI usage on ECO 2025 backbone (additive)
# ─────────────────────────────────────────────────────────────────────────
# Builds two supplementary cumulative files at the latest cumulative date
# only, both on the ECO 2025 backbone via build_cumulative_2025:
#
#   1) aei_agentic_usage_2025  -- AEI API v3+v4+v5
#   2) aei_all_usage_2025      -- AEI Conv v1-v5 + AEI API v3-v5
#
# Reuses build_cumulative_2025, print_stats, eco_2025, cx_lookup,
# AEI_API, AEI_CONV, AEI_V1_FILE from the cell above.

# Additive supplement to the cumulative builder above. Builds two files at the latest cumulative date (2026-02-12), both on the ECO 2025 backbone via `build_cumulative_2025` (AEI rows go through standard text-match + 2010-SOC title-fix recovery):

# 1. `final_aei_agentic_usage_2025_2026-02-12.csv` — AEI API v3+v4+v5
# 2. `final_aei_all_usage_2025_2026-02-12.csv` — AEI Conv v1–v5 + AEI API v3–v5 (v1's missing `auto_aug_mean` is imputed via `_impute_aei_v1_auto_aug` from the in-build AEI pool)

# Downstream Eloundu + column-reorder cells pick both up via the `final_*.csv` glob.

import pandas as pd
from pathlib import Path

LATEST_DATE = "2026-02-12"

SUPPLEMENTARY_BUILDS = [
    {
        "out_name": f"final_aei_agentic_usage_2025_{LATEST_DATE}.csv",
        "files": [AEI_API[3], AEI_API[4], AEI_API[5]],
    },
    {
        "out_name": f"final_aei_all_usage_2025_{LATEST_DATE}.csv",
        "files": [AEI_CONV[1], AEI_CONV[2], AEI_CONV[3], AEI_CONV[4], AEI_CONV[5],
                  AEI_API[3], AEI_API[4], AEI_API[5]],
    },
]

eco_pairs = set(zip(eco_2025["title_current"], eco_2025["task_normalized"]))


def task_loss_diagnostic(files: list[str]) -> None:
    """Print direct-match / title-fix-recovery / lost counts for unique
    (title, task_normalized) pairs across the input AEI files."""
    direct = recovered = lost = 0
    seen = set()
    for f in files:
        df_in = pd.read_csv(Path("../data/final") / f)
        df_in = df_in.drop_duplicates(subset=["title", "task_normalized"])
        for _, row in df_in.iterrows():
            key = (row["title"], row["task_normalized"])
            if key in seen:
                continue
            seen.add(key)
            if key in eco_pairs:
                direct += 1
                continue
            soc = row.get("soc_code_2010")
            targets = cx_lookup.get(soc) if pd.notna(soc) else None
            if targets is None and pd.notna(soc):
                targets = cx_lookup.get(str(soc).split(".")[0])
            valid = [t for t in (targets or []) if (t, row["task_normalized"]) in eco_pairs]
            if valid:
                recovered += 1
            else:
                lost += 1
    total = direct + recovered + lost
    if total == 0:
        print("    (no input pairs)")
        return
    print(f"    Task-loss diagnostic (unique input (title, task) pairs):")
    print(f"      Total input pairs:    {total:>6,}")
    print(f"      Direct text match:    {direct:>6,} ({direct/total*100:5.1f}%)")
    print(f"      Title-fix recovered:  {recovered:>6,} ({recovered/total*100:5.1f}%)")
    print(f"      Kept total:           {direct+recovered:>6,} ({(direct+recovered)/total*100:5.1f}%)")
    print(f"      Lost:                 {lost:>6,} ({lost/total*100:5.1f}%)")


for build in SUPPLEMENTARY_BUILDS:
    out_name = build["out_name"]
    files = build["files"]
    print(f"\n{'=' * 60}")
    print(f"  {out_name}")
    print(f"{'=' * 60}")
    print(f"  Sources: {files}")

    df = build_cumulative_2025(
        data_dir=Path("../data/final"),
        files=files,
        eco_2025=eco_2025,
        cx_lookup=cx_lookup,
    )
    print_stats(df, "2025")
    task_loss_diagnostic(files)

    out_path = Path("../data/final") / out_name
    df.to_csv(out_path, index=False)
    print(f"    Saved: {out_path}")



  final_aei_agentic_usage_2025_2026-02-12.csv
  Sources: ['final_aei_api_v3.csv', 'final_aei_api_v4.csv', 'final_aei_api_v5.csv']
    final_aei_api_v3.csv: 2478 pairs -> 1836 matched + 404 recovered (from 642 unmatched via title-fix crosswalk)
    final_aei_api_v4.csv: 2717 pairs -> 2016 matched + 448 recovered (from 701 unmatched via title-fix crosswalk)
    final_aei_api_v5.csv: 2823 pairs -> 2093 matched + 477 recovered (from 730 unmatched via title-fix crosswalk)
    Rows: 4,001  |  Pairs: 3,077  |  Occs: 633  |  pct total: 100.0
    Task-loss diagnostic (unique input (title, task) pairs):
      Total input pairs:     3,395
      Direct text match:     2,513 ( 74.0%)
      Title-fix recovered:     545 ( 16.1%)
      Kept total:            3,058 ( 90.1%)
      Lost:                    337 (  9.9%)
    Saved: ..\data\final\final_aei_agentic_usage_2025_2026-02-12.csv

  final_aei_all_usage_2025_2026-02-12.csv
  Sources: ['final_aei_v1.csv', 'final_aei_v2.csv', 'final_aei_v3.csv', 'fi

## DWS Ratings

In [92]:
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("../data")

# ── 1. Load ────────────────────────────────────────────────────
eco_2025 = pd.read_csv(DATA_DIR / "final" / "final_eco_2025.csv")
dws = pd.read_csv(DATA_DIR / "dws_ratings.csv")

# Drop any prior dws columns from previous runs
eco_2025 = eco_2025[[c for c in eco_2025.columns if not c.startswith("dws_star_rating")]]

# ── 2. Normalize titles ─────────────────────────────────────────
def normalize_title(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

dws["title_norm"] = dws["Occupation Title"].apply(normalize_title)

# ── 3. DWS lookup (one rating per normalized title) ─────────────
dws_lookup = (
    dws[["title_norm", "Star Ratings"]]
    .drop_duplicates(subset="title_norm")
    .rename(columns={"Star Ratings": "dws_star_rating"})
)

# ── 4. Dedupe occs, merge direct matches, impute ────────────────
occ = (
    eco_2025[["title_current", "broad_occ", "minor_occ_category", "major_occ_category"]]
    .drop_duplicates(subset="title_current")
    .copy()
)
occ["title_norm"] = occ["title_current"].apply(normalize_title)
occ = occ.merge(dws_lookup, on="title_norm", how="left").drop(columns="title_norm")

direct = occ["dws_star_rating"].notna().sum()
print(f"Direct match: {direct}/{len(occ)}")

for level_col, level_name in [
    ("broad_occ", "broad"),
    ("minor_occ_category", "minor"),
    ("major_occ_category", "major"),
]:
    known = occ[occ["dws_star_rating"].notna()]
    level_means = known.groupby(level_col)["dws_star_rating"].mean().round().astype(int).to_dict()
    missing = occ["dws_star_rating"].isna()
    filled = occ.loc[missing, level_col].map(level_means)
    fill_mask = missing & filled.notna()
    occ.loc[fill_mask, "dws_star_rating"] = filled[fill_mask]
    print(f"After {level_name}: +{fill_mask.sum()}, {occ['dws_star_rating'].isna().sum()} still missing")

occ["dws_star_rating"] = occ["dws_star_rating"].astype("Int64")

# ── 5. Merge deduped ratings back into eco_2025 and save ────────
eco_2025 = eco_2025.merge(occ[["title_current", "dws_star_rating"]], on="title_current", how="left")

out_path = DATA_DIR / "final" / "final_eco_2025.csv"
eco_2025.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(eco_2025)} rows, {eco_2025['dws_star_rating'].notna().sum()} rated)")


Direct match: 607/923
After broad: +197, 119 still missing
After minor: +103, 16 still missing
After major: +16, 0 still missing
Saved: ..\data\final\final_eco_2025.csv  (23850 rows, 23850 rated)


## Job Zones

In [93]:
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("../data")

# ── 1. Load ────────────────────────────────────────────────────
eco_2025 = pd.read_csv(DATA_DIR / "final" / "final_eco_2025.csv")
jz = pd.read_csv(DATA_DIR / "job_zones_v30.1.csv")

# Drop any prior job_zone column from previous runs
eco_2025 = eco_2025[[c for c in eco_2025.columns if c != "job_zone"]]

# ── 2. Normalize titles ─────────────────────────────────────────
def normalize_title(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

jz["title_norm"] = jz["Title"].apply(normalize_title)

# ── 3. Job zone lookup (one value per normalized title) ─────────
jz_lookup = (
    jz[["title_norm", "Job Zone"]]
    .drop_duplicates(subset="title_norm")
    .rename(columns={"Job Zone": "job_zone"})
)

# ── 4. Dedupe occs, merge ───────────────────────────────────────
occ = (
    eco_2025[["title_current"]]
    .drop_duplicates(subset="title_current")
    .copy()
)
occ["title_norm"] = occ["title_current"].apply(normalize_title)
occ = occ.merge(jz_lookup, on="title_norm", how="left").drop(columns="title_norm")

occ["job_zone"] = occ["job_zone"].astype("Int64")

matched = occ["job_zone"].notna().sum()
print(f"Matched: {matched}/{len(occ)}  ({occ['job_zone'].isna().sum()} missing)")

# ── 5. Merge back into eco_2025 and save ────────────────────────
eco_2025 = eco_2025.merge(occ[["title_current", "job_zone"]], on="title_current", how="left")

out_path = DATA_DIR / "final" / "final_eco_2025.csv"
eco_2025.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(eco_2025)} rows, {eco_2025['job_zone'].notna().sum()} rated)")

Matched: 923/923  (0 missing)
Saved: ..\data\final\final_eco_2025.csv  (23850 rows, 23850 rated)


## Emp Projections

In [94]:
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("../data")

# ── 1. Load ────────────────────────────────────────────────────
eco_2025 = pd.read_csv(DATA_DIR / "final" / "final_eco_2025.csv")
emp = pd.read_csv(DATA_DIR / "emp_projections.csv")

NEW_COL = "emp_change_pct__PROJ_2025_2034__"
SRC_COL = "Employment change, percent, 2024–34"
EMP_TITLE = "2024 National Employment Matrix title"

# Drop any prior version (rerun-safe)
eco_2025 = eco_2025[[c for c in eco_2025.columns if c != NEW_COL]]

# ── 2. Build normalized-title lookup from emp_projections ──────
emp_lookup = (
    emp[[EMP_TITLE, SRC_COL]].copy()
    .assign(title_norm=lambda d: d[EMP_TITLE].apply(normalize_text))
    .dropna(subset=["title_norm"])
    .drop_duplicates(subset="title_norm")
    .set_index("title_norm")[SRC_COL]
)

# ── 3. Merge stack: detailed -> broad_occ -> minor -> major ────
def lookup_via(col):
    return eco_2025[col].apply(normalize_text).map(emp_lookup)

detailed = lookup_via("title_current")
broad    = lookup_via("broad_occ")
minor    = lookup_via("minor_occ_category")
major    = lookup_via("major_occ_category")

eco_2025[NEW_COL] = detailed.fillna(broad).fillna(minor).fillna(major)

# ── 4. Diagnostics ─────────────────────────────────────────────
total = len(eco_2025)
n_detailed   = detailed.notna().sum()
n_broad_only = (detailed.isna() & broad.notna()).sum()
n_minor_only = (detailed.isna() & broad.isna() & minor.notna()).sum()
n_major_only = (detailed.isna() & broad.isna() & minor.isna() & major.notna()).sum()
n_missing    = eco_2025[NEW_COL].isna().sum()
print(f"Detailed-title matched:  {n_detailed}/{total}")
print(f"Broad-occ imputed:       {n_broad_only}")
print(f"Minor-cat imputed:       {n_minor_only}")
print(f"Major-cat imputed:       {n_major_only}")
print(f"Still missing:           {n_missing}")

# ── 5. Save ────────────────────────────────────────────────────
out_path = DATA_DIR / "final" / "final_eco_2025.csv"
eco_2025.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(eco_2025)} rows, {eco_2025[NEW_COL].notna().sum()} populated)")


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\3152845799.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  eco_2025[NEW_COL] = detailed.fillna(broad).fillna(minor).fillna(major)


Detailed-title matched:  19443/23850
Broad-occ imputed:       3448
Minor-cat imputed:       959
Major-cat imputed:       0
Still missing:           0
Saved: ..\data\final\final_eco_2025.csv  (23850 rows, 23850 populated)


## Eloundu Task Ratings

In [95]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
FINAL_DIR = DATA_DIR / "final"

# 1. Load Eloundu labels (full_labelset.tsv) and normalize text
eloundu = pd.read_csv(DATA_DIR / "full_labelset.tsv", sep="\t")[
    ["Title", "Task", "human_exposure_agg", "gpt4_exposure"]
].rename(columns={
    "human_exposure_agg": "eloundu_human",
    "gpt4_exposure": "eloundu_gpt4",
})
eloundu["title_norm"] = eloundu["Title"].apply(normalize_text)
eloundu["task_normalized"] = eloundu["Task"].apply(normalize_text)

# 2. Helper: mode of a categorical Series, NaN if tied
def mode_or_na(s):
    s = s.dropna()
    if s.empty:
        return pd.NA
    counts = s.value_counts()
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        return pd.NA
    return counts.index[0]

# 3. Pair lookup: one row per (title_norm, task_normalized)
pair_lookup = (
    eloundu.groupby(["title_norm", "task_normalized"], dropna=False)
    .agg(eloundu_human=("eloundu_human", mode_or_na),
         eloundu_gpt4=("eloundu_gpt4", mode_or_na))
    .reset_index()
)

# 4. Task-only fallback: one row per task_normalized, mode across all occupations
task_lookup = (
    eloundu.groupby("task_normalized", dropna=False)
    .agg(eloundu_human_t=("eloundu_human", mode_or_na),
         eloundu_gpt4_t=("eloundu_gpt4", mode_or_na))
    .reset_index()
)

# 5. Apply to every final dataset
print(f"{'file':<55} {'pair_h':>7} {'pair_g':>7} {'total_h':>8} {'total_g':>8} {'rows':>7}")
for filepath in sorted(FINAL_DIR.glob("final_*.csv")):
    df = pd.read_csv(filepath)
    n = len(df)

    # Drop prior eloundu cols (rerun-safe)
    df = df.drop(columns=[c for c in ["eloundu_human", "eloundu_gpt4"] if c in df.columns])

    # 2019-SOC datasets carry title_current; 2015-SOC datasets only have title
    title_col = "title_current" if "title_current" in df.columns else "title"
    df["_match_title_norm"] = df[title_col].apply(normalize_text)

    # Pair merge first
    df = df.merge(
        pair_lookup,
        left_on=["_match_title_norm", "task_normalized"],
        right_on=["title_norm", "task_normalized"],
        how="left",
    ).drop(columns=["title_norm"])
    pair_h = df["eloundu_human"].notna().sum()
    pair_g = df["eloundu_gpt4"].notna().sum()

    # Task-only fallback for unmatched rows
    df = df.merge(task_lookup, on="task_normalized", how="left")
    h_fill = df["eloundu_human"].isna() & df["eloundu_human_t"].notna()
    g_fill = df["eloundu_gpt4"].isna() & df["eloundu_gpt4_t"].notna()
    df.loc[h_fill, "eloundu_human"] = df.loc[h_fill, "eloundu_human_t"]
    df.loc[g_fill, "eloundu_gpt4"] = df.loc[g_fill, "eloundu_gpt4_t"]
    df = df.drop(columns=["_match_title_norm", "eloundu_human_t", "eloundu_gpt4_t"])

    total_h = df["eloundu_human"].notna().sum()
    total_g = df["eloundu_gpt4"].notna().sum()

    df.to_csv(filepath, index=False)
    print(f"{filepath.name:<55} {pair_h:>7} {pair_g:>7} {total_h:>8} {total_g:>8} {n:>7}")

print("\nDone.")


file                                                     pair_h  pair_g  total_h  total_g    rows


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4262842059.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_match_title_norm"] = df[title_col].apply(normalize_text)


final_aei_agentic_usage_2025-08-11.csv                     2294    2294     2769     2781    3044


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4262842059.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_match_title_norm"] = df[title_col].apply(normalize_text)


final_aei_agentic_usage_2025-11-13.csv                     2830    2830     3431     3443    3763


...

final_mcp_v3.csv                                          12402   12402    12402    12402   12500


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4262842059.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_match_title_norm"] = df[title_col].apply(normalize_text)


final_mcp_v4.csv                                          12695   12695    12695    12695   12793


C:\Users\teddy\AppData\Local\Temp\ipykernel_34728\4262842059.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_match_title_norm"] = df[title_col].apply(normalize_text)


final_microsoft.csv                                        7565    7565     7565     7565    7635

Done.


## Column Reorder

In [96]:
FINAL_DIR = Path("../data/final")

# Columns to move right after 'physical'
rating_cols = ["freq_mean", "importance", "relevance"]
# Extra columns only for eco_2025
eco_2025_extra_cols = ["task_prop", "dws_star_rating", "job_zone", "emp_change_pct__PROJ_2025_2034__"]
# Extra columns only for mcp files
mcp_extra_cols = ["top_mcps", "top_mcp_urls"]
# Extra columns only for cumulative files
cumulative_extra_cols = ["auto_aug_mean", "pct_normalized", "date"]
# Eloundu exposure columns - placed last in the moved block so they sit
# right before the national/state wage and employment columns.
eloundu_cols = ["eloundu_human", "eloundu_gpt4"]

for filename in sorted(os.listdir(FINAL_DIR)):
    if not filename.endswith(".csv"):
        continue

    filepath = FINAL_DIR / filename
    df = pd.read_csv(filepath)
    cols = list(df.columns)

    # Determine which columns to move for this file
    if filename == "final_eco_2025.csv":
        cols_to_move = rating_cols + eco_2025_extra_cols + eloundu_cols
    elif filename.startswith("final_mcp"):
        cols_to_move = rating_cols + mcp_extra_cols + eloundu_cols
    elif "_usage_" in filename or "cumulative" in filename:
        # Cumulative files: score cols + ratings + eco_2025 extras + eloundu (filtered to existing)
        cols_to_move = cumulative_extra_cols + rating_cols + eco_2025_extra_cols + eloundu_cols
    else:
        cols_to_move = rating_cols + eloundu_cols

    # Only move columns that actually exist in this df
    cols_to_move = [c for c in cols_to_move if c in cols]

    if not cols_to_move:
        print(f"Skipped {filename} — no columns to move")
        continue

    if "physical" not in cols:
        print(f"Skipped {filename} — no 'physical' column found")
        continue

    # Remove the columns to move from their current positions
    remaining = [c for c in cols if c not in cols_to_move]

    # Insert right after 'physical'
    phys_idx = remaining.index("physical")
    new_order = remaining[:phys_idx + 1] + cols_to_move + remaining[phys_idx + 1:]

    df = df[new_order]
    df.to_csv(filepath, index=False)
    print(f"Reordered: {filename} — inserted {cols_to_move} after 'physical'")

print("\nDone!")


Reordered: final_aei_agentic_usage_2025-08-11.csv — inserted ['auto_aug_mean', 'pct_normalized', 'date', 'freq_mean', 'importance', 'relevance', 'eloundu_human', 'eloundu_gpt4'] after 'physical'
Reordered: final_aei_agentic_usage_2025-11-13.csv — inserted ['auto_aug_mean', 'pct_normalized', 'date', 'freq_mean', 'importance', 'relevance', 'eloundu_human', 'eloundu_gpt4'] after 'physical'
Reordered: final_aei_agentic_usage_2025_2026-02-12.csv — inserted ['auto_aug_mean', 'pct_normalized', 'date', 'freq_mean', 'importance', 'relevance', 'task_prop', 'eloundu_human', 'eloundu_gpt4'] after 'physical'
Reordered: final_aei_agentic_usage_2026-02-12.csv — inserted ['auto_aug_mean', 'pct_normalized', 'date', 'freq_mean', 'importance', 'relevance', 'eloundu_human', 'eloundu_gpt4'] after 'physical'
Reordered: final_aei_all_usage_2024-12-23.csv — inserted ['auto_aug_mean', 'pct_normalized', 'date', 'freq_mean', 'importance', 'relevance', 'eloundu_human', 'eloundu_gpt4'] after 'physical'
Reordered: 

# Testing

### Diagnostic

In [97]:
df = pd.read_csv("../data/third_pass_eco_2025.csv")
# unique_title_df = df.drop_duplicates(subset=["task_normalized", "title"])
unique_title_current_df = df.drop_duplicates(subset=["task_normalized", "title_current"])
unique_title_current_df

,task,task_normalized,dwa_title,iwa_title,gwa_title,title,title_normalized,soc_code_2010,title_current,soc_code_2019,...,emp_tot_wv_2025,emp_tot_wi_2025,emp_tot_wy_2025,emp_tot_gu_2025,emp_tot_pr_2025,emp_tot_vi_2025,freq_mean,importance,relevance,task_prop
0,Direct or coordinate an organization's financi...,direct or coordinate an organizations financia...,Direct financial operations.,Manage budgets or finances.,"Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,2768.0,3126.0,98.0,70.0,4380.0,212.0,0.669603,4.52,74.44,1.0
1,"Confer with board members, organization offici...",confer with board members organization officia...,Confer with organizational members to accompli...,Communicate with others about operational plan...,"Communicating with Supervisors, Peers, or Subo...",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,2768.0,3126.0,98.0,70.0,4380.0,212.0,0.625965,4.32,81.71,1.0
2,"Prepare budgets for approval, including those ...",prepare budgets for approval including those f...,Prepare operational budgets.,Manage budgets or finances.,"Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,2768.0,3126.0,98.0,70.0,4380.0,212.0,0.353887,4.30,93.41,1.0
3,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,Implement organizational process or policy cha...,Implement procedures or processes.,Making Decisions and Solving Problems,Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,2768.0,3126.0,98.0,70.0,4380.0,212.0,0.971421,4.24,97.79,1.0
6,Prepare or present reports concerning activiti...,prepare or present reports concerning activiti...,"Prepare financial documents, reports, or budgets.","Prepare financial documents, reports, or budgets.",Documenting/Recording Information,Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,2768.0,3126.0,98.0,70.0,4380.0,212.0,0.501522,4.17,92.92,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23844,Unload cars containing liquids by connecting h...,unload cars containing liquids by connecting h...,Connect hoses to equipment or machinery.,Connect components or supply lines to equipmen...,Handling and Moving Objects,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,90.0,170.0,19.0,5.0,64.0,2.0,1.248477,4.08,64.04,1.0
23845,"Clean interiors of tank cars or tank trucks, u...",clean interiors of tank cars or tank trucks us...,Clean vessels or marine equipment.,"Clean tools, equipment, facilities, or work ar...",Performing General Physical Activities,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,90.0,170.0,19.0,5.0,64.0,2.0,0.821421,4.02,44.33,1.0
23846,Lower gauge rods into tanks or read meters to ...,lower gauge rods into tanks or read meters to ...,Measure the level or depth of water or other l...,"Measure physical characteristics of materials,...",Estimating the Quantifiable Characteristics of...,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,90.0,170.0,19.0,5.0,64.0,2.0,1.993477,3.88,65.00,1.0
23847,Operate conveyors and equipment to transfer gr...,operate conveyors and equipment to transfer gr...,Operate conveyors or other industrial material...,Operate industrial processing or production eq...,Controlling Machines and Processes,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,90.0,170.0,19.0,5.0,64.0,2.0,2.249240,3.87,47.90,1.0


In [98]:
df = pd.read_csv("../data/final/final_aei_v1.csv")
unique_title_df = df.drop_duplicates(subset=["task_normalized", "title"])
# unique_title_current_df = df.drop_duplicates(subset=["task_normalized", "title_current"])
# unique_title_current_df
unique_title_df

,task,task_normalized,dwa_title,iwa_title,gwa_title,title,title_normalized,soc_code_2010,broad_occ,minor_occ_category,...,emp_tot_ut_2025,emp_tot_vt_2025,emp_tot_va_2025,emp_tot_wa_2025,emp_tot_wv_2025,emp_tot_wi_2025,emp_tot_wy_2025,emp_tot_gu_2025,emp_tot_pr_2025,emp_tot_vi_2025
0,"Interpret and explain policies, rules, regulat...",interpret and explain policies rules regulatio...,Communicate organizational policies and proced...,"Explain regulations, policies, or procedures.",Interpreting the Meaning of Information for Ot...,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,5113.0,586.0,7929.0,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0
1,"Confer with board members, organization offici...",confer with board members organization officia...,Confer with organizational members to accompli...,Communicate with others about operational plan...,"Communicating with Supervisors, Peers, or Subo...",Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,5113.0,586.0,7929.0,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0
2,Review reports submitted by staff members to r...,review reports submitted by staff members to r...,Analyze data to inform operational decisions o...,Analyze data to improve operations.,Analyzing Data or Information,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,5113.0,586.0,7929.0,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0
3,"Serve as liaisons between organizations, share...",serve as liaisons between organizations shareh...,Liaise between departments or other groups to ...,Communicate with others about operational plan...,"Communicating with Supervisors, Peers, or Subo...",Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,5113.0,586.0,7929.0,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0
4,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,Implement organizational process or policy cha...,Implement procedures or processes.,Making Decisions and Solving Problems,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,5113.0,586.0,7929.0,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5474,"Become familiar with digging plans, machine ca...",become familiar with digging plans machine cap...,Maintain professional knowledge or certificati...,Maintain current knowledge in area of expertise.,Updating and Using Relevant Knowledge,Excavating and Loading Machine and Dragline Op...,excavating and loading machine and dragline op...,53-7032.00,Surface Mining Machine Operators and Earth Dri...,Extraction Workers,...,260.0,145.0,725.0,345.0,390.0,1085.0,236.0,12.0,152.0,6.0
5476,Stop gathering arms when cars are full.,stop gathering arms when cars are full,Operate excavation equipment.,Operate construction or excavation equipment.,Controlling Machines and Processes,"Loading Machine Operators, Underground Mining",loading machine operators underground mining,53-7033.00,Underground Mining Machine Operators,Extraction Workers,...,66.0,12.0,190.0,135.0,1250.0,111.0,30.0,3.0,36.0,1.0
5477,Collect and test samples of cleaning solutions...,collect and test samples of cleaning solutions...,"Test materials, solutions, or samples.",Test characteristics of materials or products.,"Inspecting Equipment, Structures, or Material",Cleaners of Vehicles and Equipment,cleaners of vehicles and equipment,53-7061.00,Laborers and Material Movers,Material Moving Workers,...,4580.0,620.0,8860.0,8270.0,1260.0,7040.0,720.0,140.0,1310.0,30.0
5479,Stack cargo in locations such as transit sheds...,stack cargo in locations such as transit sheds...,"Load shipments, belongings, or materials.","Load products, materials, or equipment for tra...",Performing General Physical Activities,"Laborers and Freight, Stock, and Material Move...",laborers and freight stock and material movers...,53-7062.00,L

In [99]:
df = pd.read_csv("../data/third_pass_eco_2015.csv")
unique_title_df = df.drop_duplicates(subset=["task_normalized", "title"])
# unique_title_current_df = df.drop_duplicates(subset=["task_normalized", "title_current"])
unique_title_df

,task,task_normalized,dwa_title,iwa_title,gwa_title,title,title_normalized,soc_code_2010,broad_occ,minor_occ_category,...,emp_tot_wa_2025,emp_tot_wv_2025,emp_tot_wi_2025,emp_tot_wy_2025,emp_tot_gu_2025,emp_tot_pr_2025,emp_tot_vi_2025,freq_mean,importance,relevance
0,Appoint department heads or managers and assig...,appoint department heads or managers and assig...,"Direct organizational operations, projects, or...","Direct organizational operations, activities, ...","Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0,0.348368,4.09,93.14
2,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,Implement organizational process or policy cha...,Implement procedures or processes.,Making Decisions and Solving Problems,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0,0.971421,4.24,97.79
5,"Deliver speeches, write articles, or present i...",deliver speeches write articles or present inf...,Present information to the public.,Provide information or assistance to the public.,Communicating with Persons Outside Organization,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0,0.362546,3.94,92.70
6,"Prepare budgets for approval, including those ...",prepare budgets for approval including those f...,Prepare operational budgets.,Manage budgets or finances.,"Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0,0.353887,4.30,93.41
7,Nominate citizens to boards or commissions.,nominate citizens to boards or commissions,NaN,NaN,NaN,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,5161.0,2768.0,3126.0,98.0,70.0,4380.0,212.0,0.391724,4.07,27.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24625,Check conditions and weights of vessels to ens...,check conditions and weights of vessels to ens...,Inspect cargo areas for cleanliness or condition.,Inspect facilities or equipment.,"Inspecting Equipment, Structures, or Material","Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",Material Moving Workers,...,243.0,90.0,170.0,19.0,5.0,64.0,2.0,3.606558,4.48,86.24
24626,"Operate industrial trucks, tractors, loaders a...",operate industrial trucks tractors loaders and...,Operate vehicles or material-moving equipment.,Operate transportation equipment or vehicles.,"Operating Vehicles, Mechanized Devices, or Equ...","Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",Material Moving Workers,...,243.0,90.0,170.0,19.0,5.0,64.0,2.0,2.945191,4.24,72.56
24627,Operate conveyors and equipment to transfer gr...,operate conveyors and equipment to transfer gr...,Operate conveyors or other industrial material...,Operate industrial processing or production eq...,Controlling Machines and Processes,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",Material Moving Workers,...,243.0,90.0,170.0,19.0,5.0,64.0,2.0,2.249240,3.87,47.90
24628,Copy and attach load specifications to loaded ...,copy and attach load specifications to loaded ...,Mark materials or objects for identification.,Mark materials or objects for identification.,"Identifying Objects, Actions, and Events","Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",Material Moving Workers,...,243.0,90.0,170.0,19.0,5.0,64.0,2.0,2.841213,4.43,60.24


In [100]:
import pandas as pd

df = pd.read_csv("../data/third_pass_eco_2015.csv")
df = df.drop_duplicates(subset=["task_normalized", "title"])

# Create a boolean flag for whether a row was imputed
df['is_imputed'] = df['rating_imputed'] != 'none'

# Group by title and calculate the percentage of imputed tasks
title_stats = df.groupby('title').agg(
    total_tasks=('is_imputed', 'count'),
    imputed_tasks=('is_imputed', 'sum')
)
title_stats['pct_imputed'] = title_stats['imputed_tasks'] / title_stats['total_tasks']

# Filter for titles where more than half are imputed
highly_imputed_titles_list = title_stats[title_stats['pct_imputed'] > 0.5].index

print(f"Total unique titles: {len(title_stats)}")
print(f"Titles with >50% imputed tasks: {len(highly_imputed_titles_list)}")

if len(highly_imputed_titles_list) > 0:
    # Filter the main dataframe to only include these specific titles
    high_imp_df = df[df['title'].isin(highly_imputed_titles_list)]
    
    print("\nBreakdown of imputation types for titles with >50% imputation:")
    # We exclude 'none' to see only the imputation methods used
    impute_counts = high_imp_df[high_imp_df['rating_imputed'] != 'none']['rating_imputed'].value_counts()
    print(impute_counts)

    print("\nTop 10 most imputed titles:")
    print(title_stats.loc[highly_imputed_titles_list].sort_values('pct_imputed', ascending=False).head(10))

KeyError: 'rating_imputed'

## Test Computation

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path("../data/final")
OUT_DIR = DATA_DIR / "test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# TOGGLES
# ---------------------------------------------------------
use_auto_aug = False          # multiply task comp by auto_aug_mean / 5
method = "freq"               # "freq" or "imp"
geo = "nat"                   # "nat" or "ut"
agg_level = "occupation"      # "occupation", "broad", "minor", "major"

emp_col = f"emp_tot_{geo}_2025"
wage_col = f"a_med_{geo}_2025"

prefix = f"{'aug_' if use_auto_aug else ''}{'freq' if method == 'freq' else 'imp'}_{geo}_{agg_level}"

# Map agg_level to column name
AGG_MAP = {
    "occupation": None,  # special case — handled per-file
    "broad": "broad_occ",
    "minor": "minor_occ_category",
    "major": "major_occ_category",
}

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def compute_task_comp(df, method, use_auto_aug):
    """Compute task completion value per row."""
    if method == "freq":
        tc = df["freq_mean"].copy()
    else:
        tc = df["relevance"] * (2 ** df["importance"])
    
    if use_auto_aug:
        aug_col = "auto_aug_mean"
        tc = tc * (df[aug_col] / 5)
    
    return tc


def dedup_and_compute(df, title_col, method, use_auto_aug):
    """Deduplicate to unique (title, task) pairs, compute task_comp."""
    # Take first values for other columns, but we mainly need the numeric ones
    agg_dict = {}
    for c in [emp_col, wage_col, "broad_occ", "minor_occ_category", "major_occ_category",
              "freq_mean", "importance", "relevance", "auto_aug_mean"]:
        if c in df.columns:
            agg_dict[c] = "first"
    if "auto_aug_mean" in df.columns:
        agg_dict["auto_aug_mean"] = "first"
    
    deduped = df.groupby([title_col, "task_normalized"]).agg(agg_dict).reset_index()
    deduped["task_comp"] = compute_task_comp(deduped, method, use_auto_aug)
    return deduped


def aggregate_results(ai_df, eco_df, title_col, agg_level):
    """
    Compute pct_tasks_affected, workers_affected, wages_affected
    at the specified aggregation level.
    """
    if agg_level == "occupation":
        group_col = title_col
    else:
        group_col = AGG_MAP[agg_level]

    # --- Occupation-level first (needed for workers/wages at all levels) ---
    ai_by_occ = ai_df.groupby(title_col).agg(
        ai_task_comp=("task_comp", "sum"),
    ).reset_index()

    eco_by_occ = eco_df.groupby("title_current").agg(
        eco_task_comp=("task_comp", "sum"),
        **{emp_col: (emp_col, "first")},
        **{wage_col: (wage_col, "first")},
        broad_occ=("broad_occ", "first"),
        minor_occ_category=("minor_occ_category", "first"),
        major_occ_category=("major_occ_category", "first"),
    ).reset_index()

    occ_merged = eco_by_occ.merge(
        ai_by_occ,
        left_on="title_current",
        right_on=title_col,
        how="left",
    )
    if title_col != "title_current" and title_col in occ_merged.columns:
        occ_merged.drop(columns=[title_col], inplace=True)

    occ_merged["ai_task_comp"] = occ_merged["ai_task_comp"].fillna(0)
    occ_merged["pct_tasks_affected"] = (occ_merged["ai_task_comp"] / occ_merged["eco_task_comp"]) * 100
    occ_merged["pct_tasks_affected"] = occ_merged["pct_tasks_affected"].clip(upper=100)
    occ_merged["workers_affected"] = (occ_merged["pct_tasks_affected"] / 100) * occ_merged[emp_col]
    occ_merged["wages_affected"] = (occ_merged["pct_tasks_affected"] / 100) * occ_merged[emp_col] * occ_merged[wage_col]

    if agg_level == "occupation":
        return occ_merged[["title_current", "pct_tasks_affected", "workers_affected", "wages_affected",
                           "ai_task_comp", "eco_task_comp", emp_col, wage_col]].sort_values("pct_tasks_affected", ascending=False)

    # --- Higher aggregation levels ---
    # pct_tasks_affected: sum AI comp / sum eco comp at group level
    ai_by_group = ai_df.groupby(group_col)["task_comp"].sum().reset_index(name="ai_task_comp_group")
    eco_by_group = eco_df.groupby(group_col)["task_comp"].sum().reset_index(name="eco_task_comp_group")

    group_merged = eco_by_group.merge(ai_by_group, on=group_col, how="left")
    group_merged["ai_task_comp_group"] = group_merged["ai_task_comp_group"].fillna(0)
    group_merged["pct_tasks_affected"] = (group_merged["ai_task_comp_group"] / group_merged["eco_task_comp_group"]) * 100
    group_merged["pct_tasks_affected"] = group_merged["pct_tasks_affected"].clip(upper=100)

    # workers_affected and wages_affected: sum from occupation level
    occ_by_group = occ_merged.groupby(group_col).agg(
        workers_affected=("workers_affected", "sum"),
        wages_affected=("wages_affected", "sum"),
    ).reset_index()

    group_merged = group_merged.merge(occ_by_group, on=group_col, how="left")

    return group_merged[[group_col, "pct_tasks_affected", "workers_affected", "wages_affected",
                         "ai_task_comp_group", "eco_task_comp_group"]].sort_values("pct_tasks_affected", ascending=False)


# ---------------------------------------------------------
# 1. PREP ECO BASELINE
# ---------------------------------------------------------
eco_2025 = pd.read_csv(DATA_DIR / "final_eco_2025.csv")
eco_deduped = dedup_and_compute(eco_2025, "title_current", method, False)  # no auto_aug on baseline

# Need group columns on eco for higher-level aggregation
# They're already there from dedup_and_compute's agg

# ---------------------------------------------------------
# 2. PREP CROSSWALK (for AEI)
# ---------------------------------------------------------
crosswalk = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")

# ---------------------------------------------------------
# 3. PROCESS EACH FILE
# ---------------------------------------------------------
final_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("final_") and f.endswith(".csv") and "eco" not in f])

for filename in final_files:
    df = pd.read_csv(DATA_DIR / filename)
    is_aei = "aei" in filename

    if is_aei:
        # Dedup on (title, task_normalized) with 2010 titles
        ai_deduped = dedup_and_compute(df, "title", method, use_auto_aug)

        # Merge crosswalk
        # Need soc_code_2010 — get it from original df
        soc_lookup = df[["title", "soc_code_2010"]].drop_duplicates(subset=["title"])
        ai_deduped = ai_deduped.merge(soc_lookup, on="title", how="left")

        ai_deduped = ai_deduped.merge(
            crosswalk[["O*NET-SOC 2010 Code", "O*NET-SOC 2019 Title"]],
            left_on="soc_code_2010",
            right_on="O*NET-SOC 2010 Code",
            how="left",
        )

        # Split count: how many 2019 titles does each 2010 code map to
        split_counts = (
            crosswalk.groupby("O*NET-SOC 2010 Code")["O*NET-SOC 2019 Title"]
            .nunique()
            .reset_index(name="split_count")
        )
        ai_deduped = ai_deduped.merge(split_counts, left_on="soc_code_2010", right_on="O*NET-SOC 2010 Code",
                                       how="left", suffixes=("", "_sc"))
        ai_deduped.drop(columns=[c for c in ai_deduped.columns if c.endswith("_sc")], inplace=True)
        ai_deduped["split_count"] = ai_deduped["split_count"].fillna(1)

        ai_deduped["task_comp"] = ai_deduped["task_comp"] / ai_deduped["split_count"]
        ai_deduped[emp_col] = ai_deduped[emp_col] / ai_deduped["split_count"]

        # Collapse to unique (task_normalized, O*NET-SOC 2019 Title)
        agg_cols = {"task_comp": "sum", emp_col: "sum"}
        for c in ["broad_occ", "minor_occ_category", "major_occ_category", wage_col]:
            if c in ai_deduped.columns:
                agg_cols[c] = "first"

        ai_final = (
            ai_deduped.groupby(["O*NET-SOC 2019 Title", "task_normalized"])
            .agg(agg_cols)
            .reset_index()
            .rename(columns={"O*NET-SOC 2019 Title": "title_current"})
        )
        
        # --- Apply task_prop deflation ---
        task_prop_lookup = eco_2025[["title_current", "task_prop"]].drop_duplicates(subset=["title_current"])
        ai_final = ai_final.merge(task_prop_lookup, on="title_current", how="left")
        ai_final["task_prop"] = ai_final["task_prop"].fillna(1.0).clip(lower=1.0)
        ai_final["task_comp"] = ai_final["task_comp"] / ai_final["task_prop"]
        ai_final.drop(columns=["task_prop"], inplace=True)

        # Fill in group columns from eco where missing
        eco_groups = eco_2025[["title_current", "broad_occ", "minor_occ_category", "major_occ_category"]].drop_duplicates(subset=["title_current"])
        for gc in ["broad_occ", "minor_occ_category", "major_occ_category"]:
            if gc in ai_final.columns:
                mask = ai_final[gc].isna()
                if mask.any():
                    fill = ai_final.loc[mask, ["title_current"]].merge(eco_groups[["title_current", gc]], on="title_current", how="left", suffixes=("_old", ""))
                    ai_final.loc[mask, gc] = fill[gc].values

        title_col_for_agg = "title_current"

    else:
        ai_final = dedup_and_compute(df, "title_current", method, use_auto_aug)
        title_col_for_agg = "title_current"

    # Aggregate and output
    result = aggregate_results(ai_final, eco_deduped, title_col_for_agg, agg_level)

    # Diagnostics
    if "pct_tasks_affected" in result.columns:
        over_100 = (result["ai_task_comp"] > result["eco_task_comp"]).sum()
        print(f"{filename} | {agg_level}: {len(result)} rows, {over_100} at 100% cap")

    out_name = f"{prefix}_{filename}"
    result.to_csv(OUT_DIR / out_name, index=False)

final_aei_api_v3.csv | occupation: 923 rows, 0 at 100% cap
final_aei_api_v4.csv | occupation: 923 rows, 0 at 100% cap
final_aei_v1.csv | occupation: 923 rows, 0 at 100% cap
final_aei_v2.csv | occupation: 923 rows, 0 at 100% cap
final_aei_v3.csv | occupation: 923 rows, 0 at 100% cap
final_aei_v4.csv | occupation: 923 rows, 0 at 100% cap
final_mcp_v1.csv | occupation: 923 rows, 0 at 100% cap
final_mcp_v2.csv | occupation: 923 rows, 0 at 100% cap
final_mcp_v3.csv | occupation: 923 rows, 0 at 100% cap
final_mcp_v4.csv | occupation: 923 rows, 0 at 100% cap
final_microsoft.csv | occupation: 923 rows, 0 at 100% cap
